# FEVER on Google Colab
This version includes live stage logs and saved debug traces. Reuse the original `FEVER_Colab_Bundle.zip`; only the notebook needs updating.

Start with a GPU setup check. Then run **RoBERTa, Qwen or Llama** against the full **19,998-claim official FEVER development set**, with resumable outputs in Google Drive.

**This notebook contains no new results.** The included small training/validation sets only test the software. They are not final fine-tuning experiments. This benchmark has previously been inspected; do not call it an untouched test set.

1. Open this file in [Google Colab](https://colab.research.google.com/), then select **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, create a folder named **FEVER_Project** and upload both this notebook and `FEVER_Colab_Bundle.zip` into it. Leave the ZIP zipped. The notebook finds it automatically; you do not enter data or a file path.
3. Run sections 1–5. They verify the GPU, uploaded data and source hashes. They do not start training or the full evaluation.
4. Downloads and the 90-claim development diagnostic (section 6b) are enabled. The one-claim check is optional. Run sections 1–6b, then inspect its report. Full evaluation and training stay off. The first Qwen download is approximately 5.75 GiB plus 0.08 GiB for embeddings.
5. Enable a long run only after inspecting that check. Free Colab sessions may end early. Drive checkpoints allow resuming, but free Colab does **not** guarantee background execution after closing your laptop.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [Qwen 3B](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), [Llama 3.2 3B](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).

## 1. Check your GPU
This prints the allocated GPU and available memory. It stops if the runtime is CPU-only.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch
print("Python:", sys.version.split()[0])
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)

## 2. Install the Colab dependencies
This installs Python libraries in the **Colab runtime**, not on your laptop. It does not replace Colab's CUDA-enabled PyTorch. Model weights download later, after the size estimate.

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/fever_hf_cache"
packages = ["transformers==4.57.6", "accelerate>=1.10,<2", "bitsandbytes>=0.48,<0.49",
            "faiss-cpu==1.12.0", "scikit-learn>=1.6,<2", "requests>=2.32,<3", "librosa>=0.11,<0.12"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("Libraries installed. If Colab requests a runtime restart, restart and rerun from section 1.")

## 3. Connect Drive — find the uploaded data automatically
Mounting Drive asks you to authenticate. The notebook never prints account tokens. Keep the bundle private; it does not need to be published anywhere.

Use **My Drive / FEVER_Project** for the two uploaded files. This cell creates that folder if necessary and finds the ZIP automatically. It also recognizes the earlier My Drive root location. The working extraction is temporary; new experiment results are saved under **FEVER_Project / runs**.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, json, sys

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/FEVER_Project")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
locations = [DRIVE_PROJECT / "FEVER_Colab_Bundle.zip",
             Path("/content/drive/MyDrive/FEVER_Colab_Bundle.zip")]
archive = next((p for p in locations if p.is_file()), None)
if archive is None:
    raise FileNotFoundError("The FEVER_Project folder is ready in My Drive. Upload FEVER_Colab_Bundle.zip into it, wait for the upload to finish, then rerun this cell. No path needs typing.")
print("Found your data:", archive)
PROJECT = Path("/content/fever_colab_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    # Refuse traversal and links in an uploaded ZIP.
    for info in z.infolist():
        target = (PROJECT / info.filename).resolve()
        target.relative_to(PROJECT.resolve())
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlink rejected")
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
import fever_colab_runtime as fever
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print("Project ready:", PROJECT)
# Install the debug runtime embedded in this notebook; the original Drive ZIP is unchanged.
import base64, zlib, hashlib, importlib
patch_source = zlib.decompress(base64.b64decode('eJy9fety20bW4H89BZypnQZsCKLsxBuTQaYcWc74G99WtjO7xbDwgSQoISIBDgBKVjyq2qfZB9sn2XPrC0CQUjyzX2rGIoC+9+lzP6e/+eabk3KZTr10nq6brPIWZeVln/O6yYtzb7FZLg/Tqslny8w7e/5z5L0tveuyuvSg1Ly8LpZlOq+9svDy1bqsmujg4F2ReT+//+Stynm29NLGS70mX2UjKLHaNOkUGlpn1eFsmeYrb3aRzS7XZV401MiLKr/KIu/jRV578L/UK7Lrg7++PPr2cJo3MCqoCE0VTegVJTZcZetleuOVC6+5yLwsrZY5zODdcpmuUu/7n7xVltabKsMq0cE333xzsKhK6LRcLrNZk5dFLcP2TspNAZPn7/O0yXDI+qN+DvGf38siO5AP57PwIq0vlvk05DfwK1plDSxlk4a/1WURrtLmIizrsEqLebkKK2gkW60X+ZJb4w7XUAiq6v7ew+PB6+c/nb7+EI/Vh0/v3787+/hBhers9OWnj6f46+27j97p23effv6r9+rty3dqcvDm3Qss/0X94zor1FD9D/hzhP88jr47fPLT4auibqrNrIHKtDxQBEd6SA9Hr/HfwyfR41bR24ODebbw6os0wRH7OM5geOBdxDLtCD49/u6pHxx413lzQSPnUlG5zgpfVVMVeGntLaAWQRZMEmAFltqHHqfzdLiIqiyd+8eDx98+xH+CcKpUMLyINmtcd38KbVdZs6kK7yK6yD7P8/OsbqBHGll+XqTwLfPL6W/BUJdrDw73IZpvVusaS4U1LHFymd3U8cdqk4VZgRCSpPUsz+OX6bLOgigrZgC8fhBsdYhjTaryupa1kB7H1AedBb8OaKI1TtRZD6rZZJ+hpaheL3OAlSKr/cDLYRoRLHi+9oMJdVKU1Spd5r9n0JbuQXkq+g3OiV9Hs7TOFuVyrhuCcUq1a9+U1zAb4VsNt9GmmQVRXpcL7EHP6bqC/UjKYsYbHF6ly02G2wwH5v1muszrCzgyq/UyazLvPz68ewsndXkDmCC7gsNGR3CWwWHUZebeusrmOR2wo1lZ4KQjPHwE57FdkxH+G63TCo/n6nKeVz4/6K1BLJSUl/QEYAArRTXoPSwdwhS8c9aePjsrHTyIaTbDKs3rzPsFf59WVVn56lSjOIuWEM8t0hkc+HyxyKp66KlHsDE8WOhf4PDAwyMcU18I9UmRrjJfReoRvcKnRyoC8J9D80ofDawjh+LaORMGOH0aaLjoA8jRIlosN/WFH4zKOlrUN8XMh1dwJIsSd9+Dt7IPPvYT0ohpcxdVlgEgIbxkVbiuyqYE9IdLx694O/g39EN/794L2rmYSx8p3WqEk1FfuU+6kZ6tei+f9L5E3llWbwA/88IC5i+r/Dwv0qVXZw1uah0ifZpdlGWNkPn29O9etSk83JoIdySDVR124d4sjkE4Fr3Yb7ioAPf54iaZboo5YMWqLBucI/7l5aQ3o1XszJjeHalVWuQLQCe8VC2sEBwQ3sAxhoxzEIOsxgr3uVaTCEa7sqtpkDI1jLVgEblezxL+RENlcgtL5y1SqDpHAKeaOPjrOrbYjUeLZOwoA8BM8mK9aWoa9VIFoypb1D2zs+WBoYA17p8lDB4A18d+YMjHz549+x53C999qcYqn6sJrUSF88dSt26xL3gipVjQKfcghv33cXBB74lfA9UH9JQul97xs5AaXCzyWQ6AMwdktizXyCl4r17UiscpbAE1GdEBrQlY9fsvy+FT+I+GscRhMNW+7ev+U5HpAZheaZkAcUJrusu0uPEVvVc0L5yzOoeT5jxmV/kcaFQmr1qL0DtzHHqK+NhbZATQtYeIOYU6vFNHaQEcFRWhgdAm4o7FXyw1glUnrm174QElQq9FPwDRp2SdL8vGABA0318YPuRzGke7xgEe7Toe3zWcz7f0+zP+HlPPSM8mE1pbbGI8mPyZ/h5PcC1xues/m+la0o1l+tbyre5/Liws0WCvBJywTNc40HUF7KyvfkEkkcNuM5oYapDL7GZQA/VI9gC6BJQLdHvk0dR5TQGbOYsCKLBGNqAhNCZoCtcwRBBldg2PZHKxSJryEo4U4oumuiH+C3nN87I8X2bRjHh+YTg3dVbh6hsSZ95E53Cg1F9fJh/f/e30LUHG51m2brxT+gMj0gwHkKCsuMorwAndOsw4rUrAtpo5hhO1LmHM/4ATBSciqQCka2gtfgtsClEnHOvF5hzQ+jnQ5Cy52Jjh/nXxfJ3zrPJiUcb0DNwQSRwJvtLNS6Pb/YSEVM1wmMbRgsWd5euZMhJvzVKcwdEF7kqA44SOkZfOZlldA3LFYQAv4L0EqCIWG+jSlI6dSEdc8KbcVHW2xMMP2Gk+9/TiISCydPYhm8FC1yR/pZumBPYtn3nLfAbsQkbtrJsUUELk0VC85madIXrHvz6wtAlxKUkS8MriMsORAjCOgRz4PmxZ2gBirUOFL1U4CPB0DBxe1seFjeocGEIgsPh1PBHutSJWBIkrMD01UmVgh+p0AcxgAew2YBRzLGhjlN4GFXKjF2morrP8/KIxQqWXroHofs5hotnyRoUVIMm5j4M7evzw4ZNB+DgI1c/5T94UmOEq81hErHg7vH9sUvjxO50Z56R8Udi/GnZGMbSjsKNO9EiS6U0DBHiIfd8SMAugI7Gg8yVw2ZTV7AK53NllCnQ4/rIebouFESAK7NRf89quCVMpqguyGZz5okb2HErBI27sMkOkAA8wwRoghIcTqgK4RlgYBYziZd4cLkH8hQVVQNrr+nC23qjJrTNxPSo11L9CdQ6FhtRzNNvM6bAnQAgBqJinHdD+OgXyOkmvgHVAdhlEF2SjCJRChZ91WzJDqsILRnxHgs8965UvSJrf2U3vSfuQoQivX3o/eicXaXGeGQBAuIe3H79FXUSITCLgyQyZQEDsmzWdI1RueCfvPxFvmAP+bUACyOaRg8Sh8lCFe5coVL+cPX/jASgONZT2lwdwXmcgYgAPMQiiBsjtEhDQqqxuBKKPA40rYS/rrGepzmeRKC985hZ2L5rzBSSC5iaZpbALUO3g7NPbj6/enCZnp7+8+vAKxLnYU0QODufZdHN+ePWdOjgA0gSICZb3TVnk0BaLgyfAHpSowKnK8wox1xrEEm++qUh+wunRiUPOpjwHwlaQiAi4Dilh7ciFdXRw8BEY9ynA4jktGmwRMgS88LQXsJsZkNQaSChJst6rBneJNT+MFS+zbH0IxPEqY/WOh8uXAAnImyTxEaWGIvkAuzPPseeQFj40wmo8ABxEBBZ+XcAZaqbAIiV1hjXq+PEgrC/K60TPmCnFBjqIhftghly2ieYA+BFeYe9Rn5DlfLiP1MsNmfHH5hc3JEAb0xLBBiySmTCngSlwniW6mFtHt70961X62V8A7mv8rW9BGA2+0023Vqb1pNtelrPL2CxLdPYanu3IynUCEFI0TonTK8KqXAIhi8AlHk/4zTStYiZe3H4NfSGzCNuHggGtAv6TFMipj2QM5wkJrM7KH/lKVwUhv90SUGzDdUovDQxvFbdaE2H+swpJXQRjj9WmWRx+71YDxB1/UbZpoCLtvgBz6v1UQwukiqAUESoCqzLQimXkZ6gEbhMpK48h9O7+p3AUG8D7CtAcspNIMhAm4A0BAr8iYKYRMkLXQyyy64T5VDUcQDFYCGSkNs1MDUnjFCo8DWqI/95q2dSBBDwemgMVHrT5x3wVIRdjzg28oM9mj/GNTxOLeQ3wTIPIFNv5z7N6Zk8Dn0n8J5zfAGrOZwlsy7I2x0izca+oSyIjQAotpDIIOpD4kX75sESAxHnvE3McwnkKqLsQdYjTAB8u38AAwbcFNjl6KrRAiWyXC1jILBEeA/JMXwjD6BVsEwRvPxEw1B9E3nJGG3eeT4H/6ZIppkOJKeYHDpcFu5hVV3fV1aWcqggOW/KCrLkMjc8yTbdOrwhPAbDaafNz/OXhQ3uiAOJIN9yGQr3CQjM0vjID3saPhy4yxJketE8NoM2vaMwiW1o82ESh8nL2aVtpaUiN2MFJNN1elOQoFFmZyPorUqs4Km5uAY7LHPHq4z514jbCGnUViO6oZFCizDGbdcP0FfvXOikkzduEYajVph6qvGM1Vo9oy8bHx8PjZ5NHyvv08WTiyZzNgYb3/0QBBtoXrCK4QTMBCDpD/ZJXw8cecIFIwcc8HL4KSX2qUQGx8HQsaQa0Z+HDh8CiLIG+4VxIsWjo11DOmUMKotmyBLiwU6Pv2NCD2AIqS8KMbIOhRXBtktwDTG6DsTAaBnUIIgXZ4TK7YenBMlYKycJ607D4imJCuWlaz1WKWwMsaXYNT5k+l2oydAa+Ltc+tB6SOH7gufRsLBOaxPR35NQSk40sJFar4NxUcyCBfSf1LnLYhgeXRHWX2HyB1mlnoUhrV2+dOdD+Mbg454ZH2ndcHqlfAe5Hbm3RxptGW7iLX8MJiL9cDq94o8Ir3Cgei1bk4g5fEiTniKKZm9y9P7e3bnc3vsza+xMbKdwNMssxCfgUOdDbs2bugWKZTqngES0gVFePXOwC0+pbIw2xMkPbAcC9HbI6wVeoluByRoE3Hh5/P5j0NeKuRqups/TaY9D25Lvbrlutt127rq1WmS1wGnLOh2AO3Oq54L4KECaTKWGc6c1YXQMhNTRj0uFlo3SNuNzXDPVB+3RZXm/yKD4etQ7eNk+GZToNtNlCKIBYUAbGiuUJ4JRXb395/vrVCyInqywtSA/UHuj48LvBcBIcoWa+98uoylasoozdAUjHhztmhei5SWNT9yF2r60C7X6CH+MnVstwHzIgKOg4MOICYJcmWZd1s8g/+4go4/ZaGPFvzxqidhVI2YVAe4e3E/SMcKFCi4q7/ViRc8fCsGjat5TEmbTGtHe0MiCBSdGc8RPyJIDqYNwJLnunEL7SjBC8Ob9IzC4lF6iklHKwf0dPng6Qx8MtgcdtHEJiBI/b0h9eEz6CLToVhC065ZZrEzAZnD3fblHYozWADSrLVMBIJTQn2C2Y4TmXtuyOWYbcKdr5TE0bFt3IA5Zjvb7Il5nDChkhN7pOcy63LXEzd9DDdtzJd3h/8v6WZWsyg6YwzKuM2YaRl3qmG707Rm9Dlo8S6lSiiZ4BxoqoPWF54z/E6R4HLnfTT2HbTICQaWXGqO5iDO5D/Q9c7n27pLBi4S7uXl7c7qH5ox0E36HM6kOTL5fkJoUOBv/0dhDq7uiFWD/qHzXCnS4AzchQ8RWp55uLXLRmmkcnUDEsLkpRap1uakRSMKHaMbZsQSqaUdtSLfmeIDDAaYwfB1/HI3eYSNRKEBcJP0Z9wjJQSRytjF4kGxk7/wm6fB1POnBHs5NW6LLiPdPkixtfn5AeqQDdGvT5cdvTL/1OFW43L2q0+jJ58GH1HFWk3nraCNYzAqHJKrTtMqkJyanAfkaVtRAh1sieoc6U2gZeu94sm9pLF+jGlxfS0MgzbWpDI6tlMzzOcvwJF7OTjquyhOEGRw4GrNWRGT2789TxeG3tGKLVPF+WU189FImRFo7cYh7EHV8RNsvqZcBVpRW10iR1IYKcelu6GmSZ9U3WRN4J2hdgKqSqxjPHsOLBbMvriBwWHG0D+yGhjpOaD0HQidkbzVsP1wSg6FsFFFI0idpjxPV68fEQ6pEHojLEo9fE+xxdDjzgs1ZpdYPCQUPn+5INQZcsygEiCI0swKxDqNiyqZ9gC+q8bhB/hsYXQJDR3fJfU20K0vGorqZw6z+FKGENCJKN7miGa7O2oTrPClH6Oy+ZuqL5SSsDDQjK3iJYm+3u9fjBaRFRA4j5DU09VAVVhnygtqCa1a68uCAA6K+JZnlddxXrpXM/p5Ux8QTidDJx+1mlzewiq+0EEaPpj4bb3j0sbWlyZSwuvFeFE4zkTLw3B8JDI+xQie+XZ+DCOt3xaTbOI0ea9jBB0dNzQFnvHmId3Djp89fi9H8+P/novT979+b9RyCUjxrNKgEzi0B5Vv50evYR/XPR2A2nlGCS7NGaUeMP7FiEDq3ILxEHTcyV7erlp9evvbPnf/fOTj+8f/f2w6nTn2X3nOWzY5mm03yZo8ENSL5eTW4emVZ3gVq+auaTCzPt4i5wjMxY0VX39avTF9775x8+PP/59AOO1RmabkPI+ma9XuawBmZHgv17LkSt0wof+wBw69np8xf/Sw3tLv3y6sXp25NT7+XzV68/nZ16L04/wq9do7qrc+0Ux+DJ1E1QruaE2uRtmQIOJZJex993XQ4ZGKW+Q1baroWu6vHAJQp2wyxxcPE/cxDQI5oWAb0qO64ONRDuYh/SHm2f0bu1rLBiaASN0d8YKH+LNGoG56E2NN2LBqGIDC0aV54eP/Y5erpo5gwLjw+PJ2ypEp9TOHfLeAlr51NZfxECIVyi34vdrfYJPDs9OX370Tv9Bf4l2FHiioxNWfB/idZ06HJItg3q2AEaWq8DY1fOaoS55nQ1zeawLJqXAe7UewPi5us3HruDADczq0qooV2ljKfn7GJTEHuNPjNXsAzopwKyxOwi2mEJ1h4YLYOtmFHYLOX4XugCzzdN+RFJZ/57VoX49AZpsTEaYf2Y/hV2WZeNWzUj7AAlVvLpylDS5sGM2SNlYp2V3C/inzIJgWjD7ogTFTqnW03En7wPSJs9dlqvcKlvKJjiBB0f0EKOq8T2cvJUQJdg4jiQutJ+oHW+ivSciNmIzVT/fw09QprroyME0KH1hu0aZFVChU2xWSUsetT+Y62zZ7d8Y3aoQ2N7NpKMbBv5xiDAF2uyNSFnHI8nEoRAUis5L6LPiD8IUfNELQbhk8dMA6ZI2GN6OabyQ/r30ZPHE8HD6DpoVFZm33XwwOcwnYNkCaxKjr6FrNUgA0Tw4+PvnlovReqo12uTDgduHnkmoh4jm9cIqr9nhQdtHFKr3jJf5Y1iEyf1LQomMySfugDGgFoTXzfmAnHXRNCgg5qIxxYw6tKkFfN4c4xckawoRoK1FBc5HOMitvDjP3wogwkiwixcgghFNlql9WUs38cqbZCXRXyI79Uk2hQ1oCb0Wz8UxcJV7HMDD7EIIMbNyj8OjvBBfoM0l67W/iovYqnDu261rWYKRREtYPbsqRJZh9KrcA2ofJ6voIWIIAjAE8aOLnSK3CGePFaGDBM4xPHA0wAFoPFjbCEJ3/v04ejoyeNHx8F/Ox5A8WFX0FSZ3ueE9kmF/AK2cYXxHtK0A6MBKwqJOtVOjzZEAcAeDSfI5heoDuWlEIlUQjISLYoRIxyaUfS4sjtuZIZXGSpyn0LfG+/v+WUOYkKeehIqBoh6kVd1A//C53JhA8pAoDhsMjp5l8AXfjp7DWURfUnNI0HlqDyp1+ksq8nYWVbAcqDAYldLDc1PEImQGiRsqteizvDxYBAqbi+ZAZlAH1p0gHsyIBmKFmF2kVbpDAThZLqZQ2U1/O+o14SOWIziGCI1bLvb7/SMD1VdbiqUxqTel6Jbta5m6qhgma9gmW+B8nhCPliJOLBF5Nq3yAAgkmu9vIleXv6arS8yoFUwiCo9x1eT29vWDkMX6GZPSgTUr2rIquxub6NQIoW7Otb41ez4c/7wEstnldRuDUxXkSYS2qo6JFPQL7Q5L2AxpyngIPRMTGTw2/EUqHkyrOOIoDe+C5ZHEv8CVRzJh0M9kKOiPSZWJG9uYhvv8QVloSGsHQtFWiLnN2K0QlMdArFZHPqszVbtL+F4ArChR4m+pK3IkuA2kOio9kA7IpoZT0dI25JgSJK6Qw9BOAzLjVV7GdTkQdx+00OcTtCbcG5kEV7JIz1Bb5XXJBkTDdnGeFqWIp9EIA0Niq11TV6zRFJxXLukpPDLLaDAjt4jlqlotaKDC+nLgfbu7HWUY5gkJzM6vvSTtBj0az6l0zEySElc0MhKEN8PVjprPGw/h9ZL6/Ts7N2ZujVe+yZ6DHFI9FEjwxd5RYcH1zVb5J9jxiIclHDIEWY4XjYsmIHTAcJHOBp8ZuMdZ9nnzZnnVWyqHykD0UHbrQsNCnLAUSj6jB4kHOwCGC4DYkpOzP7eAzIefjcJj4Wj2IYaGi+SSamkQvkRt3tm4qhbJiL6XSgRR7s7B5DaVGRWo2GPFTyJ6dgzU7vDZcNZEFMnlmWO6K/fbX+sV5TsomRHYAjUfMuXy6GUGF9OXEUhgA2uAbUDqAXtGaQ71Kx354mLmUecAcxmtaaXTZVnV+JPFSodB6mJ2ORWZsdCVw2opsGhtTG6fjRExjKgga6/tae6CV4kGDDNSm+onuQkNBvQMlf2Gac6W4XWTB5fUpfJIq0Iv/AbMywKxeB6/KY34uPzTG8tYwazQ7QFw87Oai3okCMwPs9sDMYjpSNL8TVA/WBgl3j/EiUcuKf6YNWxdPIAUQQfa2Ustb9A0Xl5M9Qw6VhLxFjNvRjXhT5IP7Tuw1ojw+vZGy3VTiFgdFfof0+be4jBxMgrY4zboaEnCxgnOodHjGj2MM3Go1KQuLu/o/UF8BW7jiuzhnVsAFZEt/HnsdM+EkyJSCR5jduePBq7+H1i+Sj9IxjJkjpD3busNNbeyQoPy+oNO+GthvWx2NkvnIY23gbK1sOI+Q7Cn8s7vb6hrNp4iAqe0Tq9QQ4jnOUSPRi7TJzfooHzacRT8HUbh8fDSfhkIAdRA5Nuqh+eNEPgaY4BoLohXQZQQXgtXXvCz3PTbM8X5xBtDWV9aShTiPVUtjiO2IyoB8YsahLOQf6GfJKEieA/rTOopzyftqyJsEhyKjXbsSMADdBRa1bURXw3tjkeALrpG92umWBgsh1zq09gPShk0TH/7MMYuoc6AWksESBIAPuSV4wBOm0YQqyA62I+GB43dNzQY3IoCGwcEUP/vq7UpDemyPBWXWEUfqbFZi1R0xTS34kdZ0+wgz2srrgHddlWGa1mWy3wtfhhxx1lFz/ctxUtdNBTgBDCXgcZK/rTN1YgQDM12kEq2Ao8aqEcubBrLhSrOfnCdD71nO137M0HO0uiRd6A6G8VXDrOL5X0GNQibsWfvOcU/UhZKby08Mo1a3ZCm7XiTVpdYjgf7BsS7SrUJvDPOK2GslpgSyS91puKFg7VFbCRNcEhLl6OVnXd5hGpuQ7F/qnxDuaZ8abl/IY0hzqtBxwg3Kl5XGURkkMSjfxK/ed//qf/lyGKacFfxt6vzeThr9Vffi386OFfAvoFBVSIzYVQ88W7j89fv/4n/Hr189t3Z6cnzz+cMthz60Pql39HGFS19lETtnblQCxijwqwgkVNYaP+OkQzIimt0PljHWjHVB9DB2Gn1W2of2m79C3ruJxo9TVp49fG8qkb4bD4QB9ot9+xNDoJ0Shhitj3PZDyiv3dSJm0xMB9j8p6NQDOKlWSSQHdbGlQtinrWOAMAcu1Z49vggcxz1ch/a3wu2ttz8nQ/dumhvOesyJV3e4ZKrbkjE/rj3uGAmw+HKTKLha/07DkigAyvPYoJlSTapmBdzbirnES/YRGdHUaMOATbrS1BpP+JZ3Xdjv75mlmmOuRCQ4Z9yA4Zr5yyhw0Z8Uq690b7AjzMeAj/twzMcNZSnYHHHVnkR7EW3mVCJ5phPM+duQDjJXMe0cgg28o9Q2uX0GK+tRwMtif9FLH49zdFzMzC6UTtHZgIqx5bDI+4Widh8I0J/7aJkkUAdZy6X+OY+elYV51NYnd3Zqvo+nlIzyUobR9UIZrphL8hJotGvrQOc3Wt8T1V9HNxbGDJHBuLg5pu49KI1RAWBM1VNJOQsYpyY9AVJMRlDPYZJXXNemLdScCcWu3ee1KkHBqJMyOJTQjIUWbRbOycvT2Vhs0T9JNnS4l5QOaM/vNj6HOFfOv2h9flhV3+fpN+BNQyufF/CcMCz8pi0WO0aduxPWo31SJAyKTHT+y80bc8o2h5Uz4S9SAHNRyi7FG0PvbPPuNhm2DYV8ahn0W0I4Bbo0DpDa6lrmy5g9QhxIExNtL51O/eZF8O80bNpVNiyk9JVQnQQ47VsXiW2W/bABw5uVmCswzN9yuiDwDYIdkTnXZCEX2pOOnZvwdq6uzwf8l6xi6GRNQfIO1iOldKOHrq3QNFFENB7ccD987GzHqMvyLsx6DP9ohNJ+4Sj9TDINYRb87fkyv5HP87eDZ0z7T7sIhEsIvW6+Xvqw3gvEdzldY+JFEkb89feW4FCp2e6Bj4BwJ9AX+tXj19v2nj8QoDjv+M2ZQIkCiJmbbHYTCLISlH39RVblENIZ5VpRYoxA9co+3E9JeF4Dktoy5qHVa3qDdCkARBJYlymG65VAXk01FE7TjoSdz0xGxO6zFuuM+AzY3ey/jMRv2MS0E9kWyIEWiGLOv1oBjbqv6Il1nY/IY2ZaheNgJYp4bzBjihAlQs6EJUG3HELTBTENfQuJE7ABcoIEL2nrUrvSjW459w0jMRg9FGFPXKN/WPSCUeTWmRkGpVC+Z6lXxmDn+Rio91VXmSTRdmzabIB5dXwMQ6nR5ExP9wexq2z90yCvY8RLFSG8moRgXgDGHvM9ap0l/bvv2SkNbcZ5Y77ztLQtGruPoXq36vRwQePiO/0GkBTPriNBFOx3wmJcgo6NsJzBtyEiSb51B9xuO0UiBMQ9kPAhpCsPJqEqvu7XnGakZTaWwvszXva4ieEjLumdaFklbqgZjGWFp+D8xt5bdhjfCjhPvMoZncmIF+SmeluWS3FjMcAJievGomlfkzEWWnLLuPaKGfcLYopw0He2z2O5hS1MQ44vAuur0gXiPq/GwT9m0DVl/6HgYsAWScn2/09KZ3Lbb9BDfUMh3dTMUvxStQnOUKdihq0shvago/3xL2sKPQHj519+yG/oRaJUgN260/rHW++3dNMM/E5urmxD2/IGNFRTW19STuLNW0FuntoPHZN4d9ZJJYkZfu5ErEpzJzBHZYDscJ70zCXOMf2E5zaom3cuPk94OcQDs3nlzgVzIv4kz/4COlXDuTsS3jvH+PZlyOyD7Uxj0dvbUP8p903wxvcMy4eRnqC8zzmB7eett3rR/iv+OPh22wTCTJpSHcaAgPnQRZJf9B/GTffke0Y8wO+Rci9oz3SaeVndxqy1ONHz4cFNgENV/IUv6J++9cDhOolNtBSmrOekzS1GAGj3HLMe5T2+gTV42RD/IXptUwmhftcatdUsLY3hZ/MnMpZTLRFth1TM4RuBuNP11OEnX+kNr2eUcgxYbuIcn3deSwokLTVDuke6cqHuwqds4kjFJYriODjdzN0cbAsXQITcuNaQVC0LzzXn5Y2fgdAQM2/MvsksYGLHLUbM8B4E4qstFg9FRh8fBeDBh2c4PohnmLYFWyXncwdyGVrNycYxV2bH2ScuvPaeux/kkIHcYM519tHxr1mEnsIPa7FLpvRvCWh+1vStMx2VbnDCp4e6NCbuBLMN+Vq9vQLi07SXFpDD/IvkzyfYItRn3ona28i8cjWcdE91UhBKu4GQux2ztmKUskEi+vOZazCeuo3qzWOSfOYkFe6WFipRFYTsJJDxOc8pj4eRQnufpeVHWTT7bnUdZcib3JRw2te+fUFlau19eZTJ/qzvTK78w47Apll0XuD+WXZmJ2r8pv/Kzwd3JlZ0y98n2SzX2EVxob07+vpRL3S4Np6wIdC5en2eBlpY9SZ1xaDrfsky7PxPzk8G98jCbUT4ZuIMzkXk1Xk/hiWbbpGSmxNJkE2oFaHY9yW7vTsbswIoTe2sTMm9snmgOJO5k+uVDDtgpMRFNPmcA1l6+BPGXeTE3IS+Ov+8qLzZNVsfffrdLDWf54b4A0Z5siL1ZE3XjvOnMO4kuBUptMFGBlh/i40FYz0pUq+rM2JgzU0mI8/MG9VwNpmyqbni1cH2F8WF6R9kiMeyIcBLKEitKSanjwEjnU3Nk0Fb2Ue47Vk4O8GcDZSJVr4G/3IWt9m6vbCxnrRdBwripsK6Bk9HrMTzoLIA+lW6q9J4uMfLqEHOYsv+7twLO2uPEHb25znUs1YaS41/pPNVuCnTnYOis1CDG8nHYsYs/HO+2vh1Kph4UaokdByhaznmgU+CGyzrHpnQHDKI/xH1I5oPsqBTa2UYL/FrZSRBCugbJdnFg8dh02X79w3FvXndaRd093lgjicEJ5ugiAuyWTjFdAhPVGRDYbx8HI+bQgCJtUoypbr3l1Hr2E+bmw893+cVLjCX5kbcC9xErwNJoX3eNGigLFeAKgD3hdUXM72MjbIAXpUJBzl+7fANmxGbUkBCPchydrYtzhtzUt4/Dr4ytOPC0mnRHzW1zVagknmJXlbqaHe0Ivwgcc7/13L8r4gBH2UasatjRNCpHpwzrgdYOI2dYBMzVNJ9JjCRbMQHzZvMbV7mMAjf5UNLtLOyqwsEyNlE1csVO1moziO4x5m57PlB7Jnd9QghLDX31bCDwrmXTw1a6eo1HWL4lRzInBb7ajX9F2dR3Q0MoaA2wmojNSDabckOOQw1yehLeiFjmqR6gvT9KIasNYznPKopejXdckzKSpIsUGUAxFWMTsa8o6QQ6CZngpnU7GL/WPktEj6DA//3f/0e1UxESt1Bea35hqJNTuEkmeqM+esM+7sw9welCJf3SHIPGTeO0Yq3EbfJRmEDnu14eG80AZZ3V7MPalKvDWZ95mTFKJh5Z20/MMeO8LhQFIl7WjU3ryHuiP8AgrUeTfJOgcGKEVLoky5HxBjPpQA50xHhs81xzpJAJS6cW6B9Dh508XiZUJbBJxDYrv+nLs8YRtyRhcY3e5BvbrNTWm3BP1mXkJsSZORa6LjLivugXQE3xYOQeefELHYx0ZjHKrnMZD8QVVHsv6oRCI0nKowPbxQCmffmUCWaRhdWJfhBQ0VRkXZWJJVYhS7zmtbZ2EwzoCXaDyH2LhHf3ZOOddScmMb+klIB+wvY7onUB5k3T2e5w4wXU6FiZte4onrcIpsNgB5ryxl3K23YmMW2EEovZnRpNo9bmUUpvf484CRsfEUhUtOAhfYQ0tthd98dYmLCHTwfDKQKI1LmD+YJPP8btMkMNQOaqDrokUIOTw1AZmBqZHv9IGFav5bSDzYYOLuvLeH27P6ygsz3t5K0255MdKb1qIduQ2UlYEItiHsFqPTpuu+9rj4mvCvhsjdNxk295+OmQQCc2UlvNR93SJqXXlk582wdfn5hIq933+Id02KcdEwh6MdiBjeZx+Op3m+bd4g1lZeaMnzTAzsZh5mZYw6Rc6ATOjjFL0R2VDJeMF2p9cogzc0jdNebdSKdVTupECWjx+hzQ8dpM6BEDprnHyHuJyRroOk16MQK2PC3wUhxWHhzpC888PDg5XuxEl2aifgB6ci976bUkuv0bY6K1LwatwCe932JavGeoQc+2cM7SHSdFRzjx34RzIOuS7QzEXdf1drBTby7W/ZE3UgXKIK0ApIXpnECInS03c1xazq/o5JnBa58uMAUbqUIdo841XjgCG7G8AVqYLxvYOEJbNsKJO9HIEUDnsCkPM2wmX2eYtNpD155idhMpZ2jdbJRs4b2DXeREDjZk4Y7iEs2AfAFmoDVrTxnsdKSDzUcqvEH78dExEY9dWWeZ0A2EXmxDyI/Gote59oUt65Lj7olbEy0kVco3lwKaOuqa2PDWs8h7JdnDUDhYlnBWOYsc96qVJhLfpzTH3J6a8QnqSkj3GDGfjnZ7mMLRnYeskfi0OGNmwbY+0vZ3I18g3gRKTLin7xZOgNyKbI7sJhVpAaJNXOLY8LjDNpdnOGfD56H8xXMxdXAaW+n5lAleAuQyLdNq/grPXLVZkzdVu5tcf3I5SvSQ85wv0B4tVFnAXDcVXpwCsh/vYi86Vtsck9tRy0GBUkqOaBt3R11tjVswlh7yXXhxezyozP06tNcZcitOi3mEbldkKzKcdm1gXBPkVjCaodJOoKjhvzu2JxvZprvkSu5ahSZrp8jQWpsn0VEAhaFC8sVcYeR9oGpDWBa3FYU3gKLCc1MQOUR1qqaDTcnidha1RW2+L3gGx8VvKcrrUCgn59akwvbepVYeIY4OuqSLttB7Gii68QZJZwCM6ewmoS7QIWbGgdbwAwnQgt4nNQcMIKez2NSc8gba+XyXVo81Yfc3IWEw7VcakEbaBYvTN7qrQ1SOYi2ot30qaXZSmDGVNLwJYl3WRmudkdbusFoftSY6fAc3B+UR2iS+PnqPZlPUM18wdVq2GrpqkNbsvj5XKV9mwpaZXQYrXhd9Cw7IjoxdD3W9/deCOjyjE6SytQG23QexaRhPpqv0ePX25N2b969PP55iPjarp1BHrDfjC9BolzToIn2Zb2ZEH0ZCUpArIkNLPK660+QkSM4yaN3GZHQTjxECx+5nx+5ILU5GlIx9zIk7WiVtaEinig02ouehM0pz4kI+ceHiOEziO86hf0Oua6FYXNlqGP6eVWUyz8WvfkAIm05p3D22vfUfjQ2bgzktnoTDSdtrghzkcJsmeDeUzvthiKymzDTFIJarb1kbihputA7BEiS0e3gPIHsocHG6MpC2Uw3bOEnGSprfGUxwcQyCLrl2FOsIE7/7i+OA1VLK6IKxRkSMsm9vDAhdLhTOx2aFSu7tyZNfe2u9ULmMf7mXNWrdUR+jhl9oBYdflNkwPTjzAh1H0N8Ct1F/5Cf+YuezOMY3bIrNeW/aWUgkHPGWh8GYnLn9oXrNV/K2/NrkwvOXOuHV4VbCKxu07s1LAtAbtN1SOKpGcRjAB1zcy9NfTs+YEkWEVJxj62ZLvxy6kNJJ/mGS/RbK2XFna83tYJi9Iziwfp1Dcig7yxD8mS+UUCpWWY+Apub/2GQgt2C/GGNpGEe+aKPc1J6A4ZEJsK24vYjgG38SmvZlbWudYrZ9O5sROEaObMLVQ8lYppNL8njxgkX64VjE2W2UqLs1NiRcrEPpxZ78AZeeneXqjloapk+8PwhrG+SvBfeyHOPxfQPsa3e+ySiqrGVSntVXdxFy8gTZYU3GSe2+V9s6KPS7fUj67/huHkc7d8r9CWjSMI4yFDErTbmeMSsnY7trqHEmkbQwGFQoUOR7NhDUtaOSjahW0g0ZlqAX4+u4y55EIdxsUcdiLcaCp6Daqk1tYnFSbW0xEvtsLv0JaB0iJKrIvblkjQHx69J9WaWf04vbZpvdyZ18f+6oeVO+3HKDFeaSirvSOkLjuD0QS5qpIiDT87K6iX3XUmrTDIjYrYwqQJy/OwpDm7uYRtU2jJjLTTgehYVidbdyodOKzFFCQg22TjBUN5GLgnixqJzArBlpWSUsunBgCxZP5EzNlcTm4EubE8i5x4o5snsllDNZxId7F198GB0/4WHnshk9YTRY0w/O/YQKd9k0+CK/bKqibv4HvHNC+pIL4mm1tE+mTQbZukirb+C2uuM3LE3biG+Qa6t1w5o55moEePfnvtCrISFUOpIXhYkhjoT8GB1rvv3ing3dDq8T9EJEAFgCZdYOeQTk1dStJql08dSBg1F8XXYXyQMioYyjbvsiUdQX0hV5SpI1G71dFUO16AVs9t/p0V+E5BxG2J3TOXO348EkEHoqd+KhepLMfO5LliGpRmBYEKCy+MKrL7KssZnkDetQDx06zXN1EPn9SL0Q9rpLwcWtbVOwJT6hnLHL9Carap9Oflg42USOf4iLH0hdxa7tYj+TBE5VRBX3ZQOgAhzxp+VNEgq5OURgMB3sOxii/Mjia3JepXOmnltVnDzKe2ujfoErT/HaFNtA7xTGh8VwMhQiRr1Rrft04Thnw0pSNYBcvkVPRoq3FwFSIh8+RSEb6DOX2MbhjGxWIEAj47z0g33LxIJzayCB3GL7dQ0GtwwRQNthJkWWyAJZzsYBkfhYOxCu82XZKOPl+GQQOn5MdZOtoexgMAjMfdZEuufUzSH0g+oK46nH5q8ZeqxoFw/bmod2SDaP8l3WOZsJ5t6zp0fPnnrkXEVcJrN7Hq0wJacvF801cN4iUdDIWzqSA53CxnMYbMnRZ5sB2R85ZeGDtJhxCHyWKSO3Yrsej+G2SuvfEd0UfqSQn4r/QtfPq/MN8nq1/nIiGd1CzDJ2Ui6XKYzn74AE33NearRck6dbz6giDiiqkk2TL80YMdcvAYHdlP1KusUxi8IH3Uisxnrf7WXf+zLndOHLOA92P/Q6SJ5YeCKtNKAn49LoeBXit/Os0r7TBDHm5kiBeUrI3JuE5uSixDRGVA5HRgWDAwYTolwkZ/gsaNDbhAqLPtF10uLONE9vy2Ob9tpsmMZWu5aN/wONO5XaPdhsXzwFy9lK3hwZguXTeV1eAu1/WzYv0THC8Z+1x4rOLcrFsr6ICNh6DzgbKi3h3H7IMg851jenpJEzx/XwUJDAYeeURt4LVgvUmykgkgaQk+d6tkVmOxwdr5lbMILJOB9kagCc0sT9FcPAYhGSiu/y9NdN3x6YGwY03mJAEtdboIalltCyIxSoibPhhcsr9BeHk4fm6IOtE0G5WbfOD8YPHcO/sHezLOfgZFqK4Oj4KV3f0lQkAN01Bap0O7q6V+ErmiqJFVj+z1c6CZLPL/5JL4I/6wXsOWYa9TlAe2RdHSWrEuDqZbp2DzLAPR1Io13nyf4QP3vadzUSHWIC0hkx1YgpOEqfzrUBZGWsLNRLtMHEZ34glB49fFgxzL2Fyg5aPsALeP1XOD1lRUTQ0qAa7w5DVQQrQ9Bi00rzefb8Z3ughMdme6q+xp3B1O79kXaANax9y5PZTAr5aykJjCZO7JEiIiDWSMsTBNoXOlpdYvgSMBd0Ew2xO4QUkvJS+2HeN4hWht8f0xoceF8dKXtXw4Y9i/ewyO7sRXNm6rVdtcV7l/4IRCSiR96GB/2FAYILb7lTW2TVqrpVTuMuFLb+NedsWTKH+oNI2+O0LuUC1wE9W5ezC7wigIZBzAJJ9inqgB9nh9+xf7T2xMZYGEW3MyQ1wAW0ESpkcXNUZqH6dbVZsj8zfuGra5J5NktByo4Gx9SJ4zuNKSlbaA/7PYeyijAgxhd1kGJrZowmlQACOYWbbQ73+Hyjz3O/nzMhI+0WYiwRWs/YvWrpuTjUCuyidVZqsknCXLDUfn3gSa4sxL3IBKIhi52yiK2LyONSPpDScTton70TKAiSY+owIEoXhN2ScibU0dzPQhVNSeAcUbepW83Ze7qKTdFxPhlhidgJQbZ6mspxeuMI6nuFI2OGAdIiUto5ubVuIlajiJKXYy+iwjGriMV7XZX18mpv5fDeqFFuGh50fIk58QkIQQ1MYo0OSr/ThUIZ0SvjjLAVAYYWBeHydWA623oNDZim1Ug3wFSp9vCWZ4rRQq+GoyXe8sTXT0UaUtB35nnzE/kS+x0xoh9AAnPl654QaayFKdAJkU2z85xhJEyrc7r3s8nIE7Eql+HDh5fX+Dro86XQs3OcO+0Vz3x967KckttCttaXO9N7BAo+xTwIgmUH+ToI1Hwi7Nsnpcfm5bhfip84k8Ze/9Cc84XX74XyILZUebhjZb5iRbpj1Xf33D1SUqjpgdCdtK2Ba8/TP75HRCzkLf3WtoCtBviKH7kRqP0p7ja7a83o69eA0s5h7rvDmfy4HTfuPr9MMzL26k+0B/a+MQbGPxk2ClHGZjmnHMiEMEadD87cy7UosEx+ASlr4WJZnu8ECbpyTxJ13OfoioZg/3pLoZgCTGFzv9w6UOrYwrHwv+FY6dPfc7CcT3etvgxP2I5s57j05O5eNAcrGZvdvdZNq2G6S4fw8C+iXpctYh+yfQOypV23WYfbqskGZx4P5crurfUlMRRHF2/pu3xJDoUXp+zqJAjpfjvC/cyKxnjBl8OFxsSEupxkLIwkek1ISkahYIYvjb8Nna8k/jsfvw97WVYRxb8NF+vjpywdoafTFU2VGdNY86XyFstv8aNyI3qnhrztrXHAIV4Ja4k5Ld9jPMHnGg3WFBoOXHsMTDuyh4l+YAM/VI3HZA2jNJfTDC/MJDth2iDl4OmIgweI+k6J2PGVOKcQjyrJayjQoLVFL4NZMTt4c4Ve/7eEoRZkrA2lQ4MCVYqsODFXtzQLHCx0hzCAue2Rhg8kC5GkCvXXNluH668jCkx/HbEhExMtryPHiB1B55K7RXsljQfhcfh4EsJSV+hUyu2pLTenW5Eyafr015cIKRIuaVZ0Thns5syox212nvkXgpPeAsTFAPM4Y/cJ9v+xXDauOe3zTBTE8Q5tsW+rcDK+BOC5yeHkJ+Ui/p6j9DDnqsZB8hzOhH+EdWmxlwEHM7ODS9yjWvZ3HuagLT25nzr2do7EIHHIhOX1xtFZW8EdAXWyY6wg99nzlZMiO7hOT0s8CnWdLbT2Y9z+ZHkVRME87LZMN+rgZTjPuKvsrSuxfKQIpsXDtwHFDXEnePg5CRF/GtnEOfTN0Y9wgf44ho7AGnbVGyJB75w2iPaIFlzy5qgU2tUcFGMLhbgaeX3RjhLjGNa9LvScyWxTuWYdGq1141faBUoTGvNJe0W3I4olRrbtXt2GHh30YCeg/amJhf4DTtV7nfv/gFs/efNzD2zAwdCXunGCGaw20W7NPRz1v9JFvwvWht3QXvr38cBvu8G3HeBHmPhFIIvjUF1Peq0+wbYO/h+I4kdr'))
expected_runtime_sha256 = '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06'
assert hashlib.sha256(patch_source).hexdigest() == expected_runtime_sha256
manifest_path = PROJECT / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
if manifest['files']['src/fever_colab_runtime.py'] not in ['f2ca840db6e6d871bb42a1bd3ab84ef77613d7946541add86c407b7c4b26ad55', '902e8bafd72767546bc006b9fbbc6169f447bc40b4fb4bff3e9aad9f32afef06']:
    raise RuntimeError('Unexpected bundle revision; use the original supplied ZIP.')
(PROJECT / 'src/fever_colab_runtime.py').write_bytes(patch_source)
manifest['files']['src/fever_colab_runtime.py'] = expected_runtime_sha256
manifest['notebook_patch'] = {'revision': 'colab-debug-v5', 'runtime_sha256': expected_runtime_sha256}
manifest_path.write_text(json.dumps(manifest, indent=2))
importlib.invalidate_caches()
fever = importlib.reload(fever)
assert fever.RUNTIME_REVISION == 'colab-debug-v5'
EVAL_ROWS, EVAL_LABELS = fever.verify_bundle(PROJECT)
print('Debug runtime ready. The original ZIP and earlier experiment results were preserved.')


## 4. Experiment settings
These settings are recorded and checked on resume. Changing the model, prompt, token budget, source code or package/GPU identity requires a **new experiment name**.

- **Qwen:** Qwen2.5-3B-Instruct in 4-bit NF4, the first setup test.
- **Llama:** Llama-3.2-3B-Instruct in 4-bit NF4. You must obtain access on its model page yourself and add `HF_TOKEN` in Colab Secrets. This is a new 3B model, **not** the earlier Ollama Llama 3.1 8B.
- **RoBERTa:** the exact existing project checkpoint, with 512-token inputs.
- Only one classifier is loaded at a time. RAG contexts are shared between conditions; different tokenizers and RoBERTa's shorter context remain limitations.
- This HF generator uses ordinary greedy decoding, not Ollama's JSON-schema constrained decoder. The parser accepts plain JSON or one complete Markdown JSON fence. It derives the overall label from validated parts, records any missing model label, and preserves the exact response. Malformed, truncated or invalid-citation outputs remain errors; no content is repaired. The 512-token output cap is a declared new setting.

In [ ]:
EXPERIMENT_NAME = "colab_full_dev_debug_v5" # @param {type:"string"}
MODEL_FAMILY = "qwen" # @param ["qwen", "llama", "roberta"]
ALLOW_MODEL_DOWNLOADS = True # @param {type:"boolean"}
SESSION_MINUTES = 45 # @param {type:"integer"}
MAX_NEW_TOKENS = 512 # @param {type:"integer"}
ROBERTA_MAX_LENGTH = 512 # @param [256, 512] {type:"raw"}
# Leave empty for the included original checkpoint. A completed fine-tuned model may be selected later.
ROBERTA_CHECKPOINT_OVERRIDE = "" # @param {type:"string"}
RUN_ROOT = DRIVE_PROJECT / "runs" / EXPERIMENT_NAME
previous_run = Path("/content/drive/MyDrive/FEVER_runs") / EXPERIMENT_NAME
if previous_run.is_dir() and not RUN_ROOT.exists():
    RUN_ROOT = previous_run  # Resume an existing run without moving or replacing it.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
if not 1 <= SESSION_MINUTES <= 300:
    raise ValueError("Use a bounded session between 1 and 300 minutes.")
if not 1 <= MAX_NEW_TOKENS <= 1024:
    raise ValueError("Use 1–1024 output tokens; freeze the choice before the full evaluation.")
if ROBERTA_MAX_LENGTH not in (256, 512):
    raise ValueError("RoBERTa supports 256 or 512 here, not arbitrarily long article input.")
print("Results saved to:", RUN_ROOT)

## 5. Check model access and report download sizes
This reads model metadata; it does not download model weights. The listed weight sizes are disk/network downloads **before** runtime quantization. Four-bit loading lowers GPU memory requirements, not necessarily network transfer.

If Llama access fails, use Qwen for the setup check while requesting Llama access separately. No token is written to the experiment protocol.

In [ ]:
EMBEDDING = fever.remote_metadata("sentence-transformers/all-MiniLM-L6-v2")
if MODEL_FAMILY == "roberta":
    MODEL_METADATA = {"path": ROBERTA_CHECKPOINT_OVERRIDE or str(PROJECT / "checkpoint/roberta_evidence")}
    CONDITION = "roberta_" + ("selected_checkpoint" if ROBERTA_CHECKPOINT_OVERRIDE else "existing") + "_rag_" + str(ROBERTA_MAX_LENGTH)
    print("RoBERTa uses the uploaded checkpoint; no new classifier weights download.")
else:
    MODEL_METADATA = fever.remote_metadata(fever.MODELS[MODEL_FAMILY])
    CONDITION = MODEL_FAMILY + "_3b_nf4_parts"
print("Condition:", CONDITION)
print("Full evaluation target:", len(EVAL_ROWS), "claims. No evaluation has started.")

## 6. Run one engineering check first
The one-claim check is optional and off because it has already been completed. Section 6b runs the approved 90-claim diagnostic. This check uses **one training-pilot claim**, not a reduced test set. Its article URLs come from that example's retrieved pages; full evaluation uses the exported original top-ten ranking.

Inspect the full prompt, actual article passages and raw output. A successful check verifies execution, not model accuracy. Nothing from this check is pooled into the 19,998-claim result.

In [ ]:
RUN_SMOKE_CHECK = False # @param {type:"boolean"}
if not RUN_SMOKE_CHECK:
    print("Not started. Enable RUN_SMOKE_CHECK after setup.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Read section 5, enable ALLOW_MODEL_DOWNLOADS, rerun section 4 and section 5, then rerun this cell.")
else:
    from urllib.parse import quote
    import time
    example = fever.read_rows(PROJECT / "data/train_pilot.jsonl")[0]
    pages = list(dict.fromkeys(e["page_id"] for e in example["evidence"]))
    from fetch_wikipedia_articles import page_url
    example = {"id": example["id"], "claim": example["claim"],
               "ranked_articles": [{"rank": i + 1, "url": page_url(p)} for i, p in enumerate(pages)]}
    encoder = fever.ResidentEmbedder(EMBEDDING)
    verifier = None
    try:
        context = fever.context_for(example, PROJECT, RUN_ROOT / "software_check", encoder, EMBEDDING)
        print("Claim:", example["claim"])
        print("Evidence status:", context["status"])
        if context["status"] != "READY":
            raise RuntimeError("Wikipedia preparation failed. Inspect saved error before any full run: " + str(context.get("error")))
        verifier = fever.RobertaVerifier(MODEL_METADATA["path"], ROBERTA_MAX_LENGTH) if MODEL_FAMILY == "roberta" else fever.CausalVerifier(MODEL_METADATA, PROJECT)
        result = verifier.predict(example, context, max_new_tokens=MAX_NEW_TOKENS)
        fever.write_once(RUN_ROOT / "software_check" / (CONDITION + "_" + str(time.time_ns()) + ".json"), result)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("Supplied evidence:", json.dumps(context["supplied_evidence"], indent=2, ensure_ascii=False))
    finally:
        if verifier is not None:
            verifier.close()
        del encoder
        fever.release()

## 6b. Diagnose 90 development claims — enabled
This fixed sample has 30 claims per class, selected from the original validation partition, excluding original-training, official-development and software-pilot text overlaps. It is a development diagnostic, not the final 19,998-claim evaluation.

The model receives only claims and ranked article URLs, then the unchanged full-article RAG context. Reference labels and FEVER annotations are used only for scoring/review. No prompt tuning, example-specific rules or model training occurs here.

Watch the live progress. Outputs resume from Drive after a session pauses. The report includes accuracy, macro-F1, per-class precision/recall/F1, confusion matrix and a CSV linking every prediction to its exact prompt/response and supplied evidence. Incorrect labels are marked for human review: a wrong answer alone cannot distinguish inadequate evidence from faulty reasoning. FEVER evidence references are historical; live Wikipedia may differ.

In [ ]:
DIAGNOSTIC_MANIFEST = {'scope': '90-claim development diagnostic, not final evaluation', 'class_counts': {'REFUTES': 30, 'SUPPORTS': 30, 'NOT ENOUGH INFO': 30}, 'selection': 'seed 42; original validation; excludes normalized official-dev/training/pilot overlaps', 'files': {'eval_inputs.jsonl': '24ecf9f1abdb22496aff3921c06741e343ac79034a20d64327c69ad40010640f', 'eval_labels.json': '57da5df2fc28b66c560f8a05085049b3a851c53622f6b20b8c8077cbea15b36d', 'references.json': '36aea424a1a44fff0de5be9783266ac59beb10ccd6cb304315dfadad5f2d0555'}}
DIAGNOSTIC_FILES = {'eval_inputs.jsonl': '{"id": 147861, "claim": "The Simpsons series will be extended to 2020.", "ranked_articles": [{"rank": 1, "page_id": "Tokyu_2020_series", "url": "https://en.wikipedia.org/wiki/Tokyu_2020_series"}, {"rank": 2, "page_id": "2020_-LRB-mini-series-RRB-", "url": "https://en.wikipedia.org/wiki/2020_%28mini-series%29"}, {"rank": 3, "page_id": "2020_Casting", "url": "https://en.wikipedia.org/wiki/2020_Casting"}, {"rank": 4, "page_id": "2020_Vision", "url": "https://en.wikipedia.org/wiki/2020_Vision"}, {"rank": 5, "page_id": "2020_Visions", "url": "https://en.wikipedia.org/wiki/2020_Visions"}, {"rank": 6, "page_id": "2020_group", "url": "https://en.wikipedia.org/wiki/2020_group"}, {"rank": 7, "page_id": "Behrang_2020", "url": "https://en.wikipedia.org/wiki/Behrang_2020"}, {"rank": 8, "page_id": "Cyberpunk_2020", "url": "https://en.wikipedia.org/wiki/Cyberpunk_2020"}, {"rank": 9, "page_id": "Enterprise_2020", "url": "https://en.wikipedia.org/wiki/Enterprise_2020"}, {"rank": 10, "page_id": "Europe_2020", "url": "https://en.wikipedia.org/wiki/Europe_2020"}]}\n{"id": 36376, "claim": "Robert Redford was a guest star on the legal drama Perry Mason.", "ranked_articles": [{"rank": 1, "page_id": "Legal_drama", "url": "https://en.wikipedia.org/wiki/Legal_drama"}, {"rank": 2, "page_id": "Robert_Redford", "url": "https://en.wikipedia.org/wiki/Robert_Redford"}, {"rank": 3, "page_id": "Robert_Guest", "url": "https://en.wikipedia.org/wiki/Robert_Guest"}, {"rank": 4, "page_id": "Robert_Redford_filmography", "url": "https://en.wikipedia.org/wiki/Robert_Redford_filmography"}, {"rank": 5, "page_id": "Star_Drama_Presents", "url": "https://en.wikipedia.org/wiki/Star_Drama_Presents"}, {"rank": 6, "page_id": "Redford", "url": "https://en.wikipedia.org/wiki/Redford"}, {"rank": 7, "page_id": "Alison_Redford", "url": "https://en.wikipedia.org/wiki/Alison_Redford"}, {"rank": 8, "page_id": "Amy_Redford", "url": "https://en.wikipedia.org/wiki/Amy_Redford"}, {"rank": 9, "page_id": "Angus_Redford", "url": "https://en.wikipedia.org/wiki/Angus_Redford"}, {"rank": 10, "page_id": "Blair_Redford", "url": "https://en.wikipedia.org/wiki/Blair_Redford"}]}\n{"id": 6093, "claim": "Tom Brady played for a University.", "ranked_articles": [{"rank": 1, "page_id": "Tom_Brady", "url": "https://en.wikipedia.org/wiki/Tom_Brady"}, {"rank": 2, "page_id": "Played", "url": "https://en.wikipedia.org/wiki/Played"}, {"rank": 3, "page_id": "Tom_Brady–Peyton_Manning_rivalry", "url": "https://en.wikipedia.org/wiki/Tom_Brady%E2%80%93Peyton_Manning_rivalry"}, {"rank": 4, "page_id": "Brady_v_Brady", "url": "https://en.wikipedia.org/wiki/Brady_v_Brady"}, {"rank": 5, "page_id": "Games_played", "url": "https://en.wikipedia.org/wiki/Games_played"}, {"rank": 6, "page_id": "Getting_Played", "url": "https://en.wikipedia.org/wiki/Getting_Played"}, {"rank": 7, "page_id": "Brady", "url": "https://en.wikipedia.org/wiki/Brady"}, {"rank": 8, "page_id": "Tom_Brady_-LRB-Australian_footballer-RRB-", "url": "https://en.wikipedia.org/wiki/Tom_Brady_%28Australian_footballer%29"}, {"rank": 9, "page_id": "Tom_Brady_-LRB-film_director-RRB-", "url": "https://en.wikipedia.org/wiki/Tom_Brady_%28film_director%29"}, {"rank": 10, "page_id": "Tom_Brady_-LRB-rugby_union-RRB-", "url": "https://en.wikipedia.org/wiki/Tom_Brady_%28rugby_union%29"}]}\n{"id": 23390, "claim": "Ron Perlman was born in a station wagon.", "ranked_articles": [{"rank": 1, "page_id": "Ron_Perlman", "url": "https://en.wikipedia.org/wiki/Ron_Perlman"}, {"rank": 2, "page_id": "Station_wagon", "url": "https://en.wikipedia.org/wiki/Station_wagon"}, {"rank": 3, "page_id": "Perlman", "url": "https://en.wikipedia.org/wiki/Perlman"}, {"rank": 4, "page_id": "Wagon", "url": "https://en.wikipedia.org/wiki/Wagon"}, {"rank": 5, "page_id": "Alberto_Perlman", "url": "https://en.wikipedia.org/wiki/Alberto_Perlman"}, {"rank": 6, "page_id": "Andrew_Perlman", "url": "https://en.wikipedia.org/wiki/Andrew_Perlman"}, {"rank": 7, "page_id": "Daniel_Perlman", "url": "https://en.wikipedia.org/wiki/Daniel_Perlman"}, {"rank": 8, "page_id": "Doug_Perlman", "url": "https://en.wikipedia.org/wiki/Doug_Perlman"}, {"rank": 9, "page_id": "Elliot_Perlman", "url": "https://en.wikipedia.org/wiki/Elliot_Perlman"}, {"rank": 10, "page_id": "Fredy_Perlman", "url": "https://en.wikipedia.org/wiki/Fredy_Perlman"}]}\n{"id": 44625, "claim": "My Summer of Love was directed by a hedgehog.", "ranked_articles": [{"rank": 1, "page_id": "Hans_My_Hedgehog", "url": "https://en.wikipedia.org/wiki/Hans_My_Hedgehog"}, {"rank": 2, "page_id": "Farewell_My_Summer_Love", "url": "https://en.wikipedia.org/wiki/Farewell_My_Summer_Love"}, {"rank": 3, "page_id": "My_Summer_of_Love", "url": "https://en.wikipedia.org/wiki/My_Summer_of_Love"}, {"rank": 4, "page_id": "My_Love,_My_Love", "url": "https://en.wikipedia.org/wiki/My_Love%2C_My_Love"}, {"rank": 5, "page_id": "Love,_My_Love", "url": "https://en.wikipedia.org/wiki/Love%2C_My_Love"}, {"rank": 6, "page_id": "Hedgehog", "url": "https://en.wikipedia.org/wiki/Hedgehog"}, {"rank": 7, "page_id": "My_Love", "url": "https://en.wikipedia.org/wiki/My_Love"}, {"rank": 8, "page_id": "Summer_Love_Love", "url": "https://en.wikipedia.org/wiki/Summer_Love_Love"}, {"rank": 9, "page_id": "Summer_Love", "url": "https://en.wikipedia.org/wiki/Summer_Love"}, {"rank": 10, "page_id": "My_Family_My_Love", "url": "https://en.wikipedia.org/wiki/My_Family_My_Love"}]}\n{"id": 156586, "claim": "Michael Keaton will have a role in a Spider-Man movie.", "ranked_articles": [{"rank": 1, "page_id": "Michael_Keaton", "url": "https://en.wikipedia.org/wiki/Michael_Keaton"}, {"rank": 2, "page_id": "Spider-Man", "url": "https://en.wikipedia.org/wiki/Spider-Man"}, {"rank": 3, "page_id": "Keaton", "url": "https://en.wikipedia.org/wiki/Keaton"}, {"rank": 4, "page_id": "Astonishing_Spider-Man", "url": "https://en.wikipedia.org/wiki/Astonishing_Spider-Man"}, {"rank": 5, "page_id": "Avenging_Spider-Man", "url": "https://en.wikipedia.org/wiki/Avenging_Spider-Man"}, {"rank": 6, "page_id": "Lego_Spider-Man", "url": "https://en.wikipedia.org/wiki/Lego_Spider-Man"}, {"rank": 7, "page_id": "Spider-Man_2", "url": "https://en.wikipedia.org/wiki/Spider-Man_2"}, {"rank": 8, "page_id": "Spider-Man_2099", "url": "https://en.wikipedia.org/wiki/Spider-Man_2099"}, {"rank": 9, "page_id": "Spider-Man_3", "url": "https://en.wikipedia.org/wiki/Spider-Man_3"}, {"rank": 10, "page_id": "Spider-Man_Classics", "url": "https://en.wikipedia.org/wiki/Spider-Man_Classics"}]}\n{"id": 72212, "claim": "Wyatt Earp has a sister named James.", "ranked_articles": [{"rank": 1, "page_id": "Wyatt_Earp", "url": "https://en.wikipedia.org/wiki/Wyatt_Earp"}, {"rank": 2, "page_id": "ASV_Wyatt_Earp", "url": "https://en.wikipedia.org/wiki/ASV_Wyatt_Earp"}, {"rank": 3, "page_id": "HMAS_Wyatt_Earp", "url": "https://en.wikipedia.org/wiki/HMAS_Wyatt_Earp"}, {"rank": 4, "page_id": "Mount_Wyatt_Earp", "url": "https://en.wikipedia.org/wiki/Mount_Wyatt_Earp"}, {"rank": 5, "page_id": "Wyatt_Earp_Islands", "url": "https://en.wikipedia.org/wiki/Wyatt_Earp_Islands"}, {"rank": 6, "page_id": "James_Earp", "url": "https://en.wikipedia.org/wiki/James_Earp"}, {"rank": 7, "page_id": "I_Married_Wyatt_Earp", "url": "https://en.wikipedia.org/wiki/I_Married_Wyatt_Earp"}, {"rank": 8, "page_id": "Wyatt_Earp\'s_Revenge", "url": "https://en.wikipedia.org/wiki/Wyatt_Earp%27s_Revenge"}, {"rank": 9, "page_id": "James_Wyatt", "url": "https://en.wikipedia.org/wiki/James_Wyatt"}, {"rank": 10, "page_id": "Wyatt_Earp\'s_Old_West", "url": "https://en.wikipedia.org/wiki/Wyatt_Earp%27s_Old_West"}]}\n{"id": 197148, "claim": "O. J. Simpson won the trophy.", "ranked_articles": [{"rank": 1, "page_id": "Won_James_Won", "url": "https://en.wikipedia.org/wiki/Won_James_Won"}, {"rank": 2, "page_id": "Won", "url": "https://en.wikipedia.org/wiki/Won"}, {"rank": 3, "page_id": "Simpson", "url": "https://en.wikipedia.org/wiki/Simpson"}, {"rank": 4, "page_id": "Trophy", "url": "https://en.wikipedia.org/wiki/Trophy"}, {"rank": 5, "page_id": "2000_Won", "url": "https://en.wikipedia.org/wiki/2000_Won"}, {"rank": 6, "page_id": "Andula_Won", "url": "https://en.wikipedia.org/wiki/Andula_Won"}, {"rank": 7, "page_id": "Chae-won", "url": "https://en.wikipedia.org/wiki/Chae-won"}, {"rank": 8, "page_id": "Chi-won", "url": "https://en.wikipedia.org/wiki/Chi-won"}, {"rank": 9, "page_id": "Choe_Won", "url": "https://en.wikipedia.org/wiki/Choe_Won"}, {"rank": 10, "page_id": "Do-won", "url": "https://en.wikipedia.org/wiki/Do-won"}]}\n{"id": 174565, "claim": "Bill Black has made no contributions to music.", "ranked_articles": [{"rank": 1, "page_id": "Bill_Black", "url": "https://en.wikipedia.org/wiki/Bill_Black"}, {"rank": 2, "page_id": "Black_Bill", "url": "https://en.wikipedia.org/wiki/Black_Bill"}, {"rank": 3, "page_id": "Music_Made_Addictz", "url": "https://en.wikipedia.org/wiki/Music_Made_Addictz"}, {"rank": 4, "page_id": "Black_music", "url": "https://en.wikipedia.org/wiki/Black_music"}, {"rank": 5, "page_id": "Contributions_Agency", "url": "https://en.wikipedia.org/wiki/Contributions_Agency"}, {"rank": 6, "page_id": "Contributions_directes", "url": "https://en.wikipedia.org/wiki/Contributions_directes"}, {"rank": 7, "page_id": "Islamic_contributions", "url": "https://en.wikipedia.org/wiki/Islamic_contributions"}, {"rank": 8, "page_id": "21ji_made_no_Cinderella", "url": "https://en.wikipedia.org/wiki/21ji_made_no_Cinderella"}, {"rank": 9, "page_id": "Kimi_no_Machi_Made", "url": "https://en.wikipedia.org/wiki/Kimi_no_Machi_Made"}, {"rank": 10, "page_id": "Black_Box_Music", "url": "https://en.wikipedia.org/wiki/Black_Box_Music"}]}\n{"id": 5571, "claim": "LGBT is a synonym containing the word bisexual.", "ranked_articles": [{"rank": 1, "page_id": "Synonym", "url": "https://en.wikipedia.org/wiki/Synonym"}, {"rank": 2, "page_id": "Synonym_ring", "url": "https://en.wikipedia.org/wiki/Synonym_ring"}, {"rank": 3, "page_id": "List_of_phrases_containing_the_word_vitae", "url": "https://en.wikipedia.org/wiki/List_of_phrases_containing_the_word_vitae"}, {"rank": 4, "page_id": "Bisexual_chic", "url": "https://en.wikipedia.org/wiki/Bisexual_chic"}, {"rank": 5, "page_id": "Bisexual_community", "url": "https://en.wikipedia.org/wiki/Bisexual_community"}, {"rank": 6, "page_id": "Bisexual_erasure", "url": "https://en.wikipedia.org/wiki/Bisexual_erasure"}, {"rank": 7, "page_id": "Bisexual_politics", "url": "https://en.wikipedia.org/wiki/Bisexual_politics"}, {"rank": 8, "page_id": "Bisexual_pornography", "url": "https://en.wikipedia.org/wiki/Bisexual_pornography"}, {"rank": 9, "page_id": "Word_by_Word", "url": "https://en.wikipedia.org/wiki/Word_by_Word"}, {"rank": 10, "page_id": "Word", "url": "https://en.wikipedia.org/wiki/Word"}]}\n{"id": 154833, "claim": "Ranbir Kapoor does not co-own a football team in the men\'s professional football league in India.", "ranked_articles": [{"rank": 1, "page_id": "Ranbir_Kapoor", "url": "https://en.wikipedia.org/wiki/Ranbir_Kapoor"}, {"rank": 2, "page_id": "Ranbir", "url": "https://en.wikipedia.org/wiki/Ranbir"}, {"rank": 3, "page_id": "Kapoor", "url": "https://en.wikipedia.org/wiki/Kapoor"}, {"rank": 4, "page_id": "Narender_Ranbir", "url": "https://en.wikipedia.org/wiki/Narender_Ranbir"}, {"rank": 5, "page_id": "Rana_Ranbir", "url": "https://en.wikipedia.org/wiki/Rana_Ranbir"}, {"rank": 6, "page_id": "Ranbir_Gangwa", "url": "https://en.wikipedia.org/wiki/Ranbir_Gangwa"}, {"rank": 7, "page_id": "Ranbir_Rano", "url": "https://en.wikipedia.org/wiki/Ranbir_Rano"}, {"rank": 8, "page_id": "Ranbir_Singh", "url": "https://en.wikipedia.org/wiki/Ranbir_Singh"}, {"rank": 9, "page_id": "Does", "url": "https://en.wikipedia.org/wiki/Does"}, {"rank": 10, "page_id": "List_of_awards_and_nominations_received_by_Ranbir_Kapoor", "url": "https://en.wikipedia.org/wiki/List_of_awards_and_nominations_received_by_Ranbir_Kapoor"}]}\n{"id": 47767, "claim": "Love & Mercy (film) features a portrayal of a iguana.", "ranked_articles": [{"rank": 1, "page_id": "Love_&_Mercy_-LRB-film-RRB-", "url": "https://en.wikipedia.org/wiki/Love_%26_Mercy_%28film%29"}, {"rank": 2, "page_id": "Iguana", "url": "https://en.wikipedia.org/wiki/Iguana"}, {"rank": 3, "page_id": "Love_Without_Mercy", "url": "https://en.wikipedia.org/wiki/Love_Without_Mercy"}, {"rank": 4, "page_id": "Love_and_Mercy", "url": "https://en.wikipedia.org/wiki/Love_and_Mercy"}, {"rank": 5, "page_id": "Mercy,_Mercy,_Mercy", "url": "https://en.wikipedia.org/wiki/Mercy%2C_Mercy%2C_Mercy"}, {"rank": 6, "page_id": "Iguana_-LRB-film-RRB-", "url": "https://en.wikipedia.org/wiki/Iguana_%28film%29"}, {"rank": 7, "page_id": "Mercy,_Mercy", "url": "https://en.wikipedia.org/wiki/Mercy%2C_Mercy"}, {"rank": 8, "page_id": "Music_from_Love_&_Mercy", "url": "https://en.wikipedia.org/wiki/Music_from_Love_%26_Mercy"}, {"rank": 9, "page_id": "Blue_iguana", "url": "https://en.wikipedia.org/wiki/Blue_iguana"}, {"rank": 10, "page_id": "Desert_iguana", "url": "https://en.wikipedia.org/wiki/Desert_iguana"}]}\n{"id": 206937, "claim": "Michael Keaton is an American director and actor.", "ranked_articles": [{"rank": 1, "page_id": "Michael_Keaton", "url": "https://en.wikipedia.org/wiki/Michael_Keaton"}, {"rank": 2, "page_id": "Keaton", "url": "https://en.wikipedia.org/wiki/Keaton"}, {"rank": 3, "page_id": "Ben_Keaton", "url": "https://en.wikipedia.org/wiki/Ben_Keaton"}, {"rank": 4, "page_id": "Buster_Keaton", "url": "https://en.wikipedia.org/wiki/Buster_Keaton"}, {"rank": 5, "page_id": "Camille_Keaton", "url": "https://en.wikipedia.org/wiki/Camille_Keaton"}, {"rank": 6, "page_id": "Curtis_Keaton", "url": "https://en.wikipedia.org/wiki/Curtis_Keaton"}, {"rank": 7, "page_id": "Danielle_Keaton", "url": "https://en.wikipedia.org/wiki/Danielle_Keaton"}, {"rank": 8, "page_id": "Derek_Keaton", "url": "https://en.wikipedia.org/wiki/Derek_Keaton"}, {"rank": 9, "page_id": "Diane_Keaton", "url": "https://en.wikipedia.org/wiki/Diane_Keaton"}, {"rank": 10, "page_id": "Joe_Keaton", "url": "https://en.wikipedia.org/wiki/Joe_Keaton"}]}\n{"id": 210134, "claim": "Lonesome Dove (miniseries) had 18 wins at the 1989 Emmy Awards.", "ranked_articles": [{"rank": 1, "page_id": "Lonesome_Dove_-LRB-miniseries-RRB-", "url": "https://en.wikipedia.org/wiki/Lonesome_Dove_%28miniseries%29"}, {"rank": 2, "page_id": "Lonesome_Dove", "url": "https://en.wikipedia.org/wiki/Lonesome_Dove"}, {"rank": 3, "page_id": "Lonesome_Dove_series", "url": "https://en.wikipedia.org/wiki/Lonesome_Dove_series"}, {"rank": 4, "page_id": "Return_to_Lonesome_Dove", "url": "https://en.wikipedia.org/wiki/Return_to_Lonesome_Dove"}, {"rank": 5, "page_id": "WINS", "url": "https://en.wikipedia.org/wiki/WINS"}, {"rank": 6, "page_id": "Lonesome", "url": "https://en.wikipedia.org/wiki/Lonesome"}, {"rank": 7, "page_id": "Miniseries", "url": "https://en.wikipedia.org/wiki/Miniseries"}, {"rank": 8, "page_id": "1989_ABC_Under-18_Championship", "url": "https://en.wikipedia.org/wiki/1989_ABC_Under-18_Championship"}, {"rank": 9, "page_id": "Everybody_Wins", "url": "https://en.wikipedia.org/wiki/Everybody_Wins"}, {"rank": 10, "page_id": "Funniest_Wins", "url": "https://en.wikipedia.org/wiki/Funniest_Wins"}]}\n{"id": 106579, "claim": "Clint Eastwood worked on the Western film Unforgiven in 2003.", "ranked_articles": [{"rank": 1, "page_id": "Clint_Eastwood", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood"}, {"rank": 2, "page_id": "Clint_Eastwood_bibliography", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_bibliography"}, {"rank": 3, "page_id": "Clint_Eastwood_discography", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_discography"}, {"rank": 4, "page_id": "Clint_Eastwood_filmography", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_filmography"}, {"rank": 5, "page_id": "Clint_Eastwood_&_General_Saint", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_%26_General_Saint"}, {"rank": 6, "page_id": "Unforgiven", "url": "https://en.wikipedia.org/wiki/Unforgiven"}, {"rank": 7, "page_id": "Clint_Eastwood_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_%28album%29"}, {"rank": 8, "page_id": "Clint_Eastwood_-LRB-disambiguation-RRB-", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_%28disambiguation%29"}, {"rank": 9, "page_id": "Clint_Eastwood_-LRB-musician-RRB-", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_%28musician%29"}, {"rank": 10, "page_id": "Clint_Eastwood_-LRB-song-RRB-", "url": "https://en.wikipedia.org/wiki/Clint_Eastwood_%28song%29"}]}\n{"id": 206182, "claim": "Shallow Hal is an American television show.", "ranked_articles": [{"rank": 1, "page_id": "Shallow_Hal", "url": "https://en.wikipedia.org/wiki/Shallow_Hal"}, {"rank": 2, "page_id": "Shallow", "url": "https://en.wikipedia.org/wiki/Shallow"}, {"rank": 3, "page_id": "Hyron_Shallow", "url": "https://en.wikipedia.org/wiki/Hyron_Shallow"}, {"rank": 4, "page_id": "Only_Shallow", "url": "https://en.wikipedia.org/wiki/Only_Shallow"}, {"rank": 5, "page_id": "Parvati_Shallow", "url": "https://en.wikipedia.org/wiki/Parvati_Shallow"}, {"rank": 6, "page_id": "Robert_Shallow", "url": "https://en.wikipedia.org/wiki/Robert_Shallow"}, {"rank": 7, "page_id": "Shallow_Bay", "url": "https://en.wikipedia.org/wiki/Shallow_Bay"}, {"rank": 8, "page_id": "Shallow_Bed", "url": "https://en.wikipedia.org/wiki/Shallow_Bed"}, {"rank": 9, "page_id": "Shallow_Believer", "url": "https://en.wikipedia.org/wiki/Shallow_Believer"}, {"rank": 10, "page_id": "Shallow_Brook", "url": "https://en.wikipedia.org/wiki/Shallow_Brook"}]}\n{"id": 128569, "claim": "Dark Places stars two actors born in Japan.", "ranked_articles": [{"rank": 1, "page_id": "Dark_Places", "url": "https://en.wikipedia.org/wiki/Dark_Places"}, {"rank": 2, "page_id": "The_Two_Stars", "url": "https://en.wikipedia.org/wiki/The_Two_Stars"}, {"rank": 3, "page_id": "Two_Dark_Birds", "url": "https://en.wikipedia.org/wiki/Two_Dark_Birds"}, {"rank": 4, "page_id": "My_Dark_Places", "url": "https://en.wikipedia.org/wiki/My_Dark_Places"}, {"rank": 5, "page_id": "Stars_Are_Born", "url": "https://en.wikipedia.org/wiki/Stars_Are_Born"}, {"rank": 6, "page_id": "Full_Dark,_No_Stars", "url": "https://en.wikipedia.org/wiki/Full_Dark%2C_No_Stars"}, {"rank": 7, "page_id": "Two_in_the_Dark", "url": "https://en.wikipedia.org/wiki/Two_in_the_Dark"}, {"rank": 8, "page_id": "Lucky_Stars_Go_Places", "url": "https://en.wikipedia.org/wiki/Lucky_Stars_Go_Places"}, {"rank": 9, "page_id": "Lord_of_Dark_Places", "url": "https://en.wikipedia.org/wiki/Lord_of_Dark_Places"}, {"rank": 10, "page_id": "The_Dark_Beyond_the_Stars", "url": "https://en.wikipedia.org/wiki/The_Dark_Beyond_the_Stars"}]}\n{"id": 40186, "claim": "The Berlin Wall was in Germany.", "ranked_articles": [{"rank": 1, "page_id": "Berlin_Wall", "url": "https://en.wikipedia.org/wiki/Berlin_Wall"}, {"rank": 2, "page_id": "Berlin_Customs_Wall", "url": "https://en.wikipedia.org/wiki/Berlin_Customs_Wall"}, {"rank": 3, "page_id": "Berlin_Wall_graffiti_art", "url": "https://en.wikipedia.org/wiki/Berlin_Wall_graffiti_art"}, {"rank": 4, "page_id": "Berlin_is_in_Germany", "url": "https://en.wikipedia.org/wiki/Berlin_is_in_Germany"}, {"rank": 5, "page_id": "List_of_Berlin_Wall_segments", "url": "https://en.wikipedia.org/wiki/List_of_Berlin_Wall_segments"}, {"rank": 6, "page_id": "Live_at_the_Berlin_Wall", "url": "https://en.wikipedia.org/wiki/Live_at_the_Berlin_Wall"}, {"rank": 7, "page_id": "The_Wall_–_Live_in_Berlin", "url": "https://en.wikipedia.org/wiki/The_Wall_%E2%80%93_Live_in_Berlin"}, {"rank": 8, "page_id": "Berlin,_Berlin", "url": "https://en.wikipedia.org/wiki/Berlin%2C_Berlin"}, {"rank": 9, "page_id": "Berlin_Wall_Monument_-LRB-Chicago-RRB-", "url": "https://en.wikipedia.org/wiki/Berlin_Wall_Monument_%28Chicago%29"}, {"rank": 10, "page_id": "Wall_to_Wall", "url": "https://en.wikipedia.org/wiki/Wall_to_Wall"}]}\n{"id": 179603, "claim": "Michael Giacchino composed scores.", "ranked_articles": [{"rank": 1, "page_id": "Michael_Giacchino", "url": "https://en.wikipedia.org/wiki/Michael_Giacchino"}, {"rank": 2, "page_id": "Giacchino", "url": "https://en.wikipedia.org/wiki/Giacchino"}, {"rank": 3, "page_id": "Scores", "url": "https://en.wikipedia.org/wiki/Scores"}, {"rank": 4, "page_id": "Anthony_Giacchino", "url": "https://en.wikipedia.org/wiki/Anthony_Giacchino"}, {"rank": 5, "page_id": "Through-composed", "url": "https://en.wikipedia.org/wiki/Through-composed"}, {"rank": 6, "page_id": "Famitsū_scores", "url": "https://en.wikipedia.org/wiki/Famits%C5%AB_scores"}, {"rank": 7, "page_id": "GAA_Scores", "url": "https://en.wikipedia.org/wiki/GAA_Scores"}, {"rank": 8, "page_id": "Hi_Scores", "url": "https://en.wikipedia.org/wiki/Hi_Scores"}, {"rank": 9, "page_id": "Hockey_Scores", "url": "https://en.wikipedia.org/wiki/Hockey_Scores"}, {"rank": 10, "page_id": "Live_scores", "url": "https://en.wikipedia.org/wiki/Live_scores"}]}\n{"id": 111557, "claim": "Carolina Panthers are a team from New Jersey.", "ranked_articles": [{"rank": 1, "page_id": "Carolina_Panthers", "url": "https://en.wikipedia.org/wiki/Carolina_Panthers"}, {"rank": 2, "page_id": "1995_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/1995_Carolina_Panthers_season"}, {"rank": 3, "page_id": "1996_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/1996_Carolina_Panthers_season"}, {"rank": 4, "page_id": "1997_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/1997_Carolina_Panthers_season"}, {"rank": 5, "page_id": "1998_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/1998_Carolina_Panthers_season"}, {"rank": 6, "page_id": "1999_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/1999_Carolina_Panthers_season"}, {"rank": 7, "page_id": "2000_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/2000_Carolina_Panthers_season"}, {"rank": 8, "page_id": "2001_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/2001_Carolina_Panthers_season"}, {"rank": 9, "page_id": "2002_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/2002_Carolina_Panthers_season"}, {"rank": 10, "page_id": "2003_Carolina_Panthers_season", "url": "https://en.wikipedia.org/wiki/2003_Carolina_Panthers_season"}]}\n{"id": 128891, "claim": "Jennifer Lawrence stars in The Hunger Games.", "ranked_articles": [{"rank": 1, "page_id": "Jennifer_Lawrence", "url": "https://en.wikipedia.org/wiki/Jennifer_Lawrence"}, {"rank": 2, "page_id": "The_Hunger_Games", "url": "https://en.wikipedia.org/wiki/The_Hunger_Games"}, {"rank": 3, "page_id": "Hunger", "url": "https://en.wikipedia.org/wiki/Hunger"}, {"rank": 4, "page_id": "The_Hunger_Games-COLON-_Mockingjay", "url": "https://en.wikipedia.org/wiki/The_Hunger_Games%3A_Mockingjay"}, {"rank": 5, "page_id": "Alfredo_Hunger", "url": "https://en.wikipedia.org/wiki/Alfredo_Hunger"}, {"rank": 6, "page_id": "American_Hunger", "url": "https://en.wikipedia.org/wiki/American_Hunger"}, {"rank": 7, "page_id": "Anna_Hunger", "url": "https://en.wikipedia.org/wiki/Anna_Hunger"}, {"rank": 8, "page_id": "Blood_Hunger", "url": "https://en.wikipedia.org/wiki/Blood_Hunger"}, {"rank": 9, "page_id": "Daniela_Hunger", "url": "https://en.wikipedia.org/wiki/Daniela_Hunger"}, {"rank": 10, "page_id": "Felix_Hunger", "url": "https://en.wikipedia.org/wiki/Felix_Hunger"}]}\n{"id": 4904, "claim": "Paul Simon played music.", "ranked_articles": [{"rank": 1, "page_id": "Paul_Simon", "url": "https://en.wikipedia.org/wiki/Paul_Simon"}, {"rank": 2, "page_id": "Played", "url": "https://en.wikipedia.org/wiki/Played"}, {"rank": 3, "page_id": "Paul_Ludwig_Simon", "url": "https://en.wikipedia.org/wiki/Paul_Ludwig_Simon"}, {"rank": 4, "page_id": "Paul_Simon_discography", "url": "https://en.wikipedia.org/wiki/Paul_Simon_discography"}, {"rank": 5, "page_id": "Games_played", "url": "https://en.wikipedia.org/wiki/Games_played"}, {"rank": 6, "page_id": "Getting_Played", "url": "https://en.wikipedia.org/wiki/Getting_Played"}, {"rank": 7, "page_id": "Paul_Simon_1964/1993", "url": "https://en.wikipedia.org/wiki/Paul_Simon_1964%2F1993"}, {"rank": 8, "page_id": "Paul_Simon_in_Concert", "url": "https://en.wikipedia.org/wiki/Paul_Simon_in_Concert"}, {"rank": 9, "page_id": "The_Essential_Paul_Simon", "url": "https://en.wikipedia.org/wiki/The_Essential_Paul_Simon"}, {"rank": 10, "page_id": "The_Paul_Simon_Anthology", "url": "https://en.wikipedia.org/wiki/The_Paul_Simon_Anthology"}]}\n{"id": 211660, "claim": "Jane Eyre is a film released in 2011.", "ranked_articles": [{"rank": 1, "page_id": "Jane_Eyre", "url": "https://en.wikipedia.org/wiki/Jane_Eyre"}, {"rank": 2, "page_id": "Jane_Eyre_-LRB-2011_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%282011_film%29"}, {"rank": 3, "page_id": "Jane_Eyre_-LRB-1910_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281910_film%29"}, {"rank": 4, "page_id": "Jane_Eyre_-LRB-1934_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281934_film%29"}, {"rank": 5, "page_id": "Jane_Eyre_-LRB-1943_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281943_film%29"}, {"rank": 6, "page_id": "Jane_Eyre_-LRB-1956_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281956_film%29"}, {"rank": 7, "page_id": "Jane_Eyre_-LRB-1970_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281970_film%29"}, {"rank": 8, "page_id": "Jane_Eyre_-LRB-1996_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281996_film%29"}, {"rank": 9, "page_id": "Jane_Eyre_-LRB-1997_film-RRB-", "url": "https://en.wikipedia.org/wiki/Jane_Eyre_%281997_film%29"}, {"rank": 10, "page_id": "Adaptations_of_Jane_Eyre", "url": "https://en.wikipedia.org/wiki/Adaptations_of_Jane_Eyre"}]}\n{"id": 17174, "claim": "28 Days (film) was unable to cast Dominic West.", "ranked_articles": [{"rank": 1, "page_id": "28_Days", "url": "https://en.wikipedia.org/wiki/28_Days"}, {"rank": 2, "page_id": "28_Days_-LRB-film-RRB-", "url": "https://en.wikipedia.org/wiki/28_Days_%28film%29"}, {"rank": 3, "page_id": "28_Days_Later", "url": "https://en.wikipedia.org/wiki/28_Days_Later"}, {"rank": 4, "page_id": "28_Thousand_Days", "url": "https://en.wikipedia.org/wiki/28_Thousand_Days"}, {"rank": 5, "page_id": "Unable_Records", "url": "https://en.wikipedia.org/wiki/Unable_Records"}, {"rank": 6, "page_id": "28_mm_film", "url": "https://en.wikipedia.org/wiki/28_mm_film"}, {"rank": 7, "page_id": "28_Days_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/28_Days_%28album%29"}, {"rank": 8, "page_id": "28_Days_-LRB-band-RRB-", "url": "https://en.wikipedia.org/wiki/28_Days_%28band%29"}, {"rank": 9, "page_id": "CAST", "url": "https://en.wikipedia.org/wiki/CAST"}, {"rank": 10, "page_id": "Cast", "url": "https://en.wikipedia.org/wiki/Cast"}]}\n{"id": 198135, "claim": "Julie Walters failed to be an actress.", "ranked_articles": [{"rank": 1, "page_id": "Julie_Walters", "url": "https://en.wikipedia.org/wiki/Julie_Walters"}, {"rank": 2, "page_id": "Julie_Walters_and_Friends", "url": "https://en.wikipedia.org/wiki/Julie_Walters_and_Friends"}, {"rank": 3, "page_id": "Failed_Experiments", "url": "https://en.wikipedia.org/wiki/Failed_Experiments"}, {"rank": 4, "page_id": "Failed_acreage", "url": "https://en.wikipedia.org/wiki/Failed_acreage"}, {"rank": 5, "page_id": "Failed_state", "url": "https://en.wikipedia.org/wiki/Failed_state"}, {"rank": 6, "page_id": "Failed_supernova", "url": "https://en.wikipedia.org/wiki/Failed_supernova"}, {"rank": 7, "page_id": "Walters", "url": "https://en.wikipedia.org/wiki/Walters"}, {"rank": 8, "page_id": "Julie_Bishop_-LRB-actress-RRB-", "url": "https://en.wikipedia.org/wiki/Julie_Bishop_%28actress%29"}, {"rank": 9, "page_id": "Julie_Harris_-LRB-actress-RRB-", "url": "https://en.wikipedia.org/wiki/Julie_Harris_%28actress%29"}, {"rank": 10, "page_id": "Julie_Smith_-LRB-actress-RRB-", "url": "https://en.wikipedia.org/wiki/Julie_Smith_%28actress%29"}]}\n{"id": 79089, "claim": "Steve Buscemi is a producer.", "ranked_articles": [{"rank": 1, "page_id": "Steve_Buscemi", "url": "https://en.wikipedia.org/wiki/Steve_Buscemi"}, {"rank": 2, "page_id": "Steve_Buscemi_filmography", "url": "https://en.wikipedia.org/wiki/Steve_Buscemi_filmography"}, {"rank": 3, "page_id": "Buscemi", "url": "https://en.wikipedia.org/wiki/Buscemi"}, {"rank": 4, "page_id": "Park_Bench_with_Steve_Buscemi", "url": "https://en.wikipedia.org/wiki/Park_Bench_with_Steve_Buscemi"}, {"rank": 5, "page_id": "Jon_Buscemi", "url": "https://en.wikipedia.org/wiki/Jon_Buscemi"}, {"rank": 6, "page_id": "Steve_Booker_-LRB-producer-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_Booker_%28producer%29"}, {"rank": 7, "page_id": "Steve_James_-LRB-producer-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_James_%28producer%29"}, {"rank": 8, "page_id": "Steve_Porter_-LRB-producer-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_Porter_%28producer%29"}, {"rank": 9, "page_id": "Steve_Waterman_-LRB-producer-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_Waterman_%28producer%29"}, {"rank": 10, "page_id": "Steve_White_-LRB-producer-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_White_%28producer%29"}]}\n{"id": 45987, "claim": "Shueisha\'s CEO is from Japan.", "ranked_articles": [{"rank": 1, "page_id": "Shueisha", "url": "https://en.wikipedia.org/wiki/Shueisha"}, {"rank": 2, "page_id": "Shogakukan-Shueisha_Productions", "url": "https://en.wikipedia.org/wiki/Shogakukan-Shueisha_Productions"}, {"rank": 3, "page_id": "CEO_Dancers", "url": "https://en.wikipedia.org/wiki/CEO_Dancers"}, {"rank": 4, "page_id": "CEO_Exchange", "url": "https://en.wikipedia.org/wiki/CEO_Exchange"}, {"rank": 5, "page_id": "CEO_succession", "url": "https://en.wikipedia.org/wiki/CEO_succession"}, {"rank": 6, "page_id": "Founder_CEO", "url": "https://en.wikipedia.org/wiki/Founder_CEO"}, {"rank": 7, "page_id": "The_CEO", "url": "https://en.wikipedia.org/wiki/The_CEO"}, {"rank": 8, "page_id": "CEO_Sleepout_UK", "url": "https://en.wikipedia.org/wiki/CEO_Sleepout_UK"}, {"rank": 9, "page_id": "Maria_do_Ceo", "url": "https://en.wikipedia.org/wiki/Maria_do_Ceo"}, {"rank": 10, "page_id": "Violante_do_Ceo", "url": "https://en.wikipedia.org/wiki/Violante_do_Ceo"}]}\n{"id": 81883, "claim": "The Crazies was a film written by Scott Kosar.", "ranked_articles": [{"rank": 1, "page_id": "Scott_Kosar", "url": "https://en.wikipedia.org/wiki/Scott_Kosar"}, {"rank": 2, "page_id": "Kosar", "url": "https://en.wikipedia.org/wiki/Kosar"}, {"rank": 3, "page_id": "Cameron_Crazies", "url": "https://en.wikipedia.org/wiki/Cameron_Crazies"}, {"rank": 4, "page_id": "Clinton_crazies", "url": "https://en.wikipedia.org/wiki/Clinton_crazies"}, {"rank": 5, "page_id": "Crazies_Hill", "url": "https://en.wikipedia.org/wiki/Crazies_Hill"}, {"rank": 6, "page_id": "Karaoke_Crazies", "url": "https://en.wikipedia.org/wiki/Karaoke_Crazies"}, {"rank": 7, "page_id": "The_Crazies", "url": "https://en.wikipedia.org/wiki/The_Crazies"}, {"rank": 8, "page_id": "Begushchy_Kosar", "url": "https://en.wikipedia.org/wiki/Begushchy_Kosar"}, {"rank": 9, "page_id": "Bernie_Kosar", "url": "https://en.wikipedia.org/wiki/Bernie_Kosar"}, {"rank": 10, "page_id": "Kosar_Buriro", "url": "https://en.wikipedia.org/wiki/Kosar_Buriro"}]}\n{"id": 45676, "claim": "Fairy Tail had a third series.", "ranked_articles": [{"rank": 1, "page_id": "Fairy_Tail", "url": "https://en.wikipedia.org/wiki/Fairy_Tail"}, {"rank": 2, "page_id": "Fairy_Tail_Zero", "url": "https://en.wikipedia.org/wiki/Fairy_Tail_Zero"}, {"rank": 3, "page_id": "Fairy_Tail-COLON-_Dragon_Cry", "url": "https://en.wikipedia.org/wiki/Fairy_Tail%3A_Dragon_Cry"}, {"rank": 4, "page_id": "List_of_Fairy_Tail_characters", "url": "https://en.wikipedia.org/wiki/List_of_Fairy_Tail_characters"}, {"rank": 5, "page_id": "List_of_Fairy_Tail_episodes", "url": "https://en.wikipedia.org/wiki/List_of_Fairy_Tail_episodes"}, {"rank": 6, "page_id": "List_of_Fairy_Tail_volumes", "url": "https://en.wikipedia.org/wiki/List_of_Fairy_Tail_volumes"}, {"rank": 7, "page_id": "Fairy", "url": "https://en.wikipedia.org/wiki/Fairy"}, {"rank": 8, "page_id": "Tail", "url": "https://en.wikipedia.org/wiki/Tail"}, {"rank": 9, "page_id": "Fairy_Tail_the_Movie-COLON-_Phoenix_Priestess", "url": "https://en.wikipedia.org/wiki/Fairy_Tail_the_Movie%3A_Phoenix_Priestess"}, {"rank": 10, "page_id": "List_of_Fairy_Tail_spin-off_manga", "url": "https://en.wikipedia.org/wiki/List_of_Fairy_Tail_spin-off_manga"}]}\n{"id": 210268, "claim": "William McKinley served as president.", "ranked_articles": [{"rank": 1, "page_id": "President_William_McKinley_High_School", "url": "https://en.wikipedia.org/wiki/President_William_McKinley_High_School"}, {"rank": 2, "page_id": "William_McKinley", "url": "https://en.wikipedia.org/wiki/William_McKinley"}, {"rank": 3, "page_id": "President_McKinley_Inauguration_Footage", "url": "https://en.wikipedia.org/wiki/President_McKinley_Inauguration_Footage"}, {"rank": 4, "page_id": "William_B._McKinley", "url": "https://en.wikipedia.org/wiki/William_B._McKinley"}, {"rank": 5, "page_id": "William_McKinley_Sr.", "url": "https://en.wikipedia.org/wiki/William_McKinley_Sr."}, {"rank": 6, "page_id": "William_Thomas_McKinley", "url": "https://en.wikipedia.org/wiki/William_Thomas_McKinley"}, {"rank": 7, "page_id": "McKinley", "url": "https://en.wikipedia.org/wiki/McKinley"}, {"rank": 8, "page_id": "Assassination_of_William_McKinley", "url": "https://en.wikipedia.org/wiki/Assassination_of_William_McKinley"}, {"rank": 9, "page_id": "Bibliography_of_William_McKinley", "url": "https://en.wikipedia.org/wiki/Bibliography_of_William_McKinley"}, {"rank": 10, "page_id": "Inauguration_of_William_McKinley", "url": "https://en.wikipedia.org/wiki/Inauguration_of_William_McKinley"}]}\n{"id": 170862, "claim": "A Thousand Suns is an South American album.", "ranked_articles": [{"rank": 1, "page_id": "A_Thousand_Suns", "url": "https://en.wikipedia.org/wiki/A_Thousand_Suns"}, {"rank": 2, "page_id": "A_Thousand_Suns+", "url": "https://en.wikipedia.org/wiki/A_Thousand_Suns%2B"}, {"rank": 3, "page_id": "A_Thousand_Splendid_Suns", "url": "https://en.wikipedia.org/wiki/A_Thousand_Splendid_Suns"}, {"rank": 4, "page_id": "A_Thousand_Suns_World_Tour", "url": "https://en.wikipedia.org/wiki/A_Thousand_Suns_World_Tour"}, {"rank": 5, "page_id": "Brighter_than_a_Thousand_Suns", "url": "https://en.wikipedia.org/wiki/Brighter_than_a_Thousand_Suns"}, {"rank": 6, "page_id": "City_of_a_Thousand_Suns", "url": "https://en.wikipedia.org/wiki/City_of_a_Thousand_Suns"}, {"rank": 7, "page_id": "A_Thousand_Suns_-LRB-Russell_Morris_album-RRB-", "url": "https://en.wikipedia.org/wiki/A_Thousand_Suns_%28Russell_Morris_album%29"}, {"rank": 8, "page_id": "Brighter_Than_a_Thousand_Suns_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/Brighter_Than_a_Thousand_Suns_%28album%29"}, {"rank": 9, "page_id": "The_Meeting_of_a_Thousand_Suns", "url": "https://en.wikipedia.org/wiki/The_Meeting_of_a_Thousand_Suns"}, {"rank": 10, "page_id": "Suns", "url": "https://en.wikipedia.org/wiki/Suns"}]}\n{"id": 67859, "claim": "The windigo is an evil guitar player.", "ranked_articles": [{"rank": 1, "page_id": "Guitar_Player", "url": "https://en.wikipedia.org/wiki/Guitar_Player"}, {"rank": 2, "page_id": "The_Guitar_Player", "url": "https://en.wikipedia.org/wiki/The_Guitar_Player"}, {"rank": 3, "page_id": "Lake_Windigo", "url": "https://en.wikipedia.org/wiki/Lake_Windigo"}, {"rank": 4, "page_id": "Windigo_Lake", "url": "https://en.wikipedia.org/wiki/Windigo_Lake"}, {"rank": 5, "page_id": "Windigo_River", "url": "https://en.wikipedia.org/wiki/Windigo_River"}, {"rank": 6, "page_id": "Guitar_Guitar_Guitar", "url": "https://en.wikipedia.org/wiki/Guitar_Guitar_Guitar"}, {"rank": 7, "page_id": "Guitar_Player_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/Guitar_Player_%28album%29"}, {"rank": 8, "page_id": "Merle_Travis_the_Guitar_Player", "url": "https://en.wikipedia.org/wiki/Merle_Travis_the_Guitar_Player"}, {"rank": 9, "page_id": "Rivière-Windigo,_Quebec", "url": "https://en.wikipedia.org/wiki/Rivi%C3%A8re-Windigo%2C_Quebec"}, {"rank": 10, "page_id": "Windigo_Ranger_Station", "url": "https://en.wikipedia.org/wiki/Windigo_Ranger_Station"}]}\n{"id": 73613, "claim": "Emilio Estevez writes.", "ranked_articles": [{"rank": 1, "page_id": "Emilio_Estevez", "url": "https://en.wikipedia.org/wiki/Emilio_Estevez"}, {"rank": 2, "page_id": "Estevez", "url": "https://en.wikipedia.org/wiki/Estevez"}, {"rank": 3, "page_id": "Human_Writes", "url": "https://en.wikipedia.org/wiki/Human_Writes"}, {"rank": 4, "page_id": "Kids\'_Writes", "url": "https://en.wikipedia.org/wiki/Kids%27_Writes"}, {"rank": 5, "page_id": "Abilio_Estévez", "url": "https://en.wikipedia.org/wiki/Abilio_Est%C3%A9vez"}, {"rank": 6, "page_id": "Antonio_Estévez", "url": "https://en.wikipedia.org/wiki/Antonio_Est%C3%A9vez"}, {"rank": 7, "page_id": "Belén_Estévez", "url": "https://en.wikipedia.org/wiki/Bel%C3%A9n_Est%C3%A9vez"}, {"rank": 8, "page_id": "Camilo_Estévez", "url": "https://en.wikipedia.org/wiki/Camilo_Est%C3%A9vez"}, {"rank": 9, "page_id": "Cristina_Estévez", "url": "https://en.wikipedia.org/wiki/Cristina_Est%C3%A9vez"}, {"rank": 10, "page_id": "Estela_Estévez", "url": "https://en.wikipedia.org/wiki/Estela_Est%C3%A9vez"}]}\n{"id": 56727, "claim": "David Bowie published his first single three weeks before he got sick.", "ranked_articles": [{"rank": 1, "page_id": "David_Bowie", "url": "https://en.wikipedia.org/wiki/David_Bowie"}, {"rank": 2, "page_id": "David_Bowie_discography", "url": "https://en.wikipedia.org/wiki/David_Bowie_discography"}, {"rank": 3, "page_id": "David_Bowie_filmography", "url": "https://en.wikipedia.org/wiki/David_Bowie_filmography"}, {"rank": 4, "page_id": "His_First_Crush", "url": "https://en.wikipedia.org/wiki/His_First_Crush"}, {"rank": 5, "page_id": "His_First_Flame", "url": "https://en.wikipedia.org/wiki/His_First_Flame"}, {"rank": 6, "page_id": "His_First_Love", "url": "https://en.wikipedia.org/wiki/His_First_Love"}, {"rank": 7, "page_id": "David_First", "url": "https://en.wikipedia.org/wiki/David_First"}, {"rank": 8, "page_id": "Death_of_David_Bowie", "url": "https://en.wikipedia.org/wiki/Death_of_David_Bowie"}, {"rank": 9, "page_id": "Bowie", "url": "https://en.wikipedia.org/wiki/Bowie"}, {"rank": 10, "page_id": "First_of_His_Name", "url": "https://en.wikipedia.org/wiki/First_of_His_Name"}]}\n{"id": 73853, "claim": "Steve Jobs was a tax collector.", "ranked_articles": [{"rank": 1, "page_id": "Tax_collector", "url": "https://en.wikipedia.org/wiki/Tax_collector"}, {"rank": 2, "page_id": "Steve_Jobs", "url": "https://en.wikipedia.org/wiki/Steve_Jobs"}, {"rank": 3, "page_id": "New_Jobs_tax_credit", "url": "https://en.wikipedia.org/wiki/New_Jobs_tax_credit"}, {"rank": 4, "page_id": "ICon-COLON-_Steve_Jobs", "url": "https://en.wikipedia.org/wiki/ICon%3A_Steve_Jobs"}, {"rank": 5, "page_id": "Confessions_of_a_Tax_Collector", "url": "https://en.wikipedia.org/wiki/Confessions_of_a_Tax_Collector"}, {"rank": 6, "page_id": "Customer_-LRB-tax_collector-RRB-", "url": "https://en.wikipedia.org/wiki/Customer_%28tax_collector%29"}, {"rank": 7, "page_id": ".jobs", "url": "https://en.wikipedia.org/wiki/.jobs"}, {"rank": 8, "page_id": "JOBS", "url": "https://en.wikipedia.org/wiki/JOBS"}, {"rank": 9, "page_id": "Jobs", "url": "https://en.wikipedia.org/wiki/Jobs"}, {"rank": 10, "page_id": "Steve_Jobs_-LRB-book-RRB-", "url": "https://en.wikipedia.org/wiki/Steve_Jobs_%28book%29"}]}\n{"id": 167230, "claim": "Count Basie took his son to the third-most populous city in the United States.", "ranked_articles": [{"rank": 1, "page_id": "Count_Basie", "url": "https://en.wikipedia.org/wiki/Count_Basie"}, {"rank": 2, "page_id": "Count_Basie_Orchestra", "url": "https://en.wikipedia.org/wiki/Count_Basie_Orchestra"}, {"rank": 3, "page_id": "Count_Basie_Theatre", "url": "https://en.wikipedia.org/wiki/Count_Basie_Theatre"}, {"rank": 4, "page_id": "Count_Basie/Sarah_Vaughan", "url": "https://en.wikipedia.org/wiki/Count_Basie%2FSarah_Vaughan"}, {"rank": 5, "page_id": "Count_Basie_at_Newport", "url": "https://en.wikipedia.org/wiki/Count_Basie_at_Newport"}, {"rank": 6, "page_id": "Homage_to_Count_Basie", "url": "https://en.wikipedia.org/wiki/Homage_to_Count_Basie"}, {"rank": 7, "page_id": "The_Count_Basie_Story", "url": "https://en.wikipedia.org/wiki/The_Count_Basie_Story"}, {"rank": 8, "page_id": "For_His_Son", "url": "https://en.wikipedia.org/wiki/For_His_Son"}, {"rank": 9, "page_id": "His_Only_Son", "url": "https://en.wikipedia.org/wiki/His_Only_Son"}, {"rank": 10, "page_id": "Took", "url": "https://en.wikipedia.org/wiki/Took"}]}\n{"id": 75919, "claim": "Baahubali: The Beginning is directed by M. Night Shyamalan.", "ranked_articles": [{"rank": 1, "page_id": "M._Night_Shyamalan", "url": "https://en.wikipedia.org/wiki/M._Night_Shyamalan"}, {"rank": 2, "page_id": "Baahubali-COLON-_The_Beginning", "url": "https://en.wikipedia.org/wiki/Baahubali%3A_The_Beginning"}, {"rank": 3, "page_id": "Baahubali-COLON-_The_Beginning_-LRB-soundtrack-RRB-", "url": "https://en.wikipedia.org/wiki/Baahubali%3A_The_Beginning_%28soundtrack%29"}, {"rank": 4, "page_id": "Beginning", "url": "https://en.wikipedia.org/wiki/Beginning"}, {"rank": 5, "page_id": "Directed_Edge", "url": "https://en.wikipedia.org/wiki/Directed_Edge"}, {"rank": 6, "page_id": "Directed_Electronics", "url": "https://en.wikipedia.org/wiki/Directed_Electronics"}, {"rank": 7, "page_id": "Directed_differentiation", "url": "https://en.wikipedia.org/wiki/Directed_differentiation"}, {"rank": 8, "page_id": "Directed_evolution", "url": "https://en.wikipedia.org/wiki/Directed_evolution"}, {"rank": 9, "page_id": "Directed_graph", "url": "https://en.wikipedia.org/wiki/Directed_graph"}, {"rank": 10, "page_id": "Directed_infinity", "url": "https://en.wikipedia.org/wiki/Directed_infinity"}]}\n{"id": 97558, "claim": "Rachel McAdams starred in Twilight.", "ranked_articles": [{"rank": 1, "page_id": "Rachel_McAdams", "url": "https://en.wikipedia.org/wiki/Rachel_McAdams"}, {"rank": 2, "page_id": "McAdams", "url": "https://en.wikipedia.org/wiki/McAdams"}, {"rank": 3, "page_id": "Starred_Up", "url": "https://en.wikipedia.org/wiki/Starred_Up"}, {"rank": 4, "page_id": "Starred_transform", "url": "https://en.wikipedia.org/wiki/Starred_transform"}, {"rank": 5, "page_id": "Alfred_McAdams", "url": "https://en.wikipedia.org/wiki/Alfred_McAdams"}, {"rank": 6, "page_id": "Ben_McAdams", "url": "https://en.wikipedia.org/wiki/Ben_McAdams"}, {"rank": 7, "page_id": "Billy_McAdams", "url": "https://en.wikipedia.org/wiki/Billy_McAdams"}, {"rank": 8, "page_id": "Carl_McAdams", "url": "https://en.wikipedia.org/wiki/Carl_McAdams"}, {"rank": 9, "page_id": "Dean_McAdams", "url": "https://en.wikipedia.org/wiki/Dean_McAdams"}, {"rank": 10, "page_id": "Jack_McAdams", "url": "https://en.wikipedia.org/wiki/Jack_McAdams"}]}\n{"id": 116431, "claim": "FC Barcelona has won American titles in basketball.", "ranked_articles": [{"rank": 1, "page_id": "FC_Barcelona", "url": "https://en.wikipedia.org/wiki/FC_Barcelona"}, {"rank": 2, "page_id": "FC_Barcelona_B", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_B"}, {"rank": 3, "page_id": "FC_Barcelona_Bàsquet", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_B%C3%A0squet"}, {"rank": 4, "page_id": "FC_Barcelona_C", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_C"}, {"rank": 5, "page_id": "FC_Barcelona_Futsal", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Futsal"}, {"rank": 6, "page_id": "FC_Barcelona_Handbol", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Handbol"}, {"rank": 7, "page_id": "FC_Barcelona_Hoquei", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Hoquei"}, {"rank": 8, "page_id": "FC_Barcelona_Museum", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Museum"}, {"rank": 9, "page_id": "FC_Barcelona_Rugby", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Rugby"}, {"rank": 10, "page_id": "FC_Barcelona_Voleibol", "url": "https://en.wikipedia.org/wiki/FC_Barcelona_Voleibol"}]}\n{"id": 114703, "claim": "Colin Firth has starred in movies.", "ranked_articles": [{"rank": 1, "page_id": "Colin_Firth", "url": "https://en.wikipedia.org/wiki/Colin_Firth"}, {"rank": 2, "page_id": "List_of_Colin_Firth_performances", "url": "https://en.wikipedia.org/wiki/List_of_Colin_Firth_performances"}, {"rank": 3, "page_id": "Starred_Up", "url": "https://en.wikipedia.org/wiki/Starred_Up"}, {"rank": 4, "page_id": "Starred_transform", "url": "https://en.wikipedia.org/wiki/Starred_transform"}, {"rank": 5, "page_id": "Firth", "url": "https://en.wikipedia.org/wiki/Firth"}, {"rank": 6, "page_id": "Movies!", "url": "https://en.wikipedia.org/wiki/Movies%21"}, {"rank": 7, "page_id": "Movies@", "url": "https://en.wikipedia.org/wiki/Movies%40"}, {"rank": 8, "page_id": "Starred_wood_quail", "url": "https://en.wikipedia.org/wiki/Starred_wood_quail"}, {"rank": 9, "page_id": "White-starred_robin", "url": "https://en.wikipedia.org/wiki/White-starred_robin"}, {"rank": 10, "page_id": "Albert_Firth", "url": "https://en.wikipedia.org/wiki/Albert_Firth"}]}\n{"id": 199544, "claim": "US Congress only has 30 Senators.", "ranked_articles": [{"rank": 1, "page_id": "Us_and_Us_Only", "url": "https://en.wikipedia.org/wiki/Us_and_Us_Only"}, {"rank": 2, "page_id": "Above_Us_Only_Sky", "url": "https://en.wikipedia.org/wiki/Above_Us_Only_Sky"}, {"rank": 3, "page_id": "It\'s_Only_Us", "url": "https://en.wikipedia.org/wiki/It%27s_Only_Us"}, {"rank": 4, "page_id": "1929–30_Ottawa_Senators_season", "url": "https://en.wikipedia.org/wiki/1929%E2%80%9330_Ottawa_Senators_season"}, {"rank": 5, "page_id": "Senators", "url": "https://en.wikipedia.org/wiki/Senators"}, {"rank": 6, "page_id": "30-30", "url": "https://en.wikipedia.org/wiki/30-30"}, {"rank": 7, "page_id": "¡30-30!", "url": "https://en.wikipedia.org/wiki/%C2%A130-30%21"}, {"rank": 8, "page_id": "Tell_Us_Only_the_Beautiful_Things", "url": "https://en.wikipedia.org/wiki/Tell_Us_Only_the_Beautiful_Things"}, {"rank": 9, "page_id": ".30-30_Winchester", "url": "https://en.wikipedia.org/wiki/.30-30_Winchester"}, {"rank": 10, "page_id": "30/30-150", "url": "https://en.wikipedia.org/wiki/30%2F30-150"}]}\n{"id": 226194, "claim": "Gift of the Night Fury stars the voice of an actor born on June 20.", "ranked_articles": [{"rank": 1, "page_id": "Gift_of_the_Night_Fury", "url": "https://en.wikipedia.org/wiki/Gift_of_the_Night_Fury"}, {"rank": 2, "page_id": "Last_Night_Stars", "url": "https://en.wikipedia.org/wiki/Last_Night_Stars"}, {"rank": 3, "page_id": "Night_Without_Stars", "url": "https://en.wikipedia.org/wiki/Night_Without_Stars"}, {"rank": 4, "page_id": "Voice_in_the_Night", "url": "https://en.wikipedia.org/wiki/Voice_in_the_Night"}, {"rank": 5, "page_id": "Night_of_1000_Stars", "url": "https://en.wikipedia.org/wiki/Night_of_1000_Stars"}, {"rank": 6, "page_id": "Night_of_NASCAR_stars", "url": "https://en.wikipedia.org/wiki/Night_of_NASCAR_stars"}, {"rank": 7, "page_id": "GiFT", "url": "https://en.wikipedia.org/wiki/GiFT"}, {"rank": 8, "page_id": "Gift", "url": "https://en.wikipedia.org/wiki/Gift"}, {"rank": 9, "page_id": "Fury", "url": "https://en.wikipedia.org/wiki/Fury"}, {"rank": 10, "page_id": "The_Voice_in_the_Night", "url": "https://en.wikipedia.org/wiki/The_Voice_in_the_Night"}]}\n{"id": 113256, "claim": "Santorini has a population of 15,550 dogs.", "ranked_articles": [{"rank": 1, "page_id": "Santorini", "url": "https://en.wikipedia.org/wiki/Santorini"}, {"rank": 2, "page_id": "550", "url": "https://en.wikipedia.org/wiki/550"}, {"rank": 3, "page_id": "Akrotiri,_Santorini", "url": "https://en.wikipedia.org/wiki/Akrotiri%2C_Santorini"}, {"rank": 4, "page_id": "Al_Santorini", "url": "https://en.wikipedia.org/wiki/Al_Santorini"}, {"rank": 5, "page_id": "Emporio,_Santorini", "url": "https://en.wikipedia.org/wiki/Emporio%2C_Santorini"}, {"rank": 6, "page_id": "Perissa,_Santorini", "url": "https://en.wikipedia.org/wiki/Perissa%2C_Santorini"}, {"rank": 7, "page_id": "Santorini_affair", "url": "https://en.wikipedia.org/wiki/Santorini_affair"}, {"rank": 8, "page_id": "Santorini_caldera", "url": "https://en.wikipedia.org/wiki/Santorini_caldera"}, {"rank": 9, "page_id": "Santorini_tomato", "url": "https://en.wikipedia.org/wiki/Santorini_tomato"}, {"rank": 10, "page_id": "Zodarion_santorini", "url": "https://en.wikipedia.org/wiki/Zodarion_santorini"}]}\n{"id": 173001, "claim": "Rear Window (1998 film) was one of Christopher Reeve\'s first appearances on film.", "ranked_articles": [{"rank": 1, "page_id": "Rear_Window", "url": "https://en.wikipedia.org/wiki/Rear_Window"}, {"rank": 2, "page_id": "Rear_Window_-LRB-1998_film-RRB-", "url": "https://en.wikipedia.org/wiki/Rear_Window_%281998_film%29"}, {"rank": 3, "page_id": "Rear_Window_Captioning_System", "url": "https://en.wikipedia.org/wiki/Rear_Window_Captioning_System"}, {"rank": 4, "page_id": "Window_film", "url": "https://en.wikipedia.org/wiki/Window_film"}, {"rank": 5, "page_id": "Rear_Window_-LRB-disambiguation-RRB-", "url": "https://en.wikipedia.org/wiki/Rear_Window_%28disambiguation%29"}, {"rank": 6, "page_id": "Rear", "url": "https://en.wikipedia.org/wiki/Rear"}, {"rank": 7, "page_id": "Window_insulation_film", "url": "https://en.wikipedia.org/wiki/Window_insulation_film"}, {"rank": 8, "page_id": "Window", "url": "https://en.wikipedia.org/wiki/Window"}, {"rank": 9, "page_id": "Rear_End", "url": "https://en.wikipedia.org/wiki/Rear_End"}, {"rank": 10, "page_id": "Rear_Enz", "url": "https://en.wikipedia.org/wiki/Rear_Enz"}]}\n{"id": 128610, "claim": "Back to the Future Part II was released in 1988.", "ranked_articles": [{"rank": 1, "page_id": "Back_to_the_Future_Part_II", "url": "https://en.wikipedia.org/wiki/Back_to_the_Future_Part_II"}, {"rank": 2, "page_id": "Back_to_the_Future_Part_II_&_III", "url": "https://en.wikipedia.org/wiki/Back_to_the_Future_Part_II_%26_III"}, {"rank": 3, "page_id": "Back_to_the_Future_Part_III", "url": "https://en.wikipedia.org/wiki/Back_to_the_Future_Part_III"}, {"rank": 4, "page_id": "Once_and_Future_King_Part_II", "url": "https://en.wikipedia.org/wiki/Once_and_Future_King_Part_II"}, {"rank": 5, "page_id": "Back-released_velar_click", "url": "https://en.wikipedia.org/wiki/Back-released_velar_click"}, {"rank": 6, "page_id": "Adolescence_Part_II", "url": "https://en.wikipedia.org/wiki/Adolescence_Part_II"}, {"rank": 7, "page_id": "Bacdafucup_Part_II", "url": "https://en.wikipedia.org/wiki/Bacdafucup_Part_II"}, {"rank": 8, "page_id": "Candide,_Part_II", "url": "https://en.wikipedia.org/wiki/Candide%2C_Part_II"}, {"rank": 9, "page_id": "Confessions_Part_II", "url": "https://en.wikipedia.org/wiki/Confessions_Part_II"}, {"rank": 10, "page_id": "ELO_Part_II", "url": "https://en.wikipedia.org/wiki/ELO_Part_II"}]}\n{"id": 6919, "claim": "Kajal Aggarwal was an actress in Tamil cinema.", "ranked_articles": [{"rank": 1, "page_id": "Kajal_Aggarwal", "url": "https://en.wikipedia.org/wiki/Kajal_Aggarwal"}, {"rank": 2, "page_id": "Kajal_Aggarwal_filmography", "url": "https://en.wikipedia.org/wiki/Kajal_Aggarwal_filmography"}, {"rank": 3, "page_id": "Tamil_cinema", "url": "https://en.wikipedia.org/wiki/Tamil_cinema"}, {"rank": 4, "page_id": "Malaysian_Tamil_cinema", "url": "https://en.wikipedia.org/wiki/Malaysian_Tamil_cinema"}, {"rank": 5, "page_id": "Cinema_Express_Award_for_Best_Actress_–_Tamil", "url": "https://en.wikipedia.org/wiki/Cinema_Express_Award_for_Best_Actress_%E2%80%93_Tamil"}, {"rank": 6, "page_id": "Aggarwal", "url": "https://en.wikipedia.org/wiki/Aggarwal"}, {"rank": 7, "page_id": "Sri_Lankan_Tamil_cinema", "url": "https://en.wikipedia.org/wiki/Sri_Lankan_Tamil_cinema"}, {"rank": 8, "page_id": "Kajal,_Iran", "url": "https://en.wikipedia.org/wiki/Kajal%2C_Iran"}, {"rank": 9, "page_id": "Kajal_Ahmad", "url": "https://en.wikipedia.org/wiki/Kajal_Ahmad"}, {"rank": 10, "page_id": "Kajal_Bandyopadhyay", "url": "https://en.wikipedia.org/wiki/Kajal_Bandyopadhyay"}]}\n{"id": 117470, "claim": "Lydia Hearst was born on Thursday September 19, 1984.", "ranked_articles": [{"rank": 1, "page_id": "Lydia_Hearst", "url": "https://en.wikipedia.org/wiki/Lydia_Hearst"}, {"rank": 2, "page_id": "Hearst", "url": "https://en.wikipedia.org/wiki/Hearst"}, {"rank": 3, "page_id": "Thursday", "url": "https://en.wikipedia.org/wiki/Thursday"}, {"rank": 4, "page_id": "19_September_2010_Baghdad_bombings", "url": "https://en.wikipedia.org/wiki/19_September_2010_Baghdad_bombings"}, {"rank": 5, "page_id": "Lydia", "url": "https://en.wikipedia.org/wiki/Lydia"}, {"rank": 6, "page_id": "Amanda_Hearst", "url": "https://en.wikipedia.org/wiki/Amanda_Hearst"}, {"rank": 7, "page_id": "Anne_Hearst", "url": "https://en.wikipedia.org/wiki/Anne_Hearst"}, {"rank": 8, "page_id": "Gabriela_Hearst", "url": "https://en.wikipedia.org/wiki/Gabriela_Hearst"}, {"rank": 9, "page_id": "Garrison_Hearst", "url": "https://en.wikipedia.org/wiki/Garrison_Hearst"}, {"rank": 10, "page_id": "George_Hearst", "url": "https://en.wikipedia.org/wiki/George_Hearst"}]}\n{"id": 56064, "claim": "One America News Network is in competition with OANN.", "ranked_articles": [{"rank": 1, "page_id": "One_America_News_Network", "url": "https://en.wikipedia.org/wiki/One_America_News_Network"}, {"rank": 2, "page_id": "European_Competition_Network", "url": "https://en.wikipedia.org/wiki/European_Competition_Network"}, {"rank": 3, "page_id": "International_Competition_Network", "url": "https://en.wikipedia.org/wiki/International_Competition_Network"}, {"rank": 4, "page_id": "News_One", "url": "https://en.wikipedia.org/wiki/News_One"}, {"rank": 5, "page_id": "One_News", "url": "https://en.wikipedia.org/wiki/One_News"}, {"rank": 6, "page_id": "Network_One", "url": "https://en.wikipedia.org/wiki/Network_One"}, {"rank": 7, "page_id": "America_One", "url": "https://en.wikipedia.org/wiki/America_One"}, {"rank": 8, "page_id": "One_America", "url": "https://en.wikipedia.org/wiki/One_America"}, {"rank": 9, "page_id": "Anime_News_Network", "url": "https://en.wikipedia.org/wiki/Anime_News_Network"}, {"rank": 10, "page_id": "Arab_News_Network", "url": "https://en.wikipedia.org/wiki/Arab_News_Network"}]}\n{"id": 10994, "claim": "Paul Walker starred in the film Into the Blue.", "ranked_articles": [{"rank": 1, "page_id": "Paul_Walker", "url": "https://en.wikipedia.org/wiki/Paul_Walker"}, {"rank": 2, "page_id": "Back_into_Blue", "url": "https://en.wikipedia.org/wiki/Back_into_Blue"}, {"rank": 3, "page_id": "Into_the_Blue", "url": "https://en.wikipedia.org/wiki/Into_the_Blue"}, {"rank": 4, "page_id": "Into_Film", "url": "https://en.wikipedia.org/wiki/Into_Film"}, {"rank": 5, "page_id": "Starred_Up", "url": "https://en.wikipedia.org/wiki/Starred_Up"}, {"rank": 6, "page_id": "Starred_transform", "url": "https://en.wikipedia.org/wiki/Starred_transform"}, {"rank": 7, "page_id": "Paul_Baxendale-Walker", "url": "https://en.wikipedia.org/wiki/Paul_Baxendale-Walker"}, {"rank": 8, "page_id": "Green_Shading_into_Blue", "url": "https://en.wikipedia.org/wiki/Green_Shading_into_Blue"}, {"rank": 9, "page_id": "Into_the_Blue_Again", "url": "https://en.wikipedia.org/wiki/Into_the_Blue_Again"}, {"rank": 10, "page_id": "Into_the_Still_Blue", "url": "https://en.wikipedia.org/wiki/Into_the_Still_Blue"}]}\n{"id": 3833, "claim": "Beauty Shop is a spin-off of the Barbershop film franchise.", "ranked_articles": [{"rank": 1, "page_id": "Off_spin", "url": "https://en.wikipedia.org/wiki/Off_spin"}, {"rank": 2, "page_id": "Spin-off", "url": "https://en.wikipedia.org/wiki/Spin-off"}, {"rank": 3, "page_id": "Beauty_Shop", "url": "https://en.wikipedia.org/wiki/Beauty_Shop"}, {"rank": 4, "page_id": "Beauty_shop", "url": "https://en.wikipedia.org/wiki/Beauty_shop"}, {"rank": 5, "page_id": "Corporate_spin-off", "url": "https://en.wikipedia.org/wiki/Corporate_spin-off"}, {"rank": 6, "page_id": "Economic_spin-off", "url": "https://en.wikipedia.org/wiki/Economic_spin-off"}, {"rank": 7, "page_id": "Government_spin-off", "url": "https://en.wikipedia.org/wiki/Government_spin-off"}, {"rank": 8, "page_id": "Research_spin-off", "url": "https://en.wikipedia.org/wiki/Research_spin-off"}, {"rank": 9, "page_id": "University_spin-off", "url": "https://en.wikipedia.org/wiki/University_spin-off"}, {"rank": 10, "page_id": "The_Beauty_Shop_-LRB-film-RRB-", "url": "https://en.wikipedia.org/wiki/The_Beauty_Shop_%28film%29"}]}\n{"id": 192941, "claim": "There was an artist named William Hanna.", "ranked_articles": [{"rank": 1, "page_id": "William_Hanna", "url": "https://en.wikipedia.org/wiki/William_Hanna"}, {"rank": 2, "page_id": "John_William_Hanna", "url": "https://en.wikipedia.org/wiki/John_William_Hanna"}, {"rank": 3, "page_id": "William_B._Hanna", "url": "https://en.wikipedia.org/wiki/William_B._Hanna"}, {"rank": 4, "page_id": "William_Hanna_Clarke", "url": "https://en.wikipedia.org/wiki/William_Hanna_Clarke"}, {"rank": 5, "page_id": "William_Hanna_Shomali", "url": "https://en.wikipedia.org/wiki/William_Hanna_Shomali"}, {"rank": 6, "page_id": "William_John_Hanna", "url": "https://en.wikipedia.org/wiki/William_John_Hanna"}, {"rank": 7, "page_id": "William_S._Hanna", "url": "https://en.wikipedia.org/wiki/William_S._Hanna"}, {"rank": 8, "page_id": "William_Selim_Hanna", "url": "https://en.wikipedia.org/wiki/William_Selim_Hanna"}, {"rank": 9, "page_id": "William_T._Hanna", "url": "https://en.wikipedia.org/wiki/William_T._Hanna"}, {"rank": 10, "page_id": "Hanna_Hanna", "url": "https://en.wikipedia.org/wiki/Hanna_Hanna"}]}\n{"id": 22452, "claim": "The Titanic\'s wreck was first discovered over 70 years after the sinking.", "ranked_articles": [{"rank": 1, "page_id": "Wreck_of_the_RMS_Titanic", "url": "https://en.wikipedia.org/wiki/Wreck_of_the_RMS_Titanic"}, {"rank": 2, "page_id": "DiscoverEd", "url": "https://en.wikipedia.org/wiki/DiscoverEd"}, {"rank": 3, "page_id": "Wreck", "url": "https://en.wikipedia.org/wiki/Wreck"}, {"rank": 4, "page_id": "Discovered_Again!", "url": "https://en.wikipedia.org/wiki/Discovered_Again%21"}, {"rank": 5, "page_id": "Discovered_attack", "url": "https://en.wikipedia.org/wiki/Discovered_attack"}, {"rank": 6, "page_id": "Poverty_Discovered", "url": "https://en.wikipedia.org/wiki/Poverty_Discovered"}, {"rank": 7, "page_id": "Species_first_discovered_in_Hong_Kong", "url": "https://en.wikipedia.org/wiki/Species_first_discovered_in_Hong_Kong"}, {"rank": 8, "page_id": "Cinematic_Titanic", "url": "https://en.wikipedia.org/wiki/Cinematic_Titanic"}, {"rank": 9, "page_id": "Jinx_Titanic", "url": "https://en.wikipedia.org/wiki/Jinx_Titanic"}, {"rank": 10, "page_id": "MY_Titanic", "url": "https://en.wikipedia.org/wiki/MY_Titanic"}]}\n{"id": 139219, "claim": "Beverly Hills, 90210 addressed numerous topical issues such as date rape, gay rights, animal rights, alcoholism, drug abuse, domestic violence, sex, antisemitism, racism, teenage suicide, teenage pregnancy, and AIDS.", "ranked_articles": [{"rank": 1, "page_id": "Beverly_Hills,_90210", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210"}, {"rank": 2, "page_id": "Beverly_Hills,_90210_-LRB-franchise-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210_%28franchise%29"}, {"rank": 3, "page_id": "Beverly_Hills_90210_-LRB-soundtrack-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills_90210_%28soundtrack%29"}, {"rank": 4, "page_id": "List_of_Beverly_Hills,_90210_characters", "url": "https://en.wikipedia.org/wiki/List_of_Beverly_Hills%2C_90210_characters"}, {"rank": 5, "page_id": "List_of_Beverly_Hills,_90210_episodes", "url": "https://en.wikipedia.org/wiki/List_of_Beverly_Hills%2C_90210_episodes"}, {"rank": 6, "page_id": "The_Unauthorized_Beverly_Hills,_90210_Story", "url": "https://en.wikipedia.org/wiki/The_Unauthorized_Beverly_Hills%2C_90210_Story"}, {"rank": 7, "page_id": "Beverly_Hills,_90210_-LRB-season_1-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210_%28season_1%29"}, {"rank": 8, "page_id": "Beverly_Hills,_90210_-LRB-season_10-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210_%28season_10%29"}, {"rank": 9, "page_id": "Beverly_Hills,_90210_-LRB-season_2-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210_%28season_2%29"}, {"rank": 10, "page_id": "Beverly_Hills,_90210_-LRB-season_3-RRB-", "url": "https://en.wikipedia.org/wiki/Beverly_Hills%2C_90210_%28season_3%29"}]}\n{"id": 157329, "claim": "Social justice is ignored today in philosophical debates about differences among human beings.", "ranked_articles": [{"rank": 1, "page_id": "Social_justice", "url": "https://en.wikipedia.org/wiki/Social_justice"}, {"rank": 2, "page_id": "Social_Philosophy_Today", "url": "https://en.wikipedia.org/wiki/Social_Philosophy_Today"}, {"rank": 3, "page_id": "Social_Justice_Coalition", "url": "https://en.wikipedia.org/wiki/Social_Justice_Coalition"}, {"rank": 4, "page_id": "Social_Justice_Party", "url": "https://en.wikipedia.org/wiki/Social_Justice_Party"}, {"rank": 5, "page_id": "Social_Justice_Society", "url": "https://en.wikipedia.org/wiki/Social_Justice_Society"}, {"rank": 6, "page_id": "Social_Justice_conference", "url": "https://en.wikipedia.org/wiki/Social_Justice_conference"}, {"rank": 7, "page_id": "Social_justice_art", "url": "https://en.wikipedia.org/wiki/Social_justice_art"}, {"rank": 8, "page_id": "Social_justice_feminism", "url": "https://en.wikipedia.org/wiki/Social_justice_feminism"}, {"rank": 9, "page_id": "Social_justice_warrior", "url": "https://en.wikipedia.org/wiki/Social_justice_warrior"}, {"rank": 10, "page_id": "Today_Today", "url": "https://en.wikipedia.org/wiki/Today_Today"}]}\n{"id": 199900, "claim": "Metro Boomin is incapable of being a record producer and a songwriter.", "ranked_articles": [{"rank": 1, "page_id": "Metro_Boomin", "url": "https://en.wikipedia.org/wiki/Metro_Boomin"}, {"rank": 2, "page_id": "Record_producer", "url": "https://en.wikipedia.org/wiki/Record_producer"}, {"rank": 3, "page_id": "Metro_Boomin_production_discography", "url": "https://en.wikipedia.org/wiki/Metro_Boomin_production_discography"}, {"rank": 4, "page_id": "Drunk_and_Incapable", "url": "https://en.wikipedia.org/wiki/Drunk_and_Incapable"}, {"rank": 5, "page_id": "The_Boomin\'_System", "url": "https://en.wikipedia.org/wiki/The_Boomin%27_System"}, {"rank": 6, "page_id": "Songwriter", "url": "https://en.wikipedia.org/wiki/Songwriter"}, {"rank": 7, "page_id": "40_-LRB-record_producer-RRB-", "url": "https://en.wikipedia.org/wiki/40_%28record_producer%29"}, {"rank": 8, "page_id": "6ix_-LRB-record_producer-RRB-", "url": "https://en.wikipedia.org/wiki/6ix_%28record_producer%29"}, {"rank": 9, "page_id": "Amadeus_-LRB-record_producer-RRB-", "url": "https://en.wikipedia.org/wiki/Amadeus_%28record_producer%29"}, {"rank": 10, "page_id": "Aqua_-LRB-record_producer-RRB-", "url": "https://en.wikipedia.org/wiki/Aqua_%28record_producer%29"}]}\n{"id": 44854, "claim": "Paul Newman has yet to win a single Academy Award.", "ranked_articles": [{"rank": 1, "page_id": "Paul_Newman", "url": "https://en.wikipedia.org/wiki/Paul_Newman"}, {"rank": 2, "page_id": "&_Yet_&_Yet", "url": "https://en.wikipedia.org/wiki/%26_Yet_%26_Yet"}, {"rank": 3, "page_id": "Paul_Newman_filmography", "url": "https://en.wikipedia.org/wiki/Paul_Newman_filmography"}, {"rank": 4, "page_id": "Paul_S._Newman", "url": "https://en.wikipedia.org/wiki/Paul_S._Newman"}, {"rank": 5, "page_id": "Yet", "url": "https://en.wikipedia.org/wiki/Yet"}, {"rank": 6, "page_id": "Paul_and_Win_Grace", "url": "https://en.wikipedia.org/wiki/Paul_and_Win_Grace"}, {"rank": 7, "page_id": "Win-win_game", "url": "https://en.wikipedia.org/wiki/Win-win_game"}, {"rank": 8, "page_id": "Win_Win_Myint", "url": "https://en.wikipedia.org/wiki/Win_Win_Myint"}, {"rank": 9, "page_id": ".win", "url": "https://en.wikipedia.org/wiki/.win"}, {"rank": 10, "page_id": "Win", "url": "https://en.wikipedia.org/wiki/Win"}]}\n{"id": 138808, "claim": "Gwen Stefani always declines to perform.", "ranked_articles": [{"rank": 1, "page_id": "Gwen_Stefani", "url": "https://en.wikipedia.org/wiki/Gwen_Stefani"}, {"rank": 2, "page_id": "Gwen_Stefani_discography", "url": "https://en.wikipedia.org/wiki/Gwen_Stefani_discography"}, {"rank": 3, "page_id": "PerForm", "url": "https://en.wikipedia.org/wiki/PerForm"}, {"rank": 4, "page_id": "Cool_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Cool_%28Gwen_Stefani_song%29"}, {"rank": 5, "page_id": "Crash_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Crash_%28Gwen_Stefani_song%29"}, {"rank": 6, "page_id": "MasterCard_Priceless_Surprises_Presents_Gwen_Stefani", "url": "https://en.wikipedia.org/wiki/MasterCard_Priceless_Surprises_Presents_Gwen_Stefani"}, {"rank": 7, "page_id": "Misery_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Misery_%28Gwen_Stefani_song%29"}, {"rank": 8, "page_id": "Serious_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Serious_%28Gwen_Stefani_song%29"}, {"rank": 9, "page_id": "Shine_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Shine_%28Gwen_Stefani_song%29"}, {"rank": 10, "page_id": "Yummy_-LRB-Gwen_Stefani_song-RRB-", "url": "https://en.wikipedia.org/wiki/Yummy_%28Gwen_Stefani_song%29"}]}\n{"id": 115715, "claim": "Thandie Newton is incapable of being an actress.", "ranked_articles": [{"rank": 1, "page_id": "Thandie_Newton", "url": "https://en.wikipedia.org/wiki/Thandie_Newton"}, {"rank": 2, "page_id": "Drunk_and_Incapable", "url": "https://en.wikipedia.org/wiki/Drunk_and_Incapable"}, {"rank": 3, "page_id": "Newton", "url": "https://en.wikipedia.org/wiki/Newton"}, {"rank": 4, "page_id": ".256_Newton", "url": "https://en.wikipedia.org/wiki/.256_Newton"}, {"rank": 5, "page_id": ".30_Newton", "url": "https://en.wikipedia.org/wiki/.30_Newton"}, {"rank": 6, "page_id": ".35_Newton", "url": "https://en.wikipedia.org/wiki/.35_Newton"}, {"rank": 7, "page_id": "Adam_Newton", "url": "https://en.wikipedia.org/wiki/Adam_Newton"}, {"rank": 8, "page_id": "Alan_Newton", "url": "https://en.wikipedia.org/wiki/Alan_Newton"}, {"rank": 9, "page_id": "Alfred_Newton", "url": "https://en.wikipedia.org/wiki/Alfred_Newton"}, {"rank": 10, "page_id": "Algernon_Newton", "url": "https://en.wikipedia.org/wiki/Algernon_Newton"}]}\n{"id": 46447, "claim": "Nick Jonas was put out by a subsection of Universal Music Group.", "ranked_articles": [{"rank": 1, "page_id": "Nick_Jonas", "url": "https://en.wikipedia.org/wiki/Nick_Jonas"}, {"rank": 2, "page_id": "Nick_Jonas_Live", "url": "https://en.wikipedia.org/wiki/Nick_Jonas_Live"}, {"rank": 3, "page_id": "Nick_Jonas_discography", "url": "https://en.wikipedia.org/wiki/Nick_Jonas_discography"}, {"rank": 4, "page_id": "Put_Him_Out", "url": "https://en.wikipedia.org/wiki/Put_Him_Out"}, {"rank": 5, "page_id": "Nick_Jonas_&_the_Administration", "url": "https://en.wikipedia.org/wiki/Nick_Jonas_%26_the_Administration"}, {"rank": 6, "page_id": "Put_Out_More_Flags", "url": "https://en.wikipedia.org/wiki/Put_Out_More_Flags"}, {"rank": 7, "page_id": "Put_Out_the_Light", "url": "https://en.wikipedia.org/wiki/Put_Out_the_Light"}, {"rank": 8, "page_id": "Put_That_Light_Out!", "url": "https://en.wikipedia.org/wiki/Put_That_Light_Out%21"}, {"rank": 9, "page_id": "We_Put_Out_Records", "url": "https://en.wikipedia.org/wiki/We_Put_Out_Records"}, {"rank": 10, "page_id": "Nick_Jonas_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/Nick_Jonas_%28album%29"}]}\n{"id": 170747, "claim": "Anthony Meindl has coached and worked with Ian Harding.", "ranked_articles": [{"rank": 1, "page_id": "Anthony_Meindl", "url": "https://en.wikipedia.org/wiki/Anthony_Meindl"}, {"rank": 2, "page_id": "Ian_Harding", "url": "https://en.wikipedia.org/wiki/Ian_Harding"}, {"rank": 3, "page_id": "Meindl", "url": "https://en.wikipedia.org/wiki/Meindl"}, {"rank": 4, "page_id": "Eugen_Meindl", "url": "https://en.wikipedia.org/wiki/Eugen_Meindl"}, {"rank": 5, "page_id": "Léonardo_Meindl", "url": "https://en.wikipedia.org/wiki/L%C3%A9onardo_Meindl"}, {"rank": 6, "page_id": "Anthony_Ian_Berkeley", "url": "https://en.wikipedia.org/wiki/Anthony_Ian_Berkeley"}, {"rank": 7, "page_id": "Ian_Anthony_Dale", "url": "https://en.wikipedia.org/wiki/Ian_Anthony_Dale"}, {"rank": 8, "page_id": "James_D._Meindl", "url": "https://en.wikipedia.org/wiki/James_D._Meindl"}, {"rank": 9, "page_id": "All_Worked_Up", "url": "https://en.wikipedia.org/wiki/All_Worked_Up"}, {"rank": 10, "page_id": "Worked-example_effect", "url": "https://en.wikipedia.org/wiki/Worked-example_effect"}]}\n{"id": 26908, "claim": "Billy Joel is in the performing arts industry.", "ranked_articles": [{"rank": 1, "page_id": "Performing_arts", "url": "https://en.wikipedia.org/wiki/Performing_arts"}, {"rank": 2, "page_id": "Billy_Joel", "url": "https://en.wikipedia.org/wiki/Billy_Joel"}, {"rank": 3, "page_id": "AHDS_Performing_Arts", "url": "https://en.wikipedia.org/wiki/AHDS_Performing_Arts"}, {"rank": 4, "page_id": "Aberdeen_Performing_Arts", "url": "https://en.wikipedia.org/wiki/Aberdeen_Performing_Arts"}, {"rank": 5, "page_id": "BD_Performing_Arts", "url": "https://en.wikipedia.org/wiki/BD_Performing_Arts"}, {"rank": 6, "page_id": "Gallimaufry_Performing_Arts", "url": "https://en.wikipedia.org/wiki/Gallimaufry_Performing_Arts"}, {"rank": 7, "page_id": "Mu_Performing_Arts", "url": "https://en.wikipedia.org/wiki/Mu_Performing_Arts"}, {"rank": 8, "page_id": "Performing_Arts_Building", "url": "https://en.wikipedia.org/wiki/Performing_Arts_Building"}, {"rank": 9, "page_id": "Performing_Arts_Center", "url": "https://en.wikipedia.org/wiki/Performing_Arts_Center"}, {"rank": 10, "page_id": "Performing_Arts_Collection", "url": "https://en.wikipedia.org/wiki/Performing_Arts_Collection"}]}\n{"id": 188504, "claim": "Tate Ellington appeared in a CBS thriller series.", "ranked_articles": [{"rank": 1, "page_id": "Tate_Ellington", "url": "https://en.wikipedia.org/wiki/Tate_Ellington"}, {"rank": 2, "page_id": "Thriller", "url": "https://en.wikipedia.org/wiki/Thriller"}, {"rank": 3, "page_id": "The_Appeared", "url": "https://en.wikipedia.org/wiki/The_Appeared"}, {"rank": 4, "page_id": "Ellington", "url": "https://en.wikipedia.org/wiki/Ellington"}, {"rank": 5, "page_id": "CBS", "url": "https://en.wikipedia.org/wiki/CBS"}, {"rank": 6, "page_id": "Tate", "url": "https://en.wikipedia.org/wiki/Tate"}, {"rank": 7, "page_id": "Armchair_Thriller", "url": "https://en.wikipedia.org/wiki/Armchair_Thriller"}, {"rank": 8, "page_id": "Chiller_Thriller", "url": "https://en.wikipedia.org/wiki/Chiller_Thriller"}, {"rank": 9, "page_id": "Comedy_thriller", "url": "https://en.wikipedia.org/wiki/Comedy_thriller"}, {"rank": 10, "page_id": "Erotic_thriller", "url": "https://en.wikipedia.org/wiki/Erotic_thriller"}]}\n{"id": 93642, "claim": "There is a writer named Marta Kauffman.", "ranked_articles": [{"rank": 1, "page_id": "Marta_Kauffman", "url": "https://en.wikipedia.org/wiki/Marta_Kauffman"}, {"rank": 2, "page_id": "Marta", "url": "https://en.wikipedia.org/wiki/Marta"}, {"rank": 3, "page_id": "Allan_Kauffman", "url": "https://en.wikipedia.org/wiki/Allan_Kauffman"}, {"rank": 4, "page_id": "Andrene_Kauffman", "url": "https://en.wikipedia.org/wiki/Andrene_Kauffman"}, {"rank": 5, "page_id": "Angelica_Kauffman", "url": "https://en.wikipedia.org/wiki/Angelica_Kauffman"}, {"rank": 6, "page_id": "Anne_Kauffman", "url": "https://en.wikipedia.org/wiki/Anne_Kauffman"}, {"rank": 7, "page_id": "Bill_Kauffman", "url": "https://en.wikipedia.org/wiki/Bill_Kauffman"}, {"rank": 8, "page_id": "Bob_Kauffman", "url": "https://en.wikipedia.org/wiki/Bob_Kauffman"}, {"rank": 9, "page_id": "Casey_Kauffman", "url": "https://en.wikipedia.org/wiki/Casey_Kauffman"}, {"rank": 10, "page_id": "Clark_Kauffman", "url": "https://en.wikipedia.org/wiki/Clark_Kauffman"}]}\n{"id": 108126, "claim": "2001: A Space Odyssey was not edited in England.", "ranked_articles": [{"rank": 1, "page_id": "Space_Odyssey", "url": "https://en.wikipedia.org/wiki/Space_Odyssey"}, {"rank": 2, "page_id": "2001-COLON-_A_Space_Odyssey", "url": "https://en.wikipedia.org/wiki/2001%3A_A_Space_Odyssey"}, {"rank": 3, "page_id": "2001-COLON-_A_Space_Road_Odyssey", "url": "https://en.wikipedia.org/wiki/2001%3A_A_Space_Road_Odyssey"}, {"rank": 4, "page_id": "Odyssey_Space_Research", "url": "https://en.wikipedia.org/wiki/Odyssey_Space_Research"}, {"rank": 5, "page_id": "2001_Mars_Odyssey", "url": "https://en.wikipedia.org/wiki/2001_Mars_Odyssey"}, {"rank": 6, "page_id": "Interpretations_of_2001-COLON-_A_Space_Odyssey", "url": "https://en.wikipedia.org/wiki/Interpretations_of_2001%3A_A_Space_Odyssey"}, {"rank": 7, "page_id": "Technologies_in_2001-COLON-_A_Space_Odyssey", "url": "https://en.wikipedia.org/wiki/Technologies_in_2001%3A_A_Space_Odyssey"}, {"rank": 8, "page_id": "Edited_volume", "url": "https://en.wikipedia.org/wiki/Edited_volume"}, {"rank": 9, "page_id": "2001-COLON-_A_Space_Odyssey_-LRB-comics-RRB-", "url": "https://en.wikipedia.org/wiki/2001%3A_A_Space_Odyssey_%28comics%29"}, {"rank": 10, "page_id": "2001-COLON-_A_Space_Odyssey_-LRB-disambiguation-RRB-", "url": "https://en.wikipedia.org/wiki/2001%3A_A_Space_Odyssey_%28disambiguation%29"}]}\n{"id": 145550, "claim": "Pineapple Express is a 2008 film.", "ranked_articles": [{"rank": 1, "page_id": "Pineapple_Express", "url": "https://en.wikipedia.org/wiki/Pineapple_Express"}, {"rank": 2, "page_id": "Pineapple_Express_-LRB-film-RRB-", "url": "https://en.wikipedia.org/wiki/Pineapple_Express_%28film%29"}, {"rank": 3, "page_id": "Pineapple", "url": "https://en.wikipedia.org/wiki/Pineapple"}, {"rank": 4, "page_id": "Azucarron_pineapple", "url": "https://en.wikipedia.org/wiki/Azucarron_pineapple"}, {"rank": 5, "page_id": "Big_Pineapple", "url": "https://en.wikipedia.org/wiki/Big_Pineapple"}, {"rank": 6, "page_id": "Dunmore_Pineapple", "url": "https://en.wikipedia.org/wiki/Dunmore_Pineapple"}, {"rank": 7, "page_id": "Pineapple_Air", "url": "https://en.wikipedia.org/wiki/Pineapple_Air"}, {"rank": 8, "page_id": "Pineapple_Army", "url": "https://en.wikipedia.org/wiki/Pineapple_Army"}, {"rank": 9, "page_id": "Pineapple_Bowl", "url": "https://en.wikipedia.org/wiki/Pineapple_Bowl"}, {"rank": 10, "page_id": "Pineapple_Head", "url": "https://en.wikipedia.org/wiki/Pineapple_Head"}]}\n{"id": 201271, "claim": "Natascha McElhone starred in Ronin.", "ranked_articles": [{"rank": 1, "page_id": "Natascha_McElhone", "url": "https://en.wikipedia.org/wiki/Natascha_McElhone"}, {"rank": 2, "page_id": "McElhone", "url": "https://en.wikipedia.org/wiki/McElhone"}, {"rank": 3, "page_id": "Rōnin", "url": "https://en.wikipedia.org/wiki/R%C5%8Dnin"}, {"rank": 4, "page_id": "Eric_McElhone", "url": "https://en.wikipedia.org/wiki/Eric_McElhone"}, {"rank": 5, "page_id": "Frank_McElhone", "url": "https://en.wikipedia.org/wiki/Frank_McElhone"}, {"rank": 6, "page_id": "Helen_McElhone", "url": "https://en.wikipedia.org/wiki/Helen_McElhone"}, {"rank": 7, "page_id": "Jack_McElhone", "url": "https://en.wikipedia.org/wiki/Jack_McElhone"}, {"rank": 8, "page_id": "John_McElhone", "url": "https://en.wikipedia.org/wiki/John_McElhone"}, {"rank": 9, "page_id": "Johnny_McElhone", "url": "https://en.wikipedia.org/wiki/Johnny_McElhone"}, {"rank": 10, "page_id": "Starred_Up", "url": "https://en.wikipedia.org/wiki/Starred_Up"}]}\n{"id": 207063, "claim": "Tom Hiddleston appeared in a Michael Bay film.", "ranked_articles": [{"rank": 1, "page_id": "Tom_Hiddleston", "url": "https://en.wikipedia.org/wiki/Tom_Hiddleston"}, {"rank": 2, "page_id": "Hiddleston", "url": "https://en.wikipedia.org/wiki/Hiddleston"}, {"rank": 3, "page_id": "Hugh_Hiddleston", "url": "https://en.wikipedia.org/wiki/Hugh_Hiddleston"}, {"rank": 4, "page_id": "Syd_Hiddleston", "url": "https://en.wikipedia.org/wiki/Syd_Hiddleston"}, {"rank": 5, "page_id": "Michael_Tom", "url": "https://en.wikipedia.org/wiki/Michael_Tom"}, {"rank": 6, "page_id": "Michael_Bay", "url": "https://en.wikipedia.org/wiki/Michael_Bay"}, {"rank": 7, "page_id": "The_Appeared", "url": "https://en.wikipedia.org/wiki/The_Appeared"}, {"rank": 8, "page_id": "Michael\'s_Bay", "url": "https://en.wikipedia.org/wiki/Michael%27s_Bay"}, {"rank": 9, "page_id": "Michael_Bay_filmography", "url": "https://en.wikipedia.org/wiki/Michael_Bay_filmography"}, {"rank": 10, "page_id": "Tom_Tom_Tom", "url": "https://en.wikipedia.org/wiki/Tom_Tom_Tom"}]}\n{"id": 125911, "claim": "The Ozarks are a region of the United States.", "ranked_articles": [{"rank": 1, "page_id": "Ozarks", "url": "https://en.wikipedia.org/wiki/Ozarks"}, {"rank": 2, "page_id": "United_States_physiographic_region", "url": "https://en.wikipedia.org/wiki/United_States_physiographic_region"}, {"rank": 3, "page_id": "Malibu_Ozarks", "url": "https://en.wikipedia.org/wiki/Malibu_Ozarks"}, {"rank": 4, "page_id": "United_States", "url": "https://en.wikipedia.org/wiki/United_States"}, {"rank": 5, "page_id": "Ozarks_Community_Hospital", "url": "https://en.wikipedia.org/wiki/Ozarks_Community_Hospital"}, {"rank": 6, "page_id": "Ozarks_Electric_Cooperative", "url": "https://en.wikipedia.org/wiki/Ozarks_Electric_Cooperative"}, {"rank": 7, "page_id": "United_States_Army_Installation_Management_Command_Korea_Region", "url": "https://en.wikipedia.org/wiki/United_States_Army_Installation_Management_Command_Korea_Region"}, {"rank": 8, "page_id": "United_States_v._United_States_District_Court", "url": "https://en.wikipedia.org/wiki/United_States_v._United_States_District_Court"}, {"rank": 9, "page_id": "United_States_v._United_States_Gypsum_Co.", "url": "https://en.wikipedia.org/wiki/United_States_v._United_States_Gypsum_Co."}, {"rank": 10, "page_id": "Central_United_States", "url": "https://en.wikipedia.org/wiki/Central_United_States"}]}\n{"id": 123370, "claim": "The nickname of Borussia Dortmund is die Schwarzgelben.", "ranked_articles": [{"rank": 1, "page_id": "Borussia_Dortmund", "url": "https://en.wikipedia.org/wiki/Borussia_Dortmund"}, {"rank": 2, "page_id": "Borussia_Dortmund_Handball", "url": "https://en.wikipedia.org/wiki/Borussia_Dortmund_Handball"}, {"rank": 3, "page_id": "Borussia_Dortmund_II", "url": "https://en.wikipedia.org/wiki/Borussia_Dortmund_II"}, {"rank": 4, "page_id": "Borussia_Mönchengladbach_12–0_Borussia_Dortmund", "url": "https://en.wikipedia.org/wiki/Borussia_M%C3%B6nchengladbach_12%E2%80%930_Borussia_Dortmund"}, {"rank": 5, "page_id": "Borussia_Dortmund_Youth_Sector", "url": "https://en.wikipedia.org/wiki/Borussia_Dortmund_Youth_Sector"}, {"rank": 6, "page_id": "1963–64_Borussia_Dortmund_season", "url": "https://en.wikipedia.org/wiki/1963%E2%80%9364_Borussia_Dortmund_season"}, {"rank": 7, "page_id": "1996–97_Borussia_Dortmund_season", "url": "https://en.wikipedia.org/wiki/1996%E2%80%9397_Borussia_Dortmund_season"}, {"rank": 8, "page_id": "2000–01_Borussia_Dortmund_season", "url": "https://en.wikipedia.org/wiki/2000%E2%80%9301_Borussia_Dortmund_season"}, {"rank": 9, "page_id": "2001–02_Borussia_Dortmund_season", "url": "https://en.wikipedia.org/wiki/2001%E2%80%9302_Borussia_Dortmund_season"}, {"rank": 10, "page_id": "2002–03_Borussia_Dortmund_season", "url": "https://en.wikipedia.org/wiki/2002%E2%80%9303_Borussia_Dortmund_season"}]}\n{"id": 169335, "claim": "Sam Walton died on April 5th, 1992 in his childhood home.", "ranked_articles": [{"rank": 1, "page_id": "Sam_Walton", "url": "https://en.wikipedia.org/wiki/Sam_Walton"}, {"rank": 2, "page_id": "April,_april", "url": "https://en.wikipedia.org/wiki/April%2C_april"}, {"rank": 3, "page_id": "April_1992_-LRB-album-RRB-", "url": "https://en.wikipedia.org/wiki/April_1992_%28album%29"}, {"rank": 4, "page_id": "April_1992_Nepalese_general_strike", "url": "https://en.wikipedia.org/wiki/April_1992_Nepalese_general_strike"}, {"rank": 5, "page_id": "Walton", "url": "https://en.wikipedia.org/wiki/Walton"}, {"rank": 6, "page_id": "April", "url": "https://en.wikipedia.org/wiki/April"}, {"rank": 7, "page_id": "Sam_M._Walton_College_of_Business", "url": "https://en.wikipedia.org/wiki/Sam_M._Walton_College_of_Business"}, {"rank": 8, "page_id": "Sam_Walton_-LRB-American_football-RRB-", "url": "https://en.wikipedia.org/wiki/Sam_Walton_%28American_football%29"}, {"rank": 9, "page_id": "April_29,_1992_-LRB-Miami-RRB-", "url": "https://en.wikipedia.org/wiki/April_29%2C_1992_%28Miami%29"}, {"rank": 10, "page_id": "Enchanted_April_-LRB-1992_film-RRB-", "url": "https://en.wikipedia.org/wiki/Enchanted_April_%281992_film%29"}]}\n{"id": 85145, "claim": "Slovakia is officially known as a different name.", "ranked_articles": [{"rank": 1, "page_id": "Officially_Dead", "url": "https://en.wikipedia.org/wiki/Officially_Dead"}, {"rank": 2, "page_id": "Officially_Yours", "url": "https://en.wikipedia.org/wiki/Officially_Yours"}, {"rank": 3, "page_id": "Name_days_in_Slovakia", "url": "https://en.wikipedia.org/wiki/Name_days_in_Slovakia"}, {"rank": 4, "page_id": "Different_Kinda_Different", "url": "https://en.wikipedia.org/wiki/Different_Kinda_Different"}, {"rank": 5, "page_id": "Different", "url": "https://en.wikipedia.org/wiki/Different"}, {"rank": 6, "page_id": "I_Officially_Exist", "url": "https://en.wikipedia.org/wiki/I_Officially_Exist"}, {"rank": 7, "page_id": "Officially_Missing_You", "url": "https://en.wikipedia.org/wiki/Officially_Missing_You"}, {"rank": 8, "page_id": "Slovakia", "url": "https://en.wikipedia.org/wiki/Slovakia"}, {"rank": 9, "page_id": "Formerly_Known", "url": "https://en.wikipedia.org/wiki/Formerly_Known"}, {"rank": 10, "page_id": "Known_Associates", "url": "https://en.wikipedia.org/wiki/Known_Associates"}]}\n{"id": 154547, "claim": "Black Panther is a creation of fiction.", "ranked_articles": [{"rank": 1, "page_id": "Black_panther", "url": "https://en.wikipedia.org/wiki/Black_panther"}, {"rank": 2, "page_id": "Black_Fiction", "url": "https://en.wikipedia.org/wiki/Black_Fiction"}, {"rank": 3, "page_id": "Black_Panther_Party", "url": "https://en.wikipedia.org/wiki/Black_Panther_Party"}, {"rank": 4, "page_id": "K2_Black_Panther", "url": "https://en.wikipedia.org/wiki/K2_Black_Panther"}, {"rank": 5, "page_id": "Black_science_fiction", "url": "https://en.wikipedia.org/wiki/Black_science_fiction"}, {"rank": 6, "page_id": "Black_Panther_Premium_Stout", "url": "https://en.wikipedia.org/wiki/Black_Panther_Premium_Stout"}, {"rank": 7, "page_id": "Eldridge_Cleaver,_Black_Panther", "url": "https://en.wikipedia.org/wiki/Eldridge_Cleaver%2C_Black_Panther"}, {"rank": 8, "page_id": "New_Black_Panther_Party", "url": "https://en.wikipedia.org/wiki/New_Black_Panther_Party"}, {"rank": 9, "page_id": "The_Black_Panther_Warriors", "url": "https://en.wikipedia.org/wiki/The_Black_Panther_Warriors"}, {"rank": 10, "page_id": "Creation", "url": "https://en.wikipedia.org/wiki/Creation"}]}\n{"id": 84545, "claim": "Community is available on Hulu.", "ranked_articles": [{"rank": 1, "page_id": "Hulu", "url": "https://en.wikipedia.org/wiki/Hulu"}, {"rank": 2, "page_id": "Available_expression", "url": "https://en.wikipedia.org/wiki/Available_expression"}, {"rank": 3, "page_id": "Available_light", "url": "https://en.wikipedia.org/wiki/Available_light"}, {"rank": 4, "page_id": "Available_name", "url": "https://en.wikipedia.org/wiki/Available_name"}, {"rank": 5, "page_id": "Empress_Hulü", "url": "https://en.wikipedia.org/wiki/Empress_Hul%C3%BC"}, {"rank": 6, "page_id": "Hulu,_Sweden", "url": "https://en.wikipedia.org/wiki/Hulu%2C_Sweden"}, {"rank": 7, "page_id": "Hulu_Culture", "url": "https://en.wikipedia.org/wiki/Hulu_Culture"}, {"rank": 8, "page_id": "Hulu_Rajang", "url": "https://en.wikipedia.org/wiki/Hulu_Rajang"}, {"rank": 9, "page_id": "Hulü_Guang", "url": "https://en.wikipedia.org/wiki/Hul%C3%BC_Guang"}, {"rank": 10, "page_id": "Pengkalan_Hulu", "url": "https://en.wikipedia.org/wiki/Pengkalan_Hulu"}]}\n{"id": 157345, "claim": "Linda McCartney died on April 16, 1998.", "ranked_articles": [{"rank": 1, "page_id": "Linda_McCartney", "url": "https://en.wikipedia.org/wiki/Linda_McCartney"}, {"rank": 2, "page_id": "Linda_McCartney_Foods", "url": "https://en.wikipedia.org/wiki/Linda_McCartney_Foods"}, {"rank": 3, "page_id": "Linda_McCartney\'s_Sixties", "url": "https://en.wikipedia.org/wiki/Linda_McCartney%27s_Sixties"}, {"rank": 4, "page_id": "Linda_McCartney_Racing_Team", "url": "https://en.wikipedia.org/wiki/Linda_McCartney_Racing_Team"}, {"rank": 5, "page_id": "The_Linda_McCartney_Story", "url": "https://en.wikipedia.org/wiki/The_Linda_McCartney_Story"}, {"rank": 6, "page_id": "Tornado_outbreak_of_April_15–16,_1998", "url": "https://en.wikipedia.org/wiki/Tornado_outbreak_of_April_15%E2%80%9316%2C_1998"}, {"rank": 7, "page_id": "April,_april", "url": "https://en.wikipedia.org/wiki/April%2C_april"}, {"rank": 8, "page_id": "Linda_Linda_Linda", "url": "https://en.wikipedia.org/wiki/Linda_Linda_Linda"}, {"rank": 9, "page_id": "Linda_Linda", "url": "https://en.wikipedia.org/wiki/Linda_Linda"}, {"rank": 10, "page_id": "April_-LRB-1998_film-RRB-", "url": "https://en.wikipedia.org/wiki/April_%281998_film%29"}]}\n{"id": 31746, "claim": "Ginger Baker was passed up for a spot in Cream.", "ranked_articles": [{"rank": 1, "page_id": "Ginger_Baker", "url": "https://en.wikipedia.org/wiki/Ginger_Baker"}, {"rank": 2, "page_id": "Cream-spot_ladybird", "url": "https://en.wikipedia.org/wiki/Cream-spot_ladybird"}, {"rank": 3, "page_id": "Albert_\\"Ginger\\"_Baker", "url": "https://en.wikipedia.org/wiki/Albert_%22Ginger%22_Baker"}, {"rank": 4, "page_id": "Passed_ball", "url": "https://en.wikipedia.org/wiki/Passed_ball"}, {"rank": 5, "page_id": "Passed_midshipman", "url": "https://en.wikipedia.org/wiki/Passed_midshipman"}, {"rank": 6, "page_id": "Passed_pawn", "url": "https://en.wikipedia.org/wiki/Passed_pawn"}, {"rank": 7, "page_id": "Ginger_Baker\'s_Air_Force", "url": "https://en.wikipedia.org/wiki/Ginger_Baker%27s_Air_Force"}, {"rank": 8, "page_id": "Ginger_Baker_at_His_Best", "url": "https://en.wikipedia.org/wiki/Ginger_Baker_at_His_Best"}, {"rank": 9, "page_id": "Ginger", "url": "https://en.wikipedia.org/wiki/Ginger"}, {"rank": 10, "page_id": "SpoT", "url": "https://en.wikipedia.org/wiki/SpoT"}]}\n{"id": 122056, "claim": "Stranger Things does not have two seasons.", "ranked_articles": [{"rank": 1, "page_id": "Stranger_Things", "url": "https://en.wikipedia.org/wiki/Stranger_Things"}, {"rank": 2, "page_id": "Two_Seasons", "url": "https://en.wikipedia.org/wiki/Two_Seasons"}, {"rank": 3, "page_id": "Two_Soft_Things,_Two_Hard_Things", "url": "https://en.wikipedia.org/wiki/Two_Soft_Things%2C_Two_Hard_Things"}, {"rank": 4, "page_id": "Remember_Two_Things", "url": "https://en.wikipedia.org/wiki/Remember_Two_Things"}, {"rank": 5, "page_id": "Stranger_Things_Have_Happened", "url": "https://en.wikipedia.org/wiki/Stranger_Things_Have_Happened"}, {"rank": 6, "page_id": "Two_Thousand_Seasons", "url": "https://en.wikipedia.org/wiki/Two_Thousand_Seasons"}, {"rank": 7, "page_id": "Eleven_-LRB-Stranger_Things-RRB-", "url": "https://en.wikipedia.org/wiki/Eleven_%28Stranger_Things%29"}, {"rank": 8, "page_id": "Penguin_Dreams_and_Stranger_Things", "url": "https://en.wikipedia.org/wiki/Penguin_Dreams_and_Stranger_Things"}, {"rank": 9, "page_id": "Strange_Things_and_Stranger_Places", "url": "https://en.wikipedia.org/wiki/Strange_Things_and_Stranger_Places"}, {"rank": 10, "page_id": "Stranger_Things_-LRB-disambiguation-RRB-", "url": "https://en.wikipedia.org/wiki/Stranger_Things_%28disambiguation%29"}]}\n{"id": 26196, "claim": "Antarctica is bordered by the Pacific Ocean.", "ranked_articles": [{"rank": 1, "page_id": "Ocean_Pacific", "url": "https://en.wikipedia.org/wiki/Ocean_Pacific"}, {"rank": 2, "page_id": "Pacific_Ocean", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean"}, {"rank": 3, "page_id": "Pacific_Ocean_Blue", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Blue"}, {"rank": 4, "page_id": "Pacific_Ocean_Blues", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Blues"}, {"rank": 5, "page_id": "Pacific_Ocean_Division", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Division"}, {"rank": 6, "page_id": "Pacific_Ocean_Fire", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Fire"}, {"rank": 7, "page_id": "Pacific_Ocean_Games", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Games"}, {"rank": 8, "page_id": "Pacific_Ocean_Park", "url": "https://en.wikipedia.org/wiki/Pacific_Ocean_Park"}, {"rank": 9, "page_id": "Pacific_ocean_perch", "url": "https://en.wikipedia.org/wiki/Pacific_ocean_perch"}, {"rank": 10, "page_id": "Bordered_pug", "url": "https://en.wikipedia.org/wiki/Bordered_pug"}]}\n{"id": 103743, "claim": "David Beckham hated Real Madrid.", "ranked_articles": [{"rank": 1, "page_id": "David_Beckham", "url": "https://en.wikipedia.org/wiki/David_Beckham"}, {"rank": 2, "page_id": "David_Beckham_Academy", "url": "https://en.wikipedia.org/wiki/David_Beckham_Academy"}, {"rank": 3, "page_id": "David_Beckham_Soccer", "url": "https://en.wikipedia.org/wiki/David_Beckham_Soccer"}, {"rank": 4, "page_id": "Hated", "url": "https://en.wikipedia.org/wiki/Hated"}, {"rank": 5, "page_id": "Ciudad_Real_Madrid", "url": "https://en.wikipedia.org/wiki/Ciudad_Real_Madrid"}, {"rank": 6, "page_id": "Real_Madrid_Baloncesto", "url": "https://en.wikipedia.org/wiki/Real_Madrid_Baloncesto"}, {"rank": 7, "page_id": "Real_Madrid_Balonmano", "url": "https://en.wikipedia.org/wiki/Real_Madrid_Balonmano"}, {"rank": 8, "page_id": "Real_Madrid_C", "url": "https://en.wikipedia.org/wiki/Real_Madrid_C"}, {"rank": 9, "page_id": "Real_Madrid_Castilla", "url": "https://en.wikipedia.org/wiki/Real_Madrid_Castilla"}, {"rank": 10, "page_id": "Real_Madrid_TV", "url": "https://en.wikipedia.org/wiki/Real_Madrid_TV"}]}\n{"id": 20169, "claim": "Lyme disease is a diagnosed condition.", "ranked_articles": [{"rank": 1, "page_id": "Lyme_disease", "url": "https://en.wikipedia.org/wiki/Lyme_disease"}, {"rank": 2, "page_id": "Chronic_Lyme_disease", "url": "https://en.wikipedia.org/wiki/Chronic_Lyme_disease"}, {"rank": 3, "page_id": "Lyme_disease_microbiology", "url": "https://en.wikipedia.org/wiki/Lyme_disease_microbiology"}, {"rank": 4, "page_id": "Missouri_Lyme_disease", "url": "https://en.wikipedia.org/wiki/Missouri_Lyme_disease"}, {"rank": 5, "page_id": "Lyme", "url": "https://en.wikipedia.org/wiki/Lyme"}, {"rank": 6, "page_id": "List_of_people_diagnosed_with_coeliac_disease", "url": "https://en.wikipedia.org/wiki/List_of_people_diagnosed_with_coeliac_disease"}, {"rank": 7, "page_id": "Condition", "url": "https://en.wikipedia.org/wiki/Condition"}, {"rank": 8, "page_id": "List_of_people_diagnosed_with_Crohn\'s_disease", "url": "https://en.wikipedia.org/wiki/List_of_people_diagnosed_with_Crohn%27s_disease"}, {"rank": 9, "page_id": "List_of_people_diagnosed_with_Parkinson\'s_disease", "url": "https://en.wikipedia.org/wiki/List_of_people_diagnosed_with_Parkinson%27s_disease"}, {"rank": 10, "page_id": "Lyme-Old_Lyme_High_School", "url": "https://en.wikipedia.org/wiki/Lyme-Old_Lyme_High_School"}]}\n{"id": 64487, "claim": "The AK-47 has been manufactured in many countries and is looked down upon by the international community.", "ranked_articles": [{"rank": 1, "page_id": "AK-47", "url": "https://en.wikipedia.org/wiki/AK-47"}, {"rank": 2, "page_id": "AK_47_Mayanja", "url": "https://en.wikipedia.org/wiki/AK_47_Mayanja"}, {"rank": 3, "page_id": "Oficial_AK-47", "url": "https://en.wikipedia.org/wiki/Oficial_AK-47"}, {"rank": 4, "page_id": "AK-47_-LRB-disambiguation-RRB-", "url": "https://en.wikipedia.org/wiki/AK-47_%28disambiguation%29"}, {"rank": 5, "page_id": "Manufactured_Landscapes", "url": "https://en.wikipedia.org/wiki/Manufactured_Landscapes"}, {"rank": 6, "page_id": "Manufactured_Romance", "url": "https://en.wikipedia.org/wiki/Manufactured_Romance"}, {"rank": 7, "page_id": "Manufactured_Superstars", "url": "https://en.wikipedia.org/wiki/Manufactured_Superstars"}, {"rank": 8, "page_id": "Manufactured_controversy", "url": "https://en.wikipedia.org/wiki/Manufactured_controversy"}, {"rank": 9, "page_id": "Manufactured_housing", "url": "https://en.wikipedia.org/wiki/Manufactured_housing"}, {"rank": 10, "page_id": "Manufactured_risk", "url": "https://en.wikipedia.org/wiki/Manufactured_risk"}]}\n{"id": 16136, "claim": "George Best is immortal.", "ranked_articles": [{"rank": 1, "page_id": "Immortal", "url": "https://en.wikipedia.org/wiki/Immortal"}, {"rank": 2, "page_id": "Immortal?", "url": "https://en.wikipedia.org/wiki/Immortal%3F"}, {"rank": 3, "page_id": "George_Best", "url": "https://en.wikipedia.org/wiki/George_Best"}, {"rank": 4, "page_id": "George_A._Best", "url": "https://en.wikipedia.org/wiki/George_A._Best"}, {"rank": 5, "page_id": "Immortal_Beloved", "url": "https://en.wikipedia.org/wiki/Immortal_Beloved"}, {"rank": 6, "page_id": "Immortal_Bird", "url": "https://en.wikipedia.org/wiki/Immortal_Bird"}, {"rank": 7, "page_id": "Immortal_Clay", "url": "https://en.wikipedia.org/wiki/Immortal_Clay"}, {"rank": 8, "page_id": "Immortal_Coil", "url": "https://en.wikipedia.org/wiki/Immortal_Coil"}, {"rank": 9, "page_id": "Immortal_Combat", "url": "https://en.wikipedia.org/wiki/Immortal_Combat"}, {"rank": 10, "page_id": "Immortal_Dominion", "url": "https://en.wikipedia.org/wiki/Immortal_Dominion"}]}\n{"id": 175749, "claim": "The Muse was released in 1919.", "ranked_articles": [{"rank": 1, "page_id": "Released", "url": "https://en.wikipedia.org/wiki/Released"}, {"rank": 2, "page_id": "MusE", "url": "https://en.wikipedia.org/wiki/MusE"}, {"rank": 3, "page_id": "Muse", "url": "https://en.wikipedia.org/wiki/Muse"}, {"rank": 4, "page_id": "Released_time", "url": "https://en.wikipedia.org/wiki/Released_time"}, {"rank": 5, "page_id": "A_Muse", "url": "https://en.wikipedia.org/wiki/A_Muse"}, {"rank": 6, "page_id": "Abaltat_Muse", "url": "https://en.wikipedia.org/wiki/Abaltat_Muse"}, {"rank": 7, "page_id": "Abduwali_Muse", "url": "https://en.wikipedia.org/wiki/Abduwali_Muse"}, {"rank": 8, "page_id": "Able_Muse", "url": "https://en.wikipedia.org/wiki/Able_Muse"}, {"rank": 9, "page_id": "Adobe_Muse", "url": "https://en.wikipedia.org/wiki/Adobe_Muse"}, {"rank": 10, "page_id": "Alexander_Muse", "url": "https://en.wikipedia.org/wiki/Alexander_Muse"}]}\n{"id": 124782, "claim": "Kevin Spacey worked for Yasunari Kawabata.", "ranked_articles": [{"rank": 1, "page_id": "Yasunari_Kawabata", "url": "https://en.wikipedia.org/wiki/Yasunari_Kawabata"}, {"rank": 2, "page_id": "Kevin_Spacey", "url": "https://en.wikipedia.org/wiki/Kevin_Spacey"}, {"rank": 3, "page_id": "Kevin_Spacey_filmography", "url": "https://en.wikipedia.org/wiki/Kevin_Spacey_filmography"}, {"rank": 4, "page_id": "Yasunari", "url": "https://en.wikipedia.org/wiki/Yasunari"}, {"rank": 5, "page_id": "Spacey", "url": "https://en.wikipedia.org/wiki/Spacey"}, {"rank": 6, "page_id": "The_Lake_-LRB-Yasunari_Kawabata_novel-RRB-", "url": "https://en.wikipedia.org/wiki/The_Lake_%28Yasunari_Kawabata_novel%29"}, {"rank": 7, "page_id": "Myōgiryū_Yasunari", "url": "https://en.wikipedia.org/wiki/My%C5%8Dgiry%C5%AB_Yasunari"}, {"rank": 8, "page_id": "Yasunari_Hirai", "url": "https://en.wikipedia.org/wiki/Yasunari_Hirai"}, {"rank": 9, "page_id": "Yasunari_Hiraoka", "url": "https://en.wikipedia.org/wiki/Yasunari_Hiraoka"}, {"rank": 10, "page_id": "Yasunari_Iwata", "url": "https://en.wikipedia.org/wiki/Yasunari_Iwata"}]}\n{"id": 102446, "claim": "Tezaab ran in theaters for about a year.", "ranked_articles": [{"rank": 1, "page_id": "Tezaab", "url": "https://en.wikipedia.org/wiki/Tezaab"}, {"rank": 2, "page_id": "59E59_Theaters", "url": "https://en.wikipedia.org/wiki/59E59_Theaters"}, {"rank": 3, "page_id": "Commonwealth_Theaters", "url": "https://en.wikipedia.org/wiki/Commonwealth_Theaters"}, {"rank": 4, "page_id": "Jujamcyn_Theaters", "url": "https://en.wikipedia.org/wiki/Jujamcyn_Theaters"}, {"rank": 5, "page_id": "Smithsonian_Theaters", "url": "https://en.wikipedia.org/wiki/Smithsonian_Theaters"}, {"rank": 6, "page_id": "Speakeasy_Theaters", "url": "https://en.wikipedia.org/wiki/Speakeasy_Theaters"}, {"rank": 7, "page_id": "What_About_Next_Year", "url": "https://en.wikipedia.org/wiki/What_About_Next_Year"}, {"rank": 8, "page_id": "RAN", "url": "https://en.wikipedia.org/wiki/RAN"}, {"rank": 9, "page_id": "Ran", "url": "https://en.wikipedia.org/wiki/Ran"}, {"rank": 10, "page_id": "Rán", "url": "https://en.wikipedia.org/wiki/R%C3%A1n"}]}\n{"id": 182878, "claim": "Dog Day Afternoon stars Voltaire.", "ranked_articles": [{"rank": 1, "page_id": "Dog_Day_Afternoon", "url": "https://en.wikipedia.org/wiki/Dog_Day_Afternoon"}, {"rank": 2, "page_id": "Bob_Day_Afternoon", "url": "https://en.wikipedia.org/wiki/Bob_Day_Afternoon"}, {"rank": 3, "page_id": "Hog_Day_Afternoon", "url": "https://en.wikipedia.org/wiki/Hog_Day_Afternoon"}, {"rank": 4, "page_id": "The_Dog_Stars", "url": "https://en.wikipedia.org/wiki/The_Dog_Stars"}, {"rank": 5, "page_id": "Voltaire", "url": "https://en.wikipedia.org/wiki/Voltaire"}, {"rank": 6, "page_id": "Dog-day_cicada", "url": "https://en.wikipedia.org/wiki/Dog-day_cicada"}, {"rank": 7, "page_id": "Dog_Day_Sunrise", "url": "https://en.wikipedia.org/wiki/Dog_Day_Sunrise"}, {"rank": 8, "page_id": "Afternoon", "url": "https://en.wikipedia.org/wiki/Afternoon"}, {"rank": 9, "page_id": "Aurelio_Voltaire", "url": "https://en.wikipedia.org/wiki/Aurelio_Voltaire"}, {"rank": 10, "page_id": "Boulevard_Voltaire", "url": "https://en.wikipedia.org/wiki/Boulevard_Voltaire"}]}\n{"id": 191130, "claim": "My Everything plummeted in twenty countries.", "ranked_articles": [{"rank": 1, "page_id": "My_Everything", "url": "https://en.wikipedia.org/wiki/My_Everything"}, {"rank": 2, "page_id": "Twenty_Twenty", "url": "https://en.wikipedia.org/wiki/Twenty_Twenty"}, {"rank": 3, "page_id": "Everything_Is_My_Family", "url": "https://en.wikipedia.org/wiki/Everything_Is_My_Family"}, {"rank": 4, "page_id": "Everything_My_Heart_Desires", "url": "https://en.wikipedia.org/wiki/Everything_My_Heart_Desires"}, {"rank": 5, "page_id": "Everything_in_My_Heart", "url": "https://en.wikipedia.org/wiki/Everything_in_My_Heart"}, {"rank": 6, "page_id": "There_Goes_My_Everything", "url": "https://en.wikipedia.org/wiki/There_Goes_My_Everything"}, {"rank": 7, "page_id": "You\'re_My_Everything", "url": "https://en.wikipedia.org/wiki/You%27re_My_Everything"}, {"rank": 8, "page_id": "You_Are_My_Everything", "url": "https://en.wikipedia.org/wiki/You_Are_My_Everything"}, {"rank": 9, "page_id": "Everything,_Everything", "url": "https://en.wikipedia.org/wiki/Everything%2C_Everything"}, {"rank": 10, "page_id": "Everything_Everything", "url": "https://en.wikipedia.org/wiki/Everything_Everything"}]}\n{"id": 198492, "claim": "Julius Caesar only crossed the Rubicon in 54 BC.", "ranked_articles": [{"rank": 1, "page_id": "Julius_Caesar", "url": "https://en.wikipedia.org/wiki/Julius_Caesar"}, {"rank": 2, "page_id": "Drusus_Julius_Caesar", "url": "https://en.wikipedia.org/wiki/Drusus_Julius_Caesar"}, {"rank": 3, "page_id": "Gaius_Julius_Caesar", "url": "https://en.wikipedia.org/wiki/Gaius_Julius_Caesar"}, {"rank": 4, "page_id": "Julius_Caesar_Alford", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Alford"}, {"rank": 5, "page_id": "Julius_Caesar_Aranzi", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Aranzi"}, {"rank": 6, "page_id": "Julius_Caesar_Chappelle", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Chappelle"}, {"rank": 7, "page_id": "Julius_Caesar_Czarnikow", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Czarnikow"}, {"rank": 8, "page_id": "Julius_Caesar_Herrera", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Herrera"}, {"rank": 9, "page_id": "Julius_Caesar_Ibbetson", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Ibbetson"}, {"rank": 10, "page_id": "Julius_Caesar_Scaliger", "url": "https://en.wikipedia.org/wiki/Julius_Caesar_Scaliger"}]}\n{"id": 123242, "claim": "David Beckham made his Premier League debut playing for Manchester United in 1998.", "ranked_articles": [{"rank": 1, "page_id": "David_Beckham", "url": "https://en.wikipedia.org/wiki/David_Beckham"}, {"rank": 2, "page_id": "David_Beckham_Academy", "url": "https://en.wikipedia.org/wiki/David_Beckham_Academy"}, {"rank": 3, "page_id": "David_Beckham_Soccer", "url": "https://en.wikipedia.org/wiki/David_Beckham_Soccer"}, {"rank": 4, "page_id": "Beckham", "url": "https://en.wikipedia.org/wiki/Beckham"}, {"rank": 5, "page_id": "Premier_League", "url": "https://en.wikipedia.org/wiki/Premier_League"}, {"rank": 6, "page_id": "Homme_by_David_Beckham", "url": "https://en.wikipedia.org/wiki/Homme_by_David_Beckham"}, {"rank": 7, "page_id": "David_Beckham\'s_Soccer_USA", "url": "https://en.wikipedia.org/wiki/David_Beckham%27s_Soccer_USA"}, {"rank": 8, "page_id": "Northern_Premier_League_Premier_Division", "url": "https://en.wikipedia.org/wiki/Northern_Premier_League_Premier_Division"}, {"rank": 9, "page_id": "7s_Premier_League", "url": "https://en.wikipedia.org/wiki/7s_Premier_League"}, {"rank": 10, "page_id": "ACC_Premier_League", "url": "https://en.wikipedia.org/wiki/ACC_Premier_League"}]}\n{"id": 110360, "claim": "In 1994, Barry Gibb made it into the Songwriters Hall of Fame.", "ranked_articles": [{"rank": 1, "page_id": "Barry_Gibb", "url": "https://en.wikipedia.org/wiki/Barry_Gibb"}, {"rank": 2, "page_id": "The_Barry_Gibb_Talk_Show", "url": "https://en.wikipedia.org/wiki/The_Barry_Gibb_Talk_Show"}, {"rank": 3, "page_id": "Gibb", "url": "https://en.wikipedia.org/wiki/Gibb"}, {"rank": 4, "page_id": "Songwriters_Network", "url": "https://en.wikipedia.org/wiki/Songwriters_Network"}, {"rank": 5, "page_id": "Moonlight_-LRB-Barry_Gibb_song-RRB-", "url": "https://en.wikipedia.org/wiki/Moonlight_%28Barry_Gibb_song%29"}, {"rank": 6, "page_id": "Alex_Gibb", "url": "https://en.wikipedia.org/wiki/Alex_Gibb"}, {"rank": 7, "page_id": "Alexander_Gibb", "url": "https://en.wikipedia.org/wiki/Alexander_Gibb"}, {"rank": 8, "page_id": "Alexandrine_Gibb", "url": "https://en.wikipedia.org/wiki/Alexandrine_Gibb"}, {"rank": 9, "page_id": "Ali_Gibb", "url": "https://en.wikipedia.org/wiki/Ali_Gibb"}, {"rank": 10, "page_id": "Allan_Gibb", "url": "https://en.wikipedia.org/wiki/Allan_Gibb"}]}\n{"id": 136470, "claim": "Cory Monteith played the role of a dog.", "ranked_articles": [{"rank": 1, "page_id": "Cory_Monteith", "url": "https://en.wikipedia.org/wiki/Cory_Monteith"}, {"rank": 2, "page_id": "Monteith", "url": "https://en.wikipedia.org/wiki/Monteith"}, {"rank": 3, "page_id": "Played", "url": "https://en.wikipedia.org/wiki/Played"}, {"rank": 4, "page_id": "Cory_Cory-Wright", "url": "https://en.wikipedia.org/wiki/Cory_Cory-Wright"}, {"rank": 5, "page_id": "Cory", "url": "https://en.wikipedia.org/wiki/Cory"}, {"rank": 6, "page_id": "Alec_Monteith", "url": "https://en.wikipedia.org/wiki/Alec_Monteith"}, {"rank": 7, "page_id": "Alexander_Monteith", "url": "https://en.wikipedia.org/wiki/Alexander_Monteith"}, {"rank": 8, "page_id": "Andrew_Monteith", "url": "https://en.wikipedia.org/wiki/Andrew_Monteith"}, {"rank": 9, "page_id": "Brian_Monteith", "url": "https://en.wikipedia.org/wiki/Brian_Monteith"}, {"rank": 10, "page_id": "Camp_Monteith", "url": "https://en.wikipedia.org/wiki/Camp_Monteith"}]}\n', 'eval_labels.json': '{"147861": "REFUTES", "36376": "SUPPORTS", "6093": "SUPPORTS", "23390": "NOT ENOUGH INFO", "44625": "REFUTES", "156586": "NOT ENOUGH INFO", "72212": "NOT ENOUGH INFO", "197148": "NOT ENOUGH INFO", "174565": "REFUTES", "5571": "REFUTES", "154833": "REFUTES", "47767": "NOT ENOUGH INFO", "206937": "SUPPORTS", "210134": "REFUTES", "106579": "REFUTES", "206182": "REFUTES", "128569": "NOT ENOUGH INFO", "40186": "SUPPORTS", "179603": "SUPPORTS", "111557": "NOT ENOUGH INFO", "128891": "SUPPORTS", "4904": "SUPPORTS", "211660": "SUPPORTS", "17174": "REFUTES", "198135": "REFUTES", "79089": "NOT ENOUGH INFO", "45987": "NOT ENOUGH INFO", "81883": "SUPPORTS", "45676": "NOT ENOUGH INFO", "210268": "SUPPORTS", "170862": "REFUTES", "67859": "NOT ENOUGH INFO", "73613": "SUPPORTS", "56727": "NOT ENOUGH INFO", "73853": "NOT ENOUGH INFO", "167230": "NOT ENOUGH INFO", "75919": "REFUTES", "97558": "NOT ENOUGH INFO", "116431": "NOT ENOUGH INFO", "114703": "SUPPORTS", "199544": "REFUTES", "226194": "SUPPORTS", "113256": "NOT ENOUGH INFO", "173001": "REFUTES", "128610": "REFUTES", "6919": "SUPPORTS", "117470": "NOT ENOUGH INFO", "56064": "REFUTES", "10994": "SUPPORTS", "3833": "SUPPORTS", "192941": "SUPPORTS", "22452": "SUPPORTS", "139219": "SUPPORTS", "157329": "REFUTES", "199900": "REFUTES", "44854": "REFUTES", "138808": "REFUTES", "115715": "REFUTES", "46447": "NOT ENOUGH INFO", "170747": "SUPPORTS", "26908": "SUPPORTS", "188504": "NOT ENOUGH INFO", "93642": "SUPPORTS", "108126": "REFUTES", "145550": "SUPPORTS", "201271": "SUPPORTS", "207063": "NOT ENOUGH INFO", "125911": "SUPPORTS", "123370": "NOT ENOUGH INFO", "169335": "NOT ENOUGH INFO", "85145": "NOT ENOUGH INFO", "154547": "SUPPORTS", "84545": "SUPPORTS", "157345": "REFUTES", "31746": "REFUTES", "122056": "REFUTES", "26196": "REFUTES", "103743": "NOT ENOUGH INFO", "20169": "SUPPORTS", "64487": "NOT ENOUGH INFO", "16136": "REFUTES", "175749": "REFUTES", "124782": "NOT ENOUGH INFO", "102446": "SUPPORTS", "182878": "NOT ENOUGH INFO", "191130": "NOT ENOUGH INFO", "198492": "REFUTES", "123242": "REFUTES", "110360": "SUPPORTS", "136470": "NOT ENOUGH INFO"}', 'references.json': '{"147861": {"label": "REFUTES", "annotated_evidence": [[[171451, 184937, "The_Simpsons", 15]]]}, "36376": {"label": "SUPPORTS", "annotated_evidence": [[[52575, 62540, "Robert_Redford", 4], [52575, 62540, "Perry_Mason_-LRB-TV_series-RRB-", 0]]]}, "6093": {"label": "SUPPORTS", "annotated_evidence": [[[22822, 28123, "Tom_Brady", 4]]]}, "23390": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[39337, null, null, null]]]}, "44625": {"label": "REFUTES", "annotated_evidence": [[[60991, 71460, "My_Summer_of_Love", 0], [60991, 71460, "Paweł_Pawlikowski", 0]]]}, "156586": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[180663, null, null, null]]]}, "72212": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[88902, null, null, null]]]}, "197148": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[231684, null, null, null]]]}, "174565": {"label": "REFUTES", "annotated_evidence": [[[201816, 211186, "Bill_Black", 0]]]}, "5571": {"label": "REFUTES", "annotated_evidence": [[[22202, 27320, "LGBT", 0]]]}, "154833": {"label": "REFUTES", "annotated_evidence": [[[178890, 191444, "Ranbir_Kapoor", 15]]]}, "47767": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[64146, null, null, null]]]}, "206937": {"label": "SUPPORTS", "annotated_evidence": [[[244684, 245618, "Michael_Keaton", 0]]]}, "210134": {"label": "REFUTES", "annotated_evidence": [[[248912, 248952, "Lonesome_Dove_-LRB-miniseries-RRB-", 10]]]}, "106579": {"label": "REFUTES", "annotated_evidence": [[[125102, 139443, "Clint_Eastwood", 5]]]}, "206182": {"label": "REFUTES", "annotated_evidence": [[[243731, 244906, "Shallow_Hal", 0]]]}, "128569": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[150590, null, null, null]]]}, "40186": {"label": "SUPPORTS", "annotated_evidence": [[[56521, 66717, "Berlin_Wall", 0], [56521, 66717, "Berlin", 0]]]}, "179603": {"label": "SUPPORTS", "annotated_evidence": [[[208444, 216476, "Michael_Giacchino", 1]]]}, "111557": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[285022, null, null, null], [329676, null, null, null], [329679, null, null, null]]]}, "128891": {"label": "SUPPORTS", "annotated_evidence": [[[150956, 165801, "The_Hunger_Games_-LRB-film-RRB-", 2]]]}, "4904": {"label": "SUPPORTS", "annotated_evidence": [[[21259, 26109, "Paul_Simon", 0]]]}, "211660": {"label": "SUPPORTS", "annotated_evidence": [[[250828, 250796, "Jane_Eyre_-LRB-2011_film-RRB-", 0]], [[250828, 250797, "Jane_Eyre_-LRB-2011_film-RRB-", 2]]]}, "17174": {"label": "REFUTES", "annotated_evidence": [[[32619, 39643, "28_Days_-LRB-film-RRB-", 2]]]}, "198135": {"label": "REFUTES", "annotated_evidence": [[[232982, 236011, "Julie_Walters", 0]], [[232982, 236012, "Julie_Walters", 2]], [[232982, 236013, "Julie_Walters", 5]], [[232982, 236014, "Julie_Walters", 6]], [[232982, 236015, "Julie_Walters", 8]], [[232982, 236016, "Julie_Walters", 12], [232982, 236016, "Pat_and_Margaret", 2]], [[232982, 236017, "Julie_Walters", 14], [232982, 236017, "A_Short_Stay_in_Switzerland", 1]]]}, "79089": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[96064, null, null, null]]]}, "45987": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[62325, null, null, null]]]}, "81883": {"label": "SUPPORTS", "annotated_evidence": [[[98944, 112054, "The_Crazies_-LRB-2010_film-RRB-", 0]], [[98954, 112062, "The_Crazies_-LRB-2010_film-RRB-", 0]]]}, "45676": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[62010, null, null, null]]]}, "210268": {"label": "SUPPORTS", "annotated_evidence": [[[249137, 249186, "William_McKinley", 0]], [[249137, 249187, "William_McKinley", 4]], [[249137, 249188, "William_McKinley", 21]], [[249137, 249189, "William_McKinley", 22]]]}, "170862": {"label": "REFUTES", "annotated_evidence": [[[197135, 207192, "A_Thousand_Suns", 0]]]}, "67859": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[84454, null, null, null]]]}, "73613": {"label": "SUPPORTS", "annotated_evidence": [[[25957, 31905, "Emilio_Estevez", 0]], [[25957, 31906, "Emilio_Estevez", 2]], [[300593, 291748, "Emilio_Estevez", 0]]]}, "56727": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[73089, null, null, null]]]}, "73853": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[90637, null, null, null]]]}, "167230": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[192403, null, null, null]]]}, "75919": {"label": "REFUTES", "annotated_evidence": [[[94274, 107274, "Baahubali-COLON-_The_Beginning", 0]]]}, "97558": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[115172, null, null, null]]]}, "116431": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[136801, null, null, null]]]}, "114703": {"label": "SUPPORTS", "annotated_evidence": [[[134680, 149605, "Colin_Firth", 9]], [[134680, 149606, "Colin_Firth", 8]], [[134680, 149607, "Colin_Firth", 7]]]}, "199544": {"label": "REFUTES", "annotated_evidence": [[[234656, 237484, "United_States_Congress", 3]]]}, "226194": {"label": "SUPPORTS", "annotated_evidence": [[[269674, 265935, "Gift_of_the_Night_Fury", 6], [269674, 265935, "Christopher_Mintz-Plasse", 0]]]}, "113256": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[133564, null, null, null]]]}, "173001": {"label": "REFUTES", "annotated_evidence": [[[199766, 209444, "Rear_Window_-LRB-1998_film-RRB-", 3]]]}, "128610": {"label": "REFUTES", "annotated_evidence": [[[150633, 165503, "Back_to_the_Future_Part_II", 8]], [[150633, 165504, "Back_to_the_Future_Part_II", 15]]]}, "6919": {"label": "SUPPORTS", "annotated_evidence": [[[23834, 29484, "Kajal_Aggarwal", 1], [23834, 29484, "Kajal_Aggarwal", 0]], [[23834, 29485, "Kajal_Aggarwal", 13]]]}, "117470": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[137835, null, null, null]]]}, "56064": {"label": "REFUTES", "annotated_evidence": [[[72431, 83492, "One_America_News_Network", 0]]]}, "10994": {"label": "SUPPORTS", "annotated_evidence": [[[26192, 32202, "Paul_Walker", 4]]]}, "3833": {"label": "SUPPORTS", "annotated_evidence": [[[20043, 24601, "Beauty_Shop", 1]]]}, "192941": {"label": "SUPPORTS", "annotated_evidence": [[[225892, 230475, "William_Hanna", 0]], [[225892, 230476, "William_Hanna", 3]], [[225892, 230477, "William_Hanna", 6]]]}, "22452": {"label": "SUPPORTS", "annotated_evidence": [[[59285, 69637, "RMS_Titanic", 28]], [[64177, 74907, "RMS_Titanic", 28]], [[305322, 296153, "RMS_Titanic", 28]], [[306660, 297293, "RMS_Titanic", 28]], [[306661, 297294, "RMS_Titanic", 28]]]}, "139219": {"label": "SUPPORTS", "annotated_evidence": [[[162038, 176410, "Beverly_Hills,_90210", 10]]]}, "157329": {"label": "REFUTES", "annotated_evidence": [[[181501, 193746, "Social_justice", 11]]]}, "199900": {"label": "REFUTES", "annotated_evidence": [[[235118, 237865, "Metro_Boomin", 0]]]}, "44854": {"label": "REFUTES", "annotated_evidence": [[[61214, 71706, "Paul_Newman", 1]]]}, "138808": {"label": "REFUTES", "annotated_evidence": [[[161606, 176044, "Gwen_Stefani", 17]]]}, "115715": {"label": "REFUTES", "annotated_evidence": [[[282474, 276669, "Thandie_Newton", 0]], [[284444, 278267, "Thandie_Newton", 0]], [[284444, 278268, "Thandie_Newton", 1]], [[284444, 278269, "Thandie_Newton", 4]], [[284444, 278270, "Thandie_Newton", 5]], [[284447, 278272, "Thandie_Newton", 0]], [[330646, 317251, "Thandie_Newton", 0]], [[330646, 317252, "Thandie_Newton", 1]], [[330646, 317253, "Thandie_Newton", 4]], [[330646, 317254, "Thandie_Newton", 5]], [[330651, 317257, "Thandie_Newton", 0]], [[330651, 317258, "Thandie_Newton", 1]], [[330651, 317259, "Thandie_Newton", 4]], [[330651, 317260, "Thandie_Newton", 5]]]}, "46447": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[62776, null, null, null]]]}, "170747": {"label": "SUPPORTS", "annotated_evidence": [[[197002, 207067, "Anthony_Meindl", 20]]]}, "26908": {"label": "SUPPORTS", "annotated_evidence": [[[43049, 51692, "Billy_Joel", 1]], [[43049, 51693, "Billy_Joel", 7], [43049, 51693, "Grammy_Award", 0]], [[43049, 51694, "Billy_Joel", 8]]]}, "188504": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[220031, null, null, null]]]}, "93642": {"label": "SUPPORTS", "annotated_evidence": [[[275558, 270677, "Marta_Kauffman", 0]], [[277373, 272288, "Marta_Kauffman", 0]], [[277867, 272800, "Marta_Kauffman", 0]], [[324609, 312527, "Marta_Kauffman", 0]], [[325980, 313575, "Marta_Kauffman", 0]]]}, "108126": {"label": "REFUTES", "annotated_evidence": [[[126853, 141306, "2001-COLON-_A_Space_Odyssey_-LRB-film-RRB-", 9]]]}, "145550": {"label": "SUPPORTS", "annotated_evidence": [[[168864, 182576, "Pineapple_Express_-LRB-film-RRB-", 0]]]}, "201271": {"label": "SUPPORTS", "annotated_evidence": [[[236844, 239210, "Ronin_-LRB-film-RRB-", 0]]]}, "207063": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[244850, null, null, null]]]}, "125911": {"label": "SUPPORTS", "annotated_evidence": [[[147772, 162796, "Ozarks", 0]], [[147772, 162797, "Ozarks", 1], [147772, 162797, "Missouri", 0]]]}, "123370": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[144877, null, null, null]]]}, "169335": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[195207, null, null, null]]]}, "85145": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[102246, null, null, null]]]}, "154547": {"label": "SUPPORTS", "annotated_evidence": [[[178531, 191123, "Black_Panther_-LRB-comics-RRB-", 0]]]}, "84545": {"label": "SUPPORTS", "annotated_evidence": [[[101597, 114831, "Community_-LRB-TV_series-RRB-", 15]]]}, "157345": {"label": "REFUTES", "annotated_evidence": [[[181521, 193762, "Linda_McCartney", 0]]]}, "31746": {"label": "REFUTES", "annotated_evidence": [[[91932, 104724, "Ginger_Baker", 0]], [[91932, 104725, "Ginger_Baker", 2]], [[91932, 104726, "Ginger_Baker", 8]], [[91932, 104727, "Ginger_Baker", 9], [91932, 104727, "Cream_-LRB-band-RRB-", 0]], [[91932, 104728, "Ginger_Baker", 15], [91932, 104728, "Toad_-LRB-instrumental-RRB-", 0], [91932, 104728, "Toad_-LRB-instrumental-RRB-", 1]], [[94619, 107593, "Ginger_Baker", 0]], [[94619, 107594, "Ginger_Baker", 2]], [[94619, 107595, "Ginger_Baker", 8]], [[94619, 107596, "Ginger_Baker", 15]], [[96683, 109840, "Ginger_Baker", 0]], [[96683, 109841, "Ginger_Baker", 2]], [[307589, 298062, "Ginger_Baker", 0]], [[307589, 298063, "Ginger_Baker", 2]]]}, "122056": {"label": "REFUTES", "annotated_evidence": [[[143259, 158361, "Stranger_Things", 16]]]}, "26196": {"label": "REFUTES", "annotated_evidence": [[[42651, 51299, "Antarctica", 1]]]}, "103743": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[121799, null, null, null]]]}, "20169": {"label": "SUPPORTS", "annotated_evidence": [[[35991, 43537, "Lyme_disease", 34]]]}, "64487": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[202514, null, null, null], [315759, null, null, null]]]}, "16136": {"label": "REFUTES", "annotated_evidence": [[[31532, 38260, "George_Best", 0]], [[31532, 38261, "George_Best", 23]], [[31532, 38262, "George_Best", 18]]]}, "175749": {"label": "REFUTES", "annotated_evidence": [[[203345, 212384, "The_Muse_-LRB-1999_film-RRB-", 0]]]}, "124782": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[34188, null, null, null]]]}, "102446": {"label": "SUPPORTS", "annotated_evidence": [[[122137, 136435, "Tezaab", 9]]]}, "182878": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[212846, null, null, null]]]}, "191130": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[223447, null, null, null]]]}, "198492": {"label": "REFUTES", "annotated_evidence": [[[233363, 236309, "Julius_Caesar", 12]]]}, "123242": {"label": "REFUTES", "annotated_evidence": [[[144741, 159760, "David_Beckham", 11], [144741, 159760, "Manchester_United_F.C.", 0]]]}, "110360": {"label": "SUPPORTS", "annotated_evidence": [[[129480, 144225, "Barry_Gibb", 11]]]}, "136470": {"label": "NOT ENOUGH INFO", "annotated_evidence": [[[159129, null, null, null]]]}}'}
RUN_DEVELOPMENT_DIAGNOSTIC = True # @param {type:"boolean"}
if RUN_DEVELOPMENT_DIAGNOSTIC:
    DIAGNOSTIC_ROOT = PROJECT / "development90"
    (DIAGNOSTIC_ROOT / "data").mkdir(parents=True, exist_ok=True)
    for name, content in DIAGNOSTIC_FILES.items():
        path = DIAGNOSTIC_ROOT / "data" / name
        if path.exists() and path.read_text() != content:
            raise ValueError("Existing diagnostic data differ; do not mix samples.")
        path.write_text(content)
    fever.write_once(DIAGNOSTIC_ROOT / "diagnostic.json", DIAGNOSTIC_MANIFEST)
    for name in ["src", "checkpoint"]:
        target = DIAGNOSTIC_ROOT / name
        if not target.exists(): target.symlink_to(PROJECT / name, target_is_directory=True)
    import shutil
    shutil.copy2(PROJECT / "parts_prompt.txt", DIAGNOSTIC_ROOT / "parts_prompt.txt")
    DIAGNOSTIC_ROWS = fever.verify_diagnostic(DIAGNOSTIC_ROOT)
    DIAGNOSTIC_OUT = RUN_ROOT / "development90"
    import time
    diagnostic_started = time.monotonic()
    fever.run_condition(DIAGNOSTIC_ROWS, DIAGNOSTIC_ROOT, DIAGNOSTIC_OUT, CONDITION,
        MODEL_FAMILY, MODEL_METADATA, EMBEDDING, minutes=SESSION_MINUTES,
        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH, scope="development90")
    completed = sum(p.name != "protocol.json" for p in (DIAGNOSTIC_OUT / "predictions" / CONDITION).glob("*.json"))
    remaining = SESSION_MINUTES - (time.monotonic() - diagnostic_started)/60
    if MODEL_FAMILY != "roberta" and completed == 90 and remaining >= 1:
        print("Running the existing RoBERTa baseline on the SAME 90 claims and saved RAG passages.")
        fever.run_condition(DIAGNOSTIC_ROWS, DIAGNOSTIC_ROOT, DIAGNOSTIC_OUT,
            "roberta_existing_rag_" + str(ROBERTA_MAX_LENGTH), "roberta",
            {"path": str(PROJECT / "checkpoint/roberta_evidence")}, EMBEDDING,
            minutes=remaining, max_length=ROBERTA_MAX_LENGTH, scope="development90")
else:
    print("Development diagnostic disabled.")

### Inspect the diagnostic results
Run this after the diagnostic cell finishes or pauses. Partial results remain clearly marked; missing predictions are not counted as completed claims. The confusion-matrix rows are reference labels; columns are model predictions. INVALID is an execution/output failure, not a fourth FEVER class.

In [ ]:
if "DIAGNOSTIC_OUT" in globals():
    import pandas as pd
    diagnostic_conditions = sorted(p.name for p in (DIAGNOSTIC_OUT / "predictions").glob("*") if p.is_dir())
    diagnostic_metrics = fever.diagnostic_report(DIAGNOSTIC_ROOT, DIAGNOSTIC_OUT, diagnostic_conditions)
    for name, metrics in diagnostic_metrics.items():
        display(pd.DataFrame([{"System": name, "Status": metrics["status"],
            "Completed / 90": metrics["n"], "Accuracy (%)": 100*metrics["accuracy"],
            "Macro-F1 (%)": 100*metrics["macro_f1"], "Invalid": metrics["invalid"]}]).round(2))
        display(pd.DataFrame([{"Class": label, **{key+" (%)": 100*value for key,value in values.items()}}
            for label,values in metrics["per_class"].items()]).round(2))
        display(pd.DataFrame(metrics["confusion_matrix"], index=fever.LABELS,
            columns=fever.LABELS+["INVALID"]))
    fever.monitor_status(DIAGNOSTIC_OUT, CONDITION, last_events=5)
    fever.inspect_saved(DIAGNOSTIC_OUT, CONDITION, show_reference=True, root=DIAGNOSTIC_ROOT, show_full=True)
else:
    print("Run section 6b first.")

## 7. Run / resume the full 19,998-claim evaluation
This is the long operation. It is **off by default**. It attempts the whole official set in its original order and pauses after the session budget. Rerun this cell to resume; do not delete the Drive folder.

Run one model family at a time, then change `MODEL_FAMILY` in section 4 and rerun sections 4–5 and this section. Each family reuses the same saved per-claim RAG passages. Newly fetched full articles and vectors are deleted after each claim; supplied passages, revisions and exact model calls remain.

Failed evidence preparation is an **INVALID execution**, not NEI. Completed invalid predictions remain errors; there is no silent retry. No strict FEVER evidence score is claimed for live article passages. Free Colab may stop early, and GPU availability is not guaranteed.
**Live output:** one progress bar, the current claim and stage, article successes/failures, token counts, prediction, invalid count, GPU memory in the status file, and a short raw-response preview. During a long stage, a heartbeat prints its elapsed time every 20 seconds. A rough ETA starts after three saved claims; it is not a guarantee.

Logs and status are saved in `RUN_ROOT/monitor/CONDITION/`. Repeated infrastructure failures or ten consecutive invalid outputs pause the run for inspection. Saved failures remain errors. These messages do not bypass Colab timeouts.


In [ ]:
RUN_FULL_EVALUATION = False # @param {type:"boolean"}
MAX_NEW_CLAIMS_THIS_SESSION = 0 # @param {type:"integer"}
# 0 means continue until the time budget; a positive number pauses after that many NEW claims.
if not RUN_FULL_EVALUATION:
    print("Full run is off. Target remains all 19,998 claims.")
elif not ALLOW_MODEL_DOWNLOADS:
    print("Model downloads are disabled; inspect section 5 before enabling them.")
else:
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, CONDITION, MODEL_FAMILY,
                        MODEL_METADATA, EMBEDDING, minutes=SESSION_MINUTES,
                        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH,
                        max_new_claims=MAX_NEW_CLAIMS_THIS_SESSION or None)

### Inspect a saved claim or recent events
While the evaluation cell runs, watch its live output. After pausing, stopping or a failure, run this cell to inspect Drive logs and the latest saved prediction. Choose a claim ID for its exact prompt, raw response and supplied passages. Reference labels are read only after a prediction has been saved; they never enter the model prompt.


In [ ]:
INSPECT_CLAIM_ID = "" # @param {type:"string"}
SHOW_FULL_TRACE = False # @param {type:"boolean"}
SHOW_REFERENCE_LABEL = False # @param {type:"boolean"}
fever.monitor_status(RUN_ROOT, CONDITION, last_events=8)
try:
    fever.inspect_saved(RUN_ROOT, CONDITION, claim_id=int(INSPECT_CLAIM_ID) if INSPECT_CLAIM_ID.strip() else None,
        show_reference=SHOW_REFERENCE_LABEL, root=PROJECT, show_full=SHOW_FULL_TRACE)
except FileNotFoundError:
    print("That claim has no saved prediction yet. Check the active stage in the status above.")


## 8. Check progress and score complete conditions
Partial runs show progress only. A final accuracy table is produced only when a condition covers **all 19,998 IDs**. INVALID remains in the denominator. Metrics are label accuracy, macro-F1, per-class precision/recall/F1 and confusion matrices.

A complete official-development result is still a development result. The benchmark has already been inspected. The historical audit found three normalized-text overlaps with the original checkpoint’s training pool; all 19,998 benchmark rows remain in evaluation. This is not an untouched final test.

In [ ]:
conditions = sorted(p.name for p in (RUN_ROOT / "predictions").glob("*") if p.is_dir())
for name in conditions:
    folder = RUN_ROOT / "predictions" / name
    n = sum(p.name != "protocol.json" for p in folder.glob("*.json"))
    print(name, n, "/", len(EVAL_ROWS))
METRICS = fever.score(PROJECT, RUN_ROOT, conditions, require_full=True)
if METRICS:
    import pandas as pd
    display(pd.DataFrame([{"System": name, "Claims": m["n"], "Accuracy (%)": 100*m["accuracy"],
                           "Macro-F1": m["macro_f1"], "Invalid": m["invalid"]} for name, m in METRICS.items()]))
    import matplotlib.pyplot as plt
    for name, m in METRICS.items():
        display(pd.DataFrame([{"Class": label, "Precision (%)": 100*v["precision"],
                              "Recall (%)": 100*v["recall"], "F1 (%)": 100*v["f1"]}
                             for label, v in m["per_class"].items()]))
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.imshow(m["confusion_matrix"], cmap="Blues")
        ax.set_xticks(range(4), ["SUPPORTS", "REFUTES", "NEI", "INVALID"])
        ax.set_yticks(range(3), ["SUPPORTS", "REFUTES", "NEI"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name + " — all 19,998 claims")
        for i, row in enumerate(m["confusion_matrix"]):
            for j, value in enumerate(row): ax.text(j, i, str(value), ha="center", va="center")
        plt.show()

## 8b. Connect the full training supplement
This cell looks for `FEVER_Training_Data.zip` in your Drive `FEVER_Project` folder. Preparation is separate from the original bundle, so that bundle stays unchanged. If the supplement is not ready yet, this prints a message and does not start training.

We preserve the original split: 116,356 training claims and 14,534 validation claims after excluding normalized training/official-development text overlaps from validation. The 90 diagnostic claims are validation data. The original internal test remains unused. We do not reshuffle data that the initial checkpoint has already seen into a new validation split.

In [ ]:
training_archive = DRIVE_PROJECT / "FEVER_Training_Data.zip"
FULL_TRAINING_READY = False
if not training_archive.is_file():
    print("Full training supplement is not uploaded yet. No training started.")
else:
    with zipfile.ZipFile(training_archive) as archive:
        ready = json.loads(archive.read("training_ready.json"))
        expected_names = {"data/train_full.jsonl", "data/validation_full.jsonl", "training_ready.json", "preparation_protocol.json"}
        if set(archive.namelist()) != expected_names:
            raise ValueError("Unexpected training supplement contents")
        for name in ["train_full.jsonl", "validation_full.jsonl"]:
            content = archive.read("data/" + name)
            if hashlib.sha256(content).hexdigest() != ready["files"][name]:
                raise ValueError("Training supplement checksum mismatch")
            target = PROJECT / "data" / name
            if target.exists() and target.read_bytes() != content:
                raise ValueError("Existing training data differ; do not overwrite")
            target.write_bytes(content)
    fever.write_once(PROJECT / "training_ready.json", ready)
    FULL_TRAINING_READY = True
    print("Full data ready:", ready["train_n"], "training;", ready["validation_n"], "validation.")

## 9. Optional RoBERTa layer-unfreezing check
This is a **training operation**, off by default. Run it only when you choose to start training.

- `pilot`: included **96 training + 96 validation examples**, used only to verify that training and checkpoints work.
- `full`: requires `train_full.jsonl` and `validation_full.jsonl` from the optional full-training export. The starter bundle intentionally does not prepare this hours-long local retrieval job.
- All conditions start from the **same original task-trained RoBERTa checkpoint**. This is continued fine-tuning, not starting from raw pretrained RoBERTa.
- Freeze all weights, then unfreeze the classification head and the last **1, 2 or 3 Transformer blocks**. Earlier blocks remain frozen.
- Fixed seed 42, 2 epochs, learning rate 2e-5, batch size 4, gradient accumulation 4, maximum 512 tokens. Validation selects the checkpoint; official development labels never train or select it.
- Training uses historical retrieved sentence text for every class, not the new live-article RAG corpus. Report that distribution difference. The pilot includes nonempty NEI evidence to avoid the old empty-input shortcut.

Change only the number of unfrozen layers between controlled conditions. Llama/Qwen here are inference baselines: **this notebook does not implement their fine-tuning**. Quantized LLM adapter training would be a separate experiment and is not equivalent to unfreezing their original last layers.

In [ ]:
RUN_ROBERTA_TRAINING = False # @param {type:"boolean"}
UNFREEZE_LAST_N = 1 # @param [1, 2, 3] {type:"raw"}
TRAIN_SCOPE = "full" # @param ["pilot", "full"]
if not RUN_ROBERTA_TRAINING:
    print("Training is off. Pilot training is a software check, not a completed research experiment.")
else:
    final_checkpoint = fever.finetune_roberta(PROJECT, RUN_ROOT,
                    last_layers=UNFREEZE_LAST_N, scope=TRAIN_SCOPE, minutes=SESSION_MINUTES)
    if final_checkpoint:
        print("To evaluate this model, set ROBERTA_CHECKPOINT_OVERRIDE in section 4 to:")
        print(str(final_checkpoint))
        print("Use a distinct condition name for each trained checkpoint; see the next cell.")

### Run the three controlled full-training conditions
This is off until the diagnostic has been reviewed and the full supplement is ready. All three conditions use the same initial task-trained checkpoint, training/validation files, seed and hyperparameters. Only the last 1, 2 or 3 unfrozen blocks differ. A session stops when its budget is reached; rerunning resumes incomplete training and skips completed conditions. This trains RoBERTa only; it does not train Qwen/Llama.

These training files use historical retrieved sentences, while the diagnostic/evaluation uses live-article RAG. This distribution difference remains a limitation and must be reported.

In [ ]:
RUN_CONTROLLED_FULL_TRAINING = False # @param {type:"boolean"}
if RUN_CONTROLLED_FULL_TRAINING:
    if not FULL_TRAINING_READY:
        raise RuntimeError("Connect the full training supplement in section 8b first.")
    import time
    started = time.monotonic()
    for layers in [1, 2, 3]:
        remaining = SESSION_MINUTES - (time.monotonic() - started)/60
        if remaining < 1:
            print("Session budget reached. Rerun to resume remaining conditions.")
            break
        checkpoint = fever.finetune_roberta(PROJECT, RUN_ROOT, last_layers=layers,
                                           scope="full", minutes=remaining)
        if checkpoint is None:
            print("Condition paused:", layers, "blocks. Rerun this cell to resume.")
            break
else:
    print("Full training is off pending diagnostic review. No model has been trained by this cell.")

### Compare completed training conditions on the same 90 development claims
This runs only when controlled full training is enabled. Completed checkpoints are compared with the existing baseline on the exact diagnostic claims and saved RAG passages. Partial training conditions are skipped. These development results help assess changes; they do not replace the final official-development evaluation.

In [ ]:
if RUN_CONTROLLED_FULL_TRAINING and "DIAGNOSTIC_ROOT" in globals():
    import time
    comparison_started = time.monotonic()
    for layers in [1, 2, 3]:
        trained = RUN_ROOT / "training" / ("roberta_full_last" + str(layers))
        if not (trained / "complete.json").is_file():
            print("Not yet complete:", layers, "blocks")
            continue
        remaining = SESSION_MINUTES - (time.monotonic()-comparison_started)/60
        if remaining < 1:
            print("Comparison session budget reached. Rerun to resume.")
            break
        name = "roberta_full_last" + str(layers) + "_rag_" + str(ROBERTA_MAX_LENGTH)
        fever.run_condition(DIAGNOSTIC_ROWS, DIAGNOSTIC_ROOT, DIAGNOSTIC_OUT, name,
            "roberta", {"path": str(trained / "final")}, EMBEDDING, minutes=remaining,
            max_length=ROBERTA_MAX_LENGTH, scope="development90")
    conditions = sorted(p.name for p in (DIAGNOSTIC_OUT / "predictions").glob("*") if p.is_dir())
    report = fever.diagnostic_report(DIAGNOSTIC_ROOT, DIAGNOSTIC_OUT, conditions)
    import pandas as pd
    display(pd.DataFrame([{"Condition": name, "Status": m["status"], "Claims": m["n"],
        "Accuracy (%)": 100*m["accuracy"], "Macro-F1 (%)": 100*m["macro_f1"], "Invalid": m["invalid"]}
        for name,m in report.items()]).round(2))
else:
    print("Before/after comparison waits for enabled full training and the diagnostic inputs.")

### Evaluate a completed fine-tuned checkpoint
Use the full 19,998 inputs again. The condition name includes its layer count and training scope. A pilot-trained condition is explicitly named pilot and must not be called full-training performance.

In [ ]:
RUN_FINETUNED_EVALUATION = False # @param {type:"boolean"}
if not RUN_FINETUNED_EVALUATION:
    print("Fine-tuned evaluation is off.")
else:
    trained = RUN_ROOT / "training" / ("roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N))
    if not (trained / "complete.json").is_file():
        raise RuntimeError("Training has not completed. Resume training before evaluating its final checkpoint.")
    name = "roberta_" + TRAIN_SCOPE + "_last" + str(UNFREEZE_LAST_N) + "_rag_" + str(ROBERTA_MAX_LENGTH)
    fever.run_condition(EVAL_ROWS, PROJECT, RUN_ROOT, name, "roberta",
        {"path": str(trained / "final")}, EMBEDDING, minutes=SESSION_MINUTES,
        max_new_tokens=MAX_NEW_TOKENS, max_length=ROBERTA_MAX_LENGTH)

## 10. Download a debug ZIP to send for review
Run this after pausing the evaluation/training cell. The ZIP contains the latest status and events, recent exact prompts/responses with their evidence, condition protocols, available metrics, and training-loss history. It excludes model weights, optimizer files, tokens and caches.

Leave `EXPORT_ALL_PREDICTIONS_CSV` off for a quick debug package. Turn it on after a full run if you also want compact CSVs of every saved prediction; scanning many Drive files can take time. The export does not run inference or change saved results. The ZIP is saved in **FEVER_Project / exports** and offered as a browser download. Upload that downloaded ZIP in our chat.

In [ ]:
EXPORT_DEBUG_ZIP = False # @param {type:"boolean"}
EXPORT_ALL_PREDICTIONS_CSV = False # @param {type:"boolean"}
if not EXPORT_DEBUG_ZIP:
    print("Enable EXPORT_DEBUG_ZIP and run this cell after a session pauses or fails.")
else:
    import csv, io, zipfile, platform
    from collections import deque
    from datetime import datetime, timezone
    from google.colab import files
    export_dir = DRIVE_PROJECT / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    export_path = export_dir / ("FEVER_debug_" + stamp + ".zip")
    added = set()
    recent_predictions = []
    def add_file(z, path, archive_name):
        path = Path(path)
        if path.is_file() and archive_name not in added:
            z.write(path, archive_name)
            added.add(archive_name)
    with zipfile.ZipFile(export_path, "x", zipfile.ZIP_DEFLATED) as z:
        z.writestr("export_info.json", json.dumps({"created_utc": stamp, "experiment": EXPERIMENT_NAME,
            "condition_selected": CONDITION, "python": platform.python_version(),
            "scope": "debug subset; recent full traces, not every model response",
            "all_predictions_csv_requested": EXPORT_ALL_PREDICTIONS_CSV}, indent=2))
        add_file(z, PROJECT / "manifest.json", "project_manifest.json")
        for status_path in (RUN_ROOT / "monitor").glob("*/status.json"):
            name = status_path.parent.name
            add_file(z, status_path, "monitor/" + name + "/status.json")
            logs = sorted(status_path.parent.glob("session_*.jsonl"))
            if logs:
                with logs[-1].open() as stream:
                    tail = list(deque(stream, maxlen=2000))
                z.writestr("monitor/" + name + "/recent_events.jsonl", "".join(tail))
                for line in reversed(tail):
                    event = json.loads(line)
                    if event.get("event") == "claim_saved" and event.get("claim_id") is not None:
                        key = (name, str(event["claim_id"]))
                        if key not in recent_predictions:
                            recent_predictions.append(key)
                            if sum(n == name for n, _ in recent_predictions) >= 10:
                                break
        for condition_dir in (RUN_ROOT / "predictions").glob("*"):
            if not condition_dir.is_dir():
                continue
            add_file(z, condition_dir / "protocol.json", "conditions/" + condition_dir.name + "/protocol.json")
            if EXPORT_ALL_PREDICTIONS_CSV:
                print("Exporting compact predictions:", condition_dir.name, flush=True)
                buffer = io.StringIO()
                columns = ["id", "claim", "label", "model_label", "evidence_status", "input_tokens", "output_tokens", "wall_seconds", "error"]
                writer = csv.DictWriter(buffer, fieldnames=columns, extrasaction="ignore")
                writer.writeheader()
                for path in condition_dir.glob("*.json"):
                    if path.name == "protocol.json":
                        continue
                    writer.writerow(json.loads(path.read_text()))
                z.writestr("conditions/" + condition_dir.name + "/predictions.csv", buffer.getvalue())
        for name, claim_id in recent_predictions:
            add_file(z, RUN_ROOT / "predictions" / name / (claim_id + ".json"), "traces/" + name + "/" + claim_id + ".json")
            add_file(z, RUN_ROOT / "evidence" / ("claim_" + claim_id + ".json"), "evidence/" + claim_id + ".json")
        for path in RUN_ROOT.glob("metrics_*.json"):
            add_file(z, path, "metrics/" + path.name)
        for pattern in ["training/*/protocol.json", "training/*/complete.json",
                        "training/*/monitor/status.json", "training/*/checkpoints/checkpoint-*/trainer_state.json"]:
            for path in RUN_ROOT.glob(pattern):
                add_file(z, path, str(path.relative_to(RUN_ROOT)))
        for path in RUN_ROOT.glob("training/*/monitor/session_*.jsonl"):
            with path.open() as stream:
                tail = list(deque(stream, maxlen=300))
            z.writestr(str(path.relative_to(RUN_ROOT)), "".join(tail))
        # Include the small engineering check too, if no long evaluation has run yet.
        for path in (RUN_ROOT / "software_check").glob("*.json"):
            add_file(z, path, "software_check/" + path.name)
        for path in (RUN_ROOT / "software_check" / "evidence").glob("*.json"):
            add_file(z, path, "software_check/evidence/" + path.name)
    print("Saved debug ZIP:", export_path)
    print("Size:", round(export_path.stat().st_size / 2**20, 2), "MiB")
    print("If your browser blocks the download, get this ZIP from FEVER_Project / exports in Drive.")
    files.download(str(export_path))

## What this notebook does not claim
- No GPU run has been executed by the notebook author on your Colab account. Local tests cover data guards, response validation and layer selection, not cloud throughput or final accuracy.
- The 3B NF4 models and HF decoder differ from the earlier 8B Ollama experiment. New results need their own names and must not replace the old results.
- The three model families have different architectures and effective context budgets. Same passages do not imply identical retained evidence.
- This does not implement a new symbolic architecture or a Reddit study. Your professor's intended Reddit task still needs clarification.
- Free Colab is time-limited. Save Drive outputs, resume batches, and measure throughput before estimating a full-run completion date. Do not use keep-alive tricks to bypass service limits.

All source code used by a condition is in the uploaded bundle under `src/`, with SHA-256 hashes in `manifest.json`. Exact prompts, responses, retained inputs and evidence revisions are saved under your Drive run folder.